### colab용 csv 파일 로드

In [ ]:
import pandas as pd
import os

# 1. Zip 파일 다운로드
# -O data.zip: 다운받은 파일 이름을 data.zip으로 저장 (관리하기 쉽게)
!wget -O data.zip "https://cfiles.dacon.co.kr/competitions/236619/open.zip"

# 2. 압축 해제
# -o: 덮어쓰기 허용 (여러 번 실행해도 묻지 않음)
# -d ./data: 'data'라는 폴더를 만들어 그 안에 풀기 (파일이 섞이는 것 방지)
!unzip -o data.zip -d .


--2025-12-11 14:23:37--  https://cfiles.dacon.co.kr/competitions/236619/open.zip
Resolving cfiles.dacon.co.kr (cfiles.dacon.co.kr)... 172.67.161.58, 104.21.58.153, 2606:4700:3037::ac43:a13a, ...
Connecting to cfiles.dacon.co.kr (cfiles.dacon.co.kr)|172.67.161.58|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 173878 (170K) [application/zip]
Saving to: ‘data.zip’

data.zip            100%[===================>] 169.80K  --.-KB/s    in 0.02s   

2025-12-11 14:23:38 (8.16 MB/s) - ‘data.zip’ saved [173878/173878]

Archive:  data.zip
  inflating: ./sample_submission.csv  
  inflating: ./train.csv             


# 기본 Baseline 전처리

In [ ]:
# =========================================
# 1. 라이브러리 및 설치
# =========================================
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm

# =========================================
# 2. 데이터 전처리
# =========================================
train = pd.read_csv('./train.csv')
train.head(3)

,item_id,year,month,seq,type,hs4,weight,quantity,value
0,DEWLVASR,2022,1,1.0,1,3038,14858.0,0.0,32688.0
1,ELQGMQWE,2022,1,1.0,1,2002,62195.0,0.0,110617.0
2,AHMDUILJ,2022,1,1.0,1,2102,18426.0,0.0,72766.0


## pivot 생성

In [ ]:
# year, month, item_id 기준으로 value 합산
monthly = (
    train
    .groupby(["item_id", "year", "month"], as_index=False)["value"]
    .sum()
)

# year + month → datetime ym 키 생성
monthly["ym"] = pd.to_datetime(
    monthly["year"].astype(str) + "-" + monthly["month"].astype(str).str.zfill(2)
)

# item_id × ym 피벗 테이블
pivot = (
    monthly
    .pivot(index="item_id", columns="ym", values="value")
    .fillna(0.0)
)
pivot.head(3)

ym,2022-01-01,2022-02-01,2022-03-01,2022-04-01,2022-05-01,2022-06-01,2022-07-01,2022-08-01,2022-09-01,2022-10-01,...,2024-10-01,2024-11-01,2024-12-01,2025-01-01,2025-02-01,2025-03-01,2025-04-01,2025-05-01,2025-06-01,2025-07-01
item_id,,,,,,,,,,,,,,,,,,,,,
AANGBULD,14276.0,52347.0,53549.0,0.0,26997.0,84489.0,0.0,0.0,0.0,0.0,...,428725.0,144248.0,26507.0,25691.0,25805.0,0.0,38441.0,0.0,441275.0,533478.0
AHMDUILJ,242705.0,120847.0,197317.0,126142.0,71730.0,149138.0,186617.0,169995.0,140547.0,89292.0,...,123085.0,143451.0,78649.0,125098.0,80404.0,157401.0,115509.0,127473.0,89479.0,101317.0
ANWUJOKX,0.0,0.0,0.0,63580.0,81670.0,26424.0,8470.0,0.0,0.0,80475.0,...,0.0,0.0,0.0,27980.0,0.0,0.0,0.0,0.0,0.0,0.0


# EDA 및 시각화

In [ ]:
# hs2 컬럼 생성

train['hs2'] = train['hs4'] // 100
# train['hs2'] = train['hs4'].astype(str).str[:2]  # 문자열 기반 생성 시 사용
train.head(3)

,item_id,year,month,seq,type,hs4,weight,quantity,value,hs2
0,DEWLVASR,2022,1,1.0,1,3038,14858.0,0.0,32688.0,30
1,ELQGMQWE,2022,1,1.0,1,2002,62195.0,0.0,110617.0,20
2,AHMDUILJ,2022,1,1.0,1,2102,18426.0,0.0,72766.0,21


hs4 뿐만 아니라 hs2 분석 또한 유의미하다고 판단

In [ ]:
# type값 종류 확인
train['type'].unique()

array([1])

type 값은 모두 1로 동일하므로, 해당 컬럼을 제거하는 것이 바람직하다고 판단

In [ ]:
# type 컬럼 제거
train = train.drop('type', axis=1)
train.head(3)

,item_id,year,month,seq,hs4,weight,quantity,value,hs2
0,DEWLVASR,2022,1,1.0,3038,14858.0,0.0,32688.0,30
1,ELQGMQWE,2022,1,1.0,2002,62195.0,0.0,110617.0,20
2,AHMDUILJ,2022,1,1.0,2102,18426.0,0.0,72766.0,21


## seq와 hs4 코드의 상관관계를 통한 공행성쌍 탐색

###hs4값이 한번 순회할 때마다 seq의 값이 변화한다
- hs4값을 기준으로 결측치를 판단
- 순회 직전의 hs4값이 순회 후 hs4의 최소값 위치 사이에 다른 값이 들어갈 경우 해당 행 전부 제거
- 추가로 weight값과 quantity값이 0이 아니면서 value값이 0인 행 모두 결측치로 추가 판단해 제거

In [ ]:
!pip install fastdtw

In [ ]:
import pandas as pd

train = pd.read_csv('./data/train.csv')
train["ym"] = pd.to_datetime(
    seq_hs4["year"].astype(str) + "-" + seq_hs4["month"].astype(str).str.zfill(2)
)

delete_columns = train[(train["weight"]!=0)&(train["quantity"]!=0)&(train["value"]==0)].index
fix_train = train.drop(delete_columns)

seq_hs4 = fix_train[["item_id", "year", "month", "seq", "hs4", "value"]].copy()
seq_hs4["ym"] = pd.to_datetime(
    seq_hs4["year"].astype(str) + "-" + seq_hs4["month"].astype(str).str.zfill(2)
)

def preprocessing_dataframe(df: pd.DataFrame, col: str = "hs4") -> pd.DataFrame:
  if col not in df.columns:
    raise KeyError(f"데이터프레임에 '{col}' 컬럼이 없습니다.")

  result_df = df.copy()

  hs4_values = df[col].to_numpy()          # 위치 순서의 값 (numpy array)
  index_labels = df.index.to_numpy()

  n = len(hs4_values)
  if n < 2:
    return result_df

  del_labels = []

  for i in range(1, n):
    prev = hs4_values[i-1]
    cur  = hs4_values[i]

    # 결측치(또는 non-numeric) 처리: 안전하게 skip
    if pd.isna(prev) or pd.isna(cur):
      continue

    # 숫자가 아닌 경우(문자열 등)에도 안전하게 변환 시도
    try:
      prev_val = float(prev)
      cur_val  = float(cur)
    except Exception:
    # 변환 불가면 판단 불가 -> skip (원하면 별도 처리)
      continue

        # 범위 판정 함수(명확히)
    def is_1000s(x): return 1000 <= x <= 1999
    def is_9000s(x): return 9000 <= x <= 9999
    def is_2000_8999(x): return 2000 <= x <= 8999

        # 규칙 적용
    if is_9000s(prev_val):
      # prev가 9000대일 때 cur은 1000대 또는 9000대면 허용
      if is_1000s(cur_val) or is_9000s(cur_val):
        pass  # 정상
      elif is_2000_8999(cur_val):
        # 잘못된 순회(9000대 -> 2000~8999) => cur 제거
        del_labels.append(index_labels[i])
      else:
        # 다른 값(예: 음수, 10000 이상 등)은 정책에 따라 처리할 수 있음
        # 여기서는 안전하게 제거 대상으로 둔다
        del_labels.append(index_labels[i])
    else:
      # prev가 9000대가 아닐 때, 만약 prev > cur 이면 prev 제거(원래 로직 유지)
      if prev_val > cur_val:
        del_labels.append(index_labels[i-1])

  # 중복 라벨 제거 및 존재하는 라벨만 필터
  del_labels = list(dict.fromkeys(del_labels))  # 순서 보장된 유니크
  # 실제로 존재하는 라벨만 남김 (안전성)
  existing_labels = [lbl for lbl in del_labels if lbl in result_df.index]

    # 한 번에 삭제
  if existing_labels:
    result_df = result_df.drop(index=existing_labels)

  return result_df

preprocess_seq_hs4 = preprocessing_dataframe(seq_hs4)
preprocess_seq_hs4

###검증

In [ ]:
# =============================================================================
# 수정된 전체 파이프라인 (IndexError 방지 포함)
# =============================================================================

import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.spatial.distance import euclidean
from fastdtw import fastdtw
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 유틸리티: lag 추정용 (cross-correlation)
# -----------------------------
def estimate_lag_and_corr(a, b, max_lag=6):
    if len(a) == 0 or len(b) == 0:
        return 0, 0.0
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if np.nanstd(a) == 0 or np.nanstd(b) == 0:
        return 0, 0.0
    a = (a - np.nanmean(a)) / (np.nanstd(a) + 1e-9)
    b = (b - np.nanmean(b)) / (np.nanstd(b) + 1e-9)
    min_len = min(len(a), len(b))
    if min_len < 4:
        return 0, 0.0
    a = a[-min_len:]
    b = b[-min_len:]
    best_lag = 0
    best_corr = 0.0
    for lag in range(-max_lag, max_lag + 1):
        try:
            if lag < 0:
                la = a[:lag]
                lb = b[-lag:]
            elif lag > 0:
                la = a[lag:]
                lb = b[:-lag]
            else:
                la = a
                lb = b
            if len(la) < 2 or len(lb) < 2:
                corr = 0.0
            else:
                corr = np.corrcoef(la, lb)[0,1]
                if np.isnan(corr):
                    corr = 0.0
        except Exception:
            corr = 0.0
        if abs(corr) > abs(best_corr):
            best_corr = corr
            best_lag = lag
    return int(best_lag), float(best_corr)

# -----------------------------
# optional: DTW similarity
# -----------------------------
def dtw_similarity(a, b):
    if len(a) == 0 or len(b) == 0:
        return 0.0
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if np.nanstd(a) == 0 or np.nanstd(b) == 0:
        return 0.0
    a_std = (a - np.nanmean(a)) / (np.nanstd(a) + 1e-9)
    b_std = (b - np.nanmean(b)) / (np.nanstd(b) + 1e-9)
    dist, _ = fastdtw(a_std, b_std, dist=euclidean)
    max_len = max(len(a_std), len(b_std))
    sim = np.exp(-dist / (max_len + 1e-9))
    return float(sim)

# -----------------------------
# pivot 생성 도우미
# -----------------------------
def make_pivot_from_df(df, date_col='ym', value_col='value', keep_itemid_as_str=True):
    tmp = df.copy()
    if 'ym' not in tmp.columns:
        tmp['year'] = tmp['year'].astype(int)
        tmp['month'] = tmp['month'].astype(int)
        tmp['ym'] = pd.to_datetime(tmp['year'].astype(str) + '-' + tmp['month'].astype(str) + '-01')
    if keep_itemid_as_str:
        tmp['item_id'] = tmp['item_id'].astype(str)
    pivot = tmp.pivot_table(index='item_id', columns='ym', values=value_col, aggfunc='sum')
    pivot = pivot.reindex(sorted(pivot.columns), axis=1)
    return pivot

# -----------------------------
# pairs 생성: 안전 검사 포함
# -----------------------------
def build_pairs_from_pivot(pivot, max_lag=6, min_overlap=6, dtw_filter=None, top_k=5):
    items = pivot.index.to_list()
    series_cache = {}
    for it in items:
        arr = pivot.loc[it].to_numpy(dtype=float)
        series_cache[it] = arr

    pairs = []
    for follow in tqdm(items, desc="building pairs"):
        b = series_cache[follow]
        for lead in items:
            if lead == follow:
                continue
            a = series_cache[lead]
            # 두 시계열에서 동시 관측(결측 제외) 길이
            mask = (~np.isnan(a)) & (~np.isnan(b))
            if mask.sum() < min_overlap:
                continue
            a_ = a[mask]
            b_ = b[mask]
            # 안전: 빈 배열 방지
            if len(a_) < 4 or len(b_) < 4:
                continue
            # optional dtw filter
            if dtw_filter is not None:
                try:
                    sim = dtw_similarity(a_, b_)
                except Exception:
                    sim = 0.0
                if sim < dtw_filter:
                    continue
            # lag/corr 계산
            try:
                lag, corr = estimate_lag_and_corr(a_, b_, max_lag=max_lag)
            except Exception:
                lag, corr = 0, 0.0
            pairs.append({
                'leading_item_id': str(lead),
                'following_item_id': str(follow),
                'max_corr': float(corr),
                'best_lag': int(lag)
            })

    pairs_df = pd.DataFrame(pairs)
    if pairs_df.empty:
        return pairs_df
    pairs_df['abs_corr'] = pairs_df['max_corr'].abs()
    pairs_df = pairs_df.sort_values(['following_item_id','abs_corr'], ascending=[True, False])
    pairs_df = pairs_df.groupby('following_item_id').head(top_k).reset_index(drop=True)
    pairs_df = pairs_df.drop(columns=['abs_corr'])
    return pairs_df

# -----------------------------
# build_training_data (Index 범위 체크 추가)
# -----------------------------
def build_training_data(pivot, pairs):
    months = pivot.columns.to_list()
    n_months = len(months)
    rows = []
    for row in pairs.itertuples(index=False):
        leader = str(row.leading_item_id)
        follower = str(row.following_item_id)
        lag = int(row.best_lag)
        corr = float(row.max_corr)
        if leader not in pivot.index or follower not in pivot.index:
            continue
        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)
        # 안전성: 길이 일치 확인 (둘다 n_months 길이임을 기대하지만 확실히 검사)
        la = len(a_series); lb = len(b_series)
        if la != n_months or lb != n_months:
            # 만약 길이가 다르면, 작은 길이에 맞춰서 학습 가능 구간 제한
            # 하지만 pivot 생성 방식상 보통 동일하므로 여기선 넘어가거나 계속 진행
            pass
        for t in range(max(lag, 1), n_months - 1):
            # 인덱스 범위 체크
            idx_b_t = t
            idx_b_t_1 = t - 1
            idx_a_t_lag = t - lag
            idx_b_t_plus_1 = t + 1
            if not (0 <= idx_b_t < lb and 0 <= idx_b_t_1 < lb and 0 <= idx_a_t_lag < la and 0 <= idx_b_t_plus_1 < lb):
                continue
            b_t = b_series[idx_b_t]
            b_t_1 = b_series[idx_b_t_1]
            a_t_lag = a_series[idx_a_t_lag]
            b_t_plus_1 = b_series[idx_b_t_plus_1]
            # 결측치 검사
            if np.isnan(b_t) or np.isnan(b_t_1) or np.isnan(a_t_lag) or np.isnan(b_t_plus_1):
                continue
            rows.append({
                "b_t": float(b_t),
                "b_t_1": float(b_t_1),
                "a_t_lag": float(a_t_lag),
                "max_corr": corr,
                "best_lag": float(lag),
                "target": float(b_t_plus_1),
            })
    df_train = pd.DataFrame(rows)
    return df_train

# -----------------------------
# 학습/예측 (Index 체크 강화)
# -----------------------------
def train_regressor(df_train):
    feature_cols = ['b_t', 'b_t_1', 'a_t_lag', 'max_corr', 'best_lag']
    if df_train.shape[0] < 10:
        return None, None
    X = df_train[feature_cols].values
    y = df_train['target'].values
    reg = LinearRegression()
    reg.fit(X, y)
    return reg, feature_cols

def predict_from_pairs(pivot, pairs, reg, feature_cols):
    months = pivot.columns.to_list()
    n_months = len(months)
    t_last = n_months - 1
    t_prev = n_months - 2
    preds = []
    for row in pairs.itertuples(index=False):
        leader = str(row.leading_item_id)
        follower = str(row.following_item_id)
        lag = int(row.best_lag)
        corr = float(row.max_corr)
        if leader not in pivot.index or follower not in pivot.index:
            continue
        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)
        la = len(a_series); lb = len(b_series)
        # 인덱스 계산 및 범위 체크
        idx_b_t = t_last
        idx_b_t_1 = t_prev
        idx_a_t_lag = t_last - lag
        if not (0 <= idx_b_t < lb and 0 <= idx_b_t_1 < lb and 0 <= idx_a_t_lag < la):
            # 예측 불가 (인덱스 초과) -> skip
            continue
        b_t = b_series[idx_b_t]
        b_t_1 = b_series[idx_b_t_1]
        a_t_lag = a_series[idx_a_t_lag]
        if np.isnan(b_t) or np.isnan(b_t_1) or np.isnan(a_t_lag):
            continue
        X_test = np.array([[b_t, b_t_1, a_t_lag, corr, float(lag)]])
        try:
            y_pred = reg.predict(X_test)[0]
        except Exception:
            continue
        y_pred = max(0.0, float(y_pred))
        y_pred = int(round(y_pred))
        preds.append({
            "leading_item_id": leader,
            "following_item_id": follower,
            "value": int(y_pred),
        })
    df_pred = pd.DataFrame(preds)
    return df_pred

# -----------------------------
# 전체 실행 파이프라인
# -----------------------------
def full_pipeline_from_df(df, dtw_filter=None, max_lag=6, min_overlap=6, top_k=5):
    if 'ym' not in df.columns:
        df = df.copy()
        df['year'] = df['year'].astype(int)
        df['month'] = df['month'].astype(int)
        df['ym'] = pd.to_datetime(df['year'].astype(str) + '-' + df['month'].astype(str) + '-01')
    df = df.copy()
    df['item_id'] = df['item_id'].astype(str)
    pivot = make_pivot_from_df(df, date_col='ym', value_col='value', keep_itemid_as_str=True)
    # pivot columns 수(월 수)가 0이면 중단
    if pivot.shape[1] == 0:
        raise ValueError("pivot에 month 컬럼이 없습니다. 입력 데이터의 year/month를 확인하세요.")
    pairs = build_pairs_from_pivot(pivot, max_lag=max_lag, min_overlap=min_overlap, dtw_filter=dtw_filter, top_k=top_k)
    if pairs.empty:
        print("pairs 생성 실패(빈 결과). 파라미터를 완화하세요.")
        return pd.DataFrame(columns=['leading_item_id','following_item_id','value']), pivot, pairs
    df_train_model = build_training_data(pivot, pairs)
    print('생성된 학습 데이터의 shape :', df_train_model.shape)
    reg, feature_cols = train_regressor(df_train_model)
    if reg is None:
        print("학습 데이터 부족: fallback으로 naive 예측 사용")
        rows = []
        for item in pivot.index:
            last = pivot.loc[item].dropna()
            val = int(round(last.iloc[-1])) if last.shape[0] > 0 else 0
            rows.append({'leading_item_id': item, 'following_item_id': item, 'value': val})
        return pd.DataFrame(rows), pivot, pairs
    submission_pred = predict_from_pairs(pivot, pairs, reg, feature_cols)
    following_set = set(submission_pred['following_item_id'].unique())
    rows_fallback = []
    for item in pivot.index:
        if item not in following_set:
            last = pivot.loc[item].dropna()
            val = int(round(last.iloc[-1])) if last.shape[0] > 0 else 0
            rows_fallback.append({'leading_item_id': item, 'following_item_id': item, 'value': val})
    if rows_fallback:
        fallback_df = pd.DataFrame(rows_fallback)
        submission_pred = pd.concat([submission_pred, fallback_df], ignore_index=True)
    submission_pred['leading_item_id'] = submission_pred['leading_item_id'].astype(str)
    submission_pred['following_item_id'] = submission_pred['following_item_id'].astype(str)
    submission_pred['value'] = submission_pred['value'].astype(int)
    return submission_pred.reset_index(drop=True), pivot, pairs


# =====================================
# 사용 예시
# =====================================
# df = pd.read_csv('your_preprocessed_data.csv')  # item_id 컬럼은 문자열
submission_df, pivot, pairs = full_pipeline_from_df(preprocess_seq_hs4, dtw_filter=None, max_lag=6, min_overlap=6, top_k=5)
# print(submission_df.head())


###시각화

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.plot(train["ym"], train["hs4"], marker=".", linestyle="-")
plt.title("hs4 값의 시간별 동향 (raw)")
plt.xlabel("ym")
plt.ylabel("hs4 value")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(train["ym"], train["seq"], marker=".", linestyle="-")
plt.title("seq 값의 시간별 동향 (raw)")
plt.xlabel("ym")
plt.ylabel("seq value")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(preprocess_seq_hs4["ym"], preprocess_seq_hs4["hs4"], marker=".", linestyle="-")
plt.title("hs4 값의 시간별 동향 (raw)")
plt.xlabel("ym")
plt.ylabel("hs4 value")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(preprocess_seq_hs4["ym"], preprocess_seq_hs4["seq"], marker=".", linestyle="-")
plt.title("seq 값의 시간별 동향 (raw)")
plt.xlabel("ym")
plt.ylabel("seq value")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

###분석
- 점수:0.3007814487
- 공행성 쌍 개수:2891
---
- 결측치로 판단되 제거된 행들 또한 규칙성의 띄는 경우도 존재하기에 이러한 방식의 제거는 유의미하지 못하다고 판단

## 시계열 그래프를 이용한 공행성쌍 탐색

### 시계열 그래프 간 유사도 비교하면 공행성쌍을 찾을 수 있다
- 선행 품목의 그래프를 기준으로 후행 품목의 그래프를 이동시키며 비교
- 그래프 일치율이 threshold 이상이면 공행성 쌍으로 판단
- ⇒ 그래프 모양이 비슷한가? 를 기준으로 판단. 음의 상관관계 또한 고려함.

#### 검증

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# 2. 패턴 분석용 전처리 (No-Log Version)
def preprocess_for_pattern_no_log(pivot_df):
    # [변경점] 로그 변환 없이 바로 차분(Diff)
    # 절대적인 금액의 증감 패턴을 봅니다.
    df_diff = pivot_df.diff(axis=1).fillna(0)

    # 스케일이 다르므로 MinMax Scaling은 필수
    scaler = MinMaxScaler()
    df_scaled = pd.DataFrame(scaler.fit_transform(df_diff.T).T,
                             index=df_diff.index,
                             columns=df_diff.columns)
    return df_scaled

print(">>> 패턴 데이터 생성 (No Log)...")
pivot_pattern = preprocess_for_pattern_no_log(pivot)

# 3. 공행성 탐색 (Zero-Masking 적용)
def find_pairs_no_log(pattern_df, raw_df, max_lag=6, corr_threshold=0.45):
    items = pattern_df.index.to_list()
    results = []

    mat_pat = pattern_df.values
    mat_raw = raw_df.values

    for i, leader in tqdm(enumerate(items), total=len(items), desc="Searching (No-Log)"):
        # 유령 품목 제거
        if np.count_nonzero(mat_raw[i]) < 12: continue

        leader_pat = mat_pat[i]
        leader_raw = mat_raw[i]

        for j, follower in enumerate(items):
            if i == j: continue
            if np.count_nonzero(mat_raw[j]) < 12: continue

            best_lag = 0
            best_corr = 0.0

            for lag in range(1, max_lag + 1):
                try:
                    # Lag Slicing
                    l_pat = leader_pat[:-lag]
                    f_pat = mat_pat[j][lag:]

                    l_raw = leader_raw[:-lag]
                    f_raw = mat_raw[j][lag:]

                    # [안전장치] 0인 구간 제외 (Zero-Masking)
                    # 0에서 튀는 스파이크 왜곡 방지
                    valid_mask = (l_raw > 0) & (f_raw > 0)

                    if np.sum(valid_mask) < 6: continue

                    # 상관계수 계산
                    corr = np.corrcoef(l_pat[valid_mask], f_pat[valid_mask])[0, 1]

                    if np.isnan(corr): corr = 0
                    if abs(corr) > abs(best_corr):
                        best_corr = corr
                        best_lag = lag
                except: continue

            if abs(best_corr) >= corr_threshold:
                results.append({
                    "leading_item_id": leader,
                    "following_item_id": follower,
                    "best_lag": best_lag,
                    "max_corr": best_corr,
                })

    return pd.DataFrame(results)

# [설정] Threshold 0.45 (로그를 뺐으니 조금 엄격하게)
pairs_nolog = find_pairs_no_log(pivot_pattern, pivot, corr_threshold=0.45)
print(f"탐색된 쌍: {len(pairs_nolog)}개")

>>> 패턴 데이터 생성 (No Log)...


Searching (No-Log): 100%|██████████| 100/100 [00:08<00:00, 12.21it/s]

탐색된 쌍: 1058개


#### 시각화

In [ ]:
# -----------------------------------------------------------
# 3. 시각화 검증 (그래프 개선)
# -----------------------------------------------------------
def plot_nolog_verification_improved(pivot_pattern, pivot_raw, leader, follower, lag):
    """
    0을 포함한 모든 데이터를 선명하게 시각화
    """
    # 데이터 추출
    l_pat = pivot_pattern.loc[leader].values
    f_pat = pivot_pattern.loc[follower].values

    l_raw = pivot_raw.loc[leader].values
    f_raw = pivot_raw.loc[follower].values

    # Lag 적용
    l_pat_sh = l_pat[:-lag]
    l_raw_sh = l_raw[:-lag]
    f_pat_sh = f_pat[lag:]
    f_raw_sh = f_raw[lag:]

    # 0 포함 상관계수 계산
    corr = np.corrcoef(l_pat_sh, f_pat_sh)[0, 1]

    # 시각화 설정
    fig, ax = plt.subplots(1, 2, figsize=(20, 7))
    x_axis = np.arange(len(l_pat_sh))

    # [왼쪽] 패턴(Diff+Scaled) 비교
    # 선을 굵게(lw=2), 마커 추가(marker='o')하여 잘 보이게 함
    ax[0].plot(l_pat_sh, color='dodgerblue', alpha=0.9, lw=2, marker='.', label='Leader (Leading)')
    ax[0].plot(f_pat_sh, color='crimson', alpha=0.8, lw=2, marker='.', label='Follower (Lagged)')

    ax[0].set_title(f"Pattern Match (0 Included)\\nLag: {lag} | Corr: {corr:.3f}", fontsize=15, fontweight='bold')
    ax[0].legend(fontsize=12)
    ax[0].grid(True, linestyle='--', alpha=0.6)
    ax[0].set_xlabel("Time Step")
    ax[0].set_ylabel("Scaled Diff Value")

    # [오른쪽] 원본 거래금액(Raw Value) 비교 (이축 그래프)
    ax1 = ax[1]
    ax2 = ax1.twinx()

    # Leader
    line1 = ax1.plot(l_raw_sh, color='navy', lw=2.5, linestyle='-', marker='o', markersize=4, label=f'Leader: {leader}')
    ax1.set_ylabel("Leader Amount", color='navy', fontsize=12)
    ax1.tick_params(axis='y', labelcolor='navy')

    # Follower
    line2 = ax2.plot(f_raw_sh, color='darkorange', lw=2.5, linestyle='-', marker='s', markersize=4, label=f'Follower: {follower}')
    ax2.set_ylabel("Follower Amount", color='darkred', fontsize=12)
    ax2.tick_params(axis='y', labelcolor='darkred')

    # 범례 합치기
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='upper left', fontsize=12)

    ax1.set_title("Raw Trade Amount Comparison (Dual Axis)", fontsize=15, fontweight='bold')
    ax1.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

for idx, row in tail_pairs.iterrows():
        plot_nolog_verification_improved(
            pivot_pattern, pivot,
            row['leading_item_id'],
            row['following_item_id'],
            int(row['best_lag'])
        )

#### 상관계수

In [ ]:
pairs = pairs_nolog

In [ ]:
import pandas as pd

pairs['abs_corr'] = pairs['max_corr'].abs()

# 1. 전체 평균 상관계수 구하기
mean_corr = pairs['abs_corr'].mean()
print(f"- 전체 평균 상관계수: {mean_corr:.4f}")

# 2. 분위수(Quantile) 계산
# 상위 20% 기준값 (80% 지점)
top_20_cutoff = pairs['abs_corr'].quantile(0.80)

# 하위 20% 기준값 (20% 지점)
bottom_20_cutoff = pairs['abs_corr'].quantile(0.20)

print(f"- 상위 20% 커트라인: {top_20_cutoff:.4f} 이상")
print(f"- 하위 20% 커트라인: {bottom_20_cutoff:.4f} 이하")

# 3. 데이터 추출 (Filtering)
top_20_df = pairs[pairs['abs_corr'] >= top_20_cutoff]
bottom_20_df = pairs[pairs['abs_corr'] <= bottom_20_cutoff]

# 4. 결과 출력
print(f"\n[상위 20% 데이터] - 총 {len(top_20_df)}개")
# 상위 5개만 확인
print(top_20_df[['leading_item_id', 'following_item_id', 'best_lag', 'abs_corr']].head())

print(f"\n[하위 20% 데이터] - 총 {len(bottom_20_df)}개")
# 하위 5개만 확인
print(bottom_20_df[['leading_item_id', 'following_item_id', 'best_lag', 'abs_corr']].head())

- 전체 평균 상관계수: 0.5371
- 상위 20% 커트라인: 0.5854 이상
- 하위 20% 커트라인: 0.4740 이하

[상위 20% 데이터] - 총 212개
   leading_item_id following_item_id  best_lag  abs_corr
6         AANGBULD          FWUCPMMW         1  0.607415
10        AANGBULD          NAQIHUKZ         6  0.695869
15        AANGBULD          VUAFAIYJ         2  0.675200
17        AANGBULD          ZCELVYQU         4  0.691720
32        APQGTRMF          FTSVTTSR         1  0.585745

[하위 20% 데이터] - 총 212개
   leading_item_id following_item_id  best_lag  abs_corr
1         AANGBULD          BLANHGYY         1  0.472296
2         AANGBULD          DEWLVASR         4  0.473927
9         AANGBULD          LLHREMKS         5  0.463276
11        AANGBULD          OJIFIHMZ         4  0.450510
19        AHMDUILJ          BUZIIBYG         4  0.455667


#### 분석
- 점수: 0.285355875
- 공행성 쌍 개수: 1058
- 전체 평균 상관계수: 0.5371
- 상위 20% 커트라인: 0.5854 이상
- 하위 20% 커트라인: 0.4740 이하
---
- 0이 다수인 그래프의 한번의 거래가 모든 급등락한 그래프와 쌍으로 맺어지는 현상으로 왜곡 현상이 발생한 것으로 추측


### value=0 인 경우를 제외하여 그래프를 비교하면 더 나을 것이다
 - 이전 가설에서 가장 문제로 판단했던 0을 완전히 제거하고 비교
 - 0인 부분을 제외하고 나머지 부분간 그래프 유사도를 판단

#### 검증

In [ ]:
from sklearn.preprocessing import MinMaxScaler

def preprocess_for_pattern_no_log(pivot_df):
    # 로그 변환 없이 차분(Diff)
    df_diff = pivot_df.diff(axis=1).fillna(0)

    # MinMax Scaling
    scaler = MinMaxScaler()
    df_scaled = pd.DataFrame(scaler.fit_transform(df_diff.T).T,
                             index=df_diff.index,
                             columns=df_diff.columns)
    return df_scaled

print(">>> 패턴 데이터 생성 (No Log, 0 포함)...")
pivot_pattern = preprocess_for_pattern_no_log(pivot)

# -----------------------------------------------------------
# 2. 공행성 탐색 (Zero 포함 로직 수정)
# -----------------------------------------------------------
def find_pairs_no_log_with_zero(pattern_df, raw_df, max_lag=6, corr_threshold=0.45):
    items = pattern_df.index.to_list()
    results = []

    mat_pat = pattern_df.values
    mat_raw = raw_df.values

    # 전체 데이터 길이를 미리 계산
    n_time_steps = mat_pat.shape[1]

    for i, leader in tqdm(enumerate(items), total=len(items), desc="Searching (Include-Zero)"):
        # (선택사항) 데이터가 너무 적은 유령 품목은 여전히 거르는 것이 좋음 (최소 12개월치 데이터)
        if np.count_nonzero(mat_raw[i]) < 12: continue

        leader_pat = mat_pat[i]

        for j, follower in enumerate(items):
            if i == j: continue
            if np.count_nonzero(mat_raw[j]) < 12: continue

            best_lag = 0
            best_corr = 0.0

            for lag in range(1, max_lag + 1):
                try:
                    # Lag Slicing
                    # 0인 구간을 거르지 않고 전체 데이터를 사용합니다.
                    l_pat = leader_pat[:-lag]
                    f_pat = mat_pat[j][lag:]

                    # 상관계수 계산 (전체 구간)
                    corr = np.corrcoef(l_pat, f_pat)[0, 1]

                    if np.isnan(corr): corr = 0

                    # 절대값 기준 최대 상관계수 찾기
                    if abs(corr) > abs(best_corr):
                        best_corr = corr
                        best_lag = lag
                except: continue

            if abs(best_corr) >= corr_threshold:
                results.append({
                    "leading_item_id": leader,
                    "following_item_id": follower,
                    "best_lag": best_lag,
                    "max_corr": best_corr,
                })

    return pd.DataFrame(results)

# 탐색 실행
pairs_nolog = find_pairs_no_log_with_zero(pivot_pattern, pivot, corr_threshold=0.45)
print(f"탐색된 쌍 (0 포함): {len(pairs_nolog)}개")

# 샘플 확인
if len(pairs_nolog) > 0:
    print("\n>>> 공행성 쌍 시각화 검증 <<<")
    pairs_nolog['abs_corr'] = pairs_nolog['max_corr'].abs()
    tail_pairs = pairs_nolog.sort_values(by='abs_corr').tail(3)

else:
    print("탐색된 쌍이 없습니다.")

>>> 패턴 데이터 생성 (No Log, 0 포함)...


Searching (Include-Zero): 100%|██████████| 100/100 [00:05<00:00, 17.20it/s]

탐색된 쌍 (0 포함): 839개

>>> 공행성 쌍 시각화 검증 <<<


#### 시각화

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_nolog_verification(pivot_pattern, pivot_raw, leader, follower, lag):
    """
    No-Log 전략 검증용 시각화
    1. 왼쪽: 전처리된 패턴(Diff + Scaling) 매칭 여부 (유효 구간만 강조)
    2. 오른쪽: 실제 거래량(Raw Value) 비교 (Dual Axis 적용) + 0 구간 표시
    """
    # 1. 데이터 추출
    # 패턴 데이터
    l_pat = pivot_pattern.loc[leader].values
    f_pat = pivot_pattern.loc[follower].values

    # 원본 데이터 (Linear Scale)
    l_raw = pivot_raw.loc[leader].values
    f_raw = pivot_raw.loc[follower].values

    # 2. Lag Slicing (시차 적용)
    l_pat_sh = l_pat[:-lag]
    l_raw_sh = l_raw[:-lag]

    f_pat_sh = f_pat[lag:]
    f_raw_sh = f_raw[lag:]

    # 3. Mask 생성 (둘 다 0보다 큰 구간)
    valid_mask = (l_raw_sh > 0) & (f_raw_sh > 0)
    x_axis = np.arange(len(l_pat_sh))

    # 상관계수 재계산 (제목 표시용)
    if np.sum(valid_mask) > 1:
        corr = np.corrcoef(l_pat_sh[valid_mask], f_pat_sh[valid_mask])[0, 1]
    else:
        corr = 0.0

    # --- Plotting ---
    fig, ax = plt.subplots(1, 2, figsize=(20, 6))

    # [Plot 1] 패턴 매칭 (Mask 적용)
    # 무시된 구간(0 포함)은 흐릿하게
    ax[0].plot(l_pat_sh, color='gray', alpha=0.2, linestyle='--')
    ax[0].plot(f_pat_sh, color='gray', alpha=0.2, linestyle='-')

    # 유효 구간(Valid)은 진하게 점으로 표시
    ax[0].scatter(x_axis[valid_mask], l_pat_sh[valid_mask], color='blue', s=20, label='Leader Pattern (Valid)')
    ax[0].scatter(x_axis[valid_mask], f_pat_sh[valid_mask], color='red', s=20, label='Follower Pattern (Valid)')

    ax[0].set_title(f"Pattern Match (No-Log, Diff)\nValid Corr: {corr:.3f}", fontsize=14)
    ax[0].legend()
    ax[0].grid(True, alpha=0.3)

    # [Plot 2] 실제 거래량 (Dual Axis) - 이게 제일 중요함!
    ax1 = ax[2-1] # 오른쪽 그래프
    ax2 = ax1.twinx() # 오른쪽 축 생성 (단위가 다를 수 있으므로)

    # 0인 구간(무시된 구간) 회색 음영 처리
    # valid_mask가 False인 구간을 칠함
    for i in range(len(valid_mask)):
        if not valid_mask[i]:
            ax1.axvspan(i-0.5, i+0.5, color='gray', alpha=0.1, lw=0)

    # Leader (왼쪽 축, 파란 점선)
    line1 = ax1.plot(l_raw_sh, color='blue', linestyle='--', label=f'Leader: {leader} (Left Axis)')
    ax1.set_ylabel("Leader Volume", color='blue')
    ax1.tick_params(axis='y', labelcolor='blue')

    # Follower (오른쪽 축, 빨간 실선)
    line2 = ax2.plot(f_raw_sh, color='red', linestyle='-', label=f'Follower: {follower} (Right Axis)')
    ax2.set_ylabel("Follower Volume", color='red')
    ax2.tick_params(axis='y', labelcolor='red')

    # 범례 합치기
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='upper left')

    ax1.set_title("Raw Trade Volume (Linear Scale)\nGray Zone = Ignored (Zero or Sparse)", fontsize=14)
    ax1.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

if len(pairs_nolog) > 0:
    # 상관계수 절대값 기준 정렬
    pairs_nolog['abs_corr'] = pairs_nolog['max_corr'].abs()
    tail_pairs = pairs_nolog.sort_values(by='abs_corr').tail(3)

    print(">>> [No-Log] 하위 공행성 쌍 시각화 검증 <<<")
    for idx, row in tail_pairs.iterrows():
        plot_nolog_verification(pivot_pattern, pivot,
                                row['leading_item_id'],
                                row['following_item_id'],
                                int(row['best_lag']))
else:
    print("탐색된 쌍이 없습니다.")

#### 상관계수

In [ ]:
pairs = pairs_nolog

In [ ]:
import pandas as pd

pairs['abs_corr'] = pairs['max_corr'].abs()

# 1. 전체 평균 상관계수 구하기
mean_corr = pairs['abs_corr'].mean()
print(f"- 전체 평균 상관계수: {mean_corr:.4f}")

# 2. 분위수(Quantile) 계산
# 상위 20% 기준값 (80% 지점)
top_20_cutoff = pairs['abs_corr'].quantile(0.80)

# 하위 20% 기준값 (20% 지점)
bottom_20_cutoff = pairs['abs_corr'].quantile(0.20)

print(f"- 상위 20% 커트라인: {top_20_cutoff:.4f} 이상")
print(f"- 하위 20% 커트라인: {bottom_20_cutoff:.4f} 이하")

# 3. 데이터 추출 (Filtering)
top_20_df = pairs[pairs['abs_corr'] >= top_20_cutoff]
bottom_20_df = pairs[pairs['abs_corr'] <= bottom_20_cutoff]

# 4. 결과 출력
print(f"\n[상위 20% 데이터] - 총 {len(top_20_df)}개")
# 상위 5개만 확인
print(top_20_df[['leading_item_id', 'following_item_id', 'best_lag', 'abs_corr']].head())

print(f"\n[하위 20% 데이터] - 총 {len(bottom_20_df)}개")
# 하위 5개만 확인
print(bottom_20_df[['leading_item_id', 'following_item_id', 'best_lag', 'abs_corr']].head())

- 전체 평균 상관계수: 0.5252
- 상위 20% 커트라인: 0.5708 이상
- 하위 20% 커트라인: 0.4695 이하

[상위 20% 데이터] - 총 168개
   leading_item_id following_item_id  best_lag  abs_corr
5         AANGBULD          NAQIHUKZ         6  0.595313
23        APQGTRMF          FTSVTTSR         1  0.585380
26        APQGTRMF          KJNSOAHR         1  0.604216
36        ATLDMDBO          DJBLNPNC         1  0.613821
37        ATLDMDBO          FDXPMYGF         4  0.634156

[하위 20% 데이터] - 총 168개
   leading_item_id following_item_id  best_lag  abs_corr
4         AANGBULD          LTOYKIML         2  0.450146
6         AANGBULD          OJIFIHMZ         4  0.450353
7         AANGBULD          SAAYMURU         5  0.463116
8         AANGBULD          STZDBITS         4  0.463278
10        AANGBULD          XIIEJNEE         6  0.463377


#### 분석
- 점수: 0.204536744
- 공행성 쌍 개수: 839
- 전체 평균 상관계수: 0.5252
- 상위 20% 커트라인: 0.5708 이상
- 하위 20% 커트라인: 0.4695 이하
---
- 원인으로 추측되었던 0을 제거하니 오히려 점수가 크게 감소
- 필터로 거르는 방식이 아닌 공행성쌍의 개수가 늘어나는 방식이 필요했음
- 즉, 해당 접근 방식은 틀린 것으로 결론

## 변화량 데이터를 기반으로 유사도 계산
- 미분 값이 변화량을 기반으로 판단하는 것이 더 나은 공행성쌍을 탐색할 수 있을 것이다
- 추가로 코사인 유사도 기반 분류 및 산점도 시각화
  - 두 아이템이 공행성(미분 유사도)이 높다면, 점들이 우상향 대각선( y=x ) 형태로 뭉치게 된다
  - 대각선에 유사하게 존재할수록 강한 인과관계

### 검증

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# =========================================
# 2. 미분(기울기) 패턴 생성 함수
# =========================================
def calculate_slope_pattern(pivot_df):
    """
    데이터를 미분(차분)하여 기울기 패턴을 생성하고 스케일링합니다.
    """
    # 1차 미분 (차분): 이번 달 - 지난 달 = 기울기(Slope)
    # 변화량(증감)의 형태를 봅니다.
    df_slope = pivot_df.diff(axis=1).fillna(0)

    # 패턴 비교를 위해 MinMax Scaling (0~1 사이로 변환)
    # 거래 규모가 달라도 패턴(기울기)이 비슷하면 매칭하기 위함
    scaler = MinMaxScaler()
    df_scaled = pd.DataFrame(scaler.fit_transform(df_slope.T).T,
                             index=df_slope.index,
                             columns=df_slope.columns)
    return df_scaled

print(">>> 기울기(미분) 패턴 생성 중...")
pivot_slope = calculate_slope_pattern(pivot)

# =========================================
# 3. 기울기 패턴 매칭 (분류) 함수
# =========================================
def find_slope_pairs(pattern_df, raw_df, max_lag=6, corr_threshold=0.5):
    items = pattern_df.index.to_list()
    results = []

    mat_pat = pattern_df.values
    mat_raw = raw_df.values

    # 진행상황 표시
    for i, leader in tqdm(enumerate(items), total=len(items), desc="Slope Pattern Matching"):
        # 데이터가 너무 적은(0이 많은) 종목 제외
        if np.count_nonzero(mat_raw[i]) < 12: continue

        leader_pat = mat_pat[i]
        leader_raw = mat_raw[i]

        for j, follower in enumerate(items):
            if i == j: continue # 자기 자신 제외
            if np.count_nonzero(mat_raw[j]) < 12: continue

            best_lag = 0
            best_corr = 0.0

            # Lag(지연)를 1부터 max_lag까지 적용해보며 상관관계 확인
            for lag in range(1, max_lag + 1):
                try:
                    # 선행 종목(Leader)은 앞부분, 후행 종목(Follower)은 뒷부분을 잘라서 비교
                    l_pat = leader_pat[:-lag] # Leader의 현재까지 패턴
                    f_pat = mat_pat[j][lag:]  # Follower의 미래(Lag 이후) 패턴

                    l_raw_slice = leader_raw[:-lag]
                    f_raw_slice = mat_raw[j][lag:]

                    # 0인 구간이 너무 많으면 노이즈가 되므로, 둘 다 거래가 있는 구간 위주로 볼 수도 있음
                    # 여기서는 전체 기간에 대해 계산하되, 데이터가 충분한지 체크
                    valid_mask = (l_raw_slice > 0) | (f_raw_slice > 0) # 둘 중 하나라도 거래가 있으면 포함
                    if np.sum(valid_mask) < 6: continue

                    # 피어슨 상관계수 계산
                    corr = np.corrcoef(l_pat, f_pat)[0, 1]

                    if np.isnan(corr): corr = 0

                    # 최적의 Lag 찾기
                    if corr > best_corr: # 양의 상관관계만 (비슷한 패턴)
                        best_corr = corr
                        best_lag = lag
                except: continue

            # 임계값 넘으면 결과 저장
            if best_corr >= corr_threshold:
                results.append({
                    "leader": leader,
                    "follower": follower,
                    "lag": best_lag,
                    "correlation": best_corr
                })

    # 상관계수 높은 순으로 정렬
    results_df = pd.DataFrame(results)
    if not results_df.empty:
        results_df = results_df.sort_values(by='correlation', ascending=False).reset_index(drop=True)

    return results_df

# 실행 (Threshold 0.5 이상인 쌍 찾기)
slope_pairs = find_slope_pairs(pivot_slope, pivot, max_lag=6, corr_threshold=0.5)
print(f"탐색된 패턴 쌍: {len(slope_pairs)}개")

### 시각화

In [ ]:
# =========================================
# 4. 시각화 함수 (미분 패턴 + 원본 데이터)
# =========================================
def plot_slope_match(pattern_df, raw_df, leader, follower, lag):
    # 데이터 준비
    l_pat = pattern_df.loc[leader].values
    f_pat = pattern_df.loc[follower].values

    l_raw = raw_df.loc[leader].values
    f_raw = raw_df.loc[follower].values

    # Lag 적용하여 길이 맞춤 (상관관계 계산 시 사용된 구간)
    l_pat_vis = l_pat[:-lag]
    f_pat_vis = f_pat[lag:]

    l_raw_vis = l_raw[:-lag]
    f_raw_vis = f_raw[lag:]

    # 상관계수 재계산 (타이틀 표시용)
    corr = np.corrcoef(l_pat_vis, f_pat_vis)[0, 1]

    # 그래프 그리기
    fig, ax = plt.subplots(2, 1, figsize=(15, 10))

    # 1) [위] 미분(기울기) 패턴 비교 (Scaled Slope)
    ax[0].plot(l_pat_vis, color='blue', label=f'Leader Slope: {leader}', linewidth=2)
    ax[0].plot(f_pat_vis, color='red', linestyle='--', label=f'Follower Slope (Lag {lag}): {follower}', linewidth=2)
    ax[0].set_title(f"[Slope Pattern Match] Corr: {corr:.4f} (Lag: {lag} months)", fontsize=14, fontweight='bold')
    ax[0].set_ylabel("Scaled Slope (Change)")
    ax[0].legend(loc='upper left')
    ax[0].grid(True, alpha=0.3)

    # 2) [아래] 원본 거래액 비교 (Raw Amount) - 이중축
    ax1 = ax[1]
    ax2 = ax1.twinx()

    line1 = ax1.plot(l_raw_vis, color='navy', marker='o', label=f'Leader Raw: {leader}', alpha=0.8)
    line2 = ax2.plot(f_raw_vis, color='darkred', marker='s', label=f'Follower Raw (Lagged): {follower}', alpha=0.8)

    ax1.set_title("Raw Value Comparison (Lag Applied)", fontsize=14, fontweight='bold')
    ax1.set_ylabel("Leader Amount", color='navy')
    ax2.set_ylabel("Follower Amount", color='darkred')
    ax1.set_xlabel("Time Step (Aligned)")

    # 범례 합치기
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='upper left')
    ax1.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# 상위 5개 결과 시각화
if not slope_pairs.empty:
    print(">>> 상위 5개 매칭 결과 시각화")
    for idx, row in slope_pairs.head(5).iterrows():
        plot_slope_match(pivot_slope, pivot, row['leader'], row['follower'], int(row['lag']))
else:
    print("매칭된 쌍이 없습니다. Threshold를 낮춰보세요.")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

# =========================================
# 3. 미분값(변화량) 기준 - 코사인 유사도 분석
# =========================================

# 3-1. 미분 데이터 (결측치 0 처리: 변화 없음으로 간주)
diff_pivot = pivot.diff(axis=1).iloc[:, 1:].fillna(0)

# 3-2. 코사인 유사도 계산
# 코사인 유사도는 -1 ~ 1 사이 값을 가지며, 1에 가까울수록 '변화의 방향'이 같습니다.
# diff_pivot은 (아이템 x 시간)이므로 그대로 넣으면 아이템 간 유사도가 나옵니다.
cosine_sim_matrix = cosine_similarity(diff_pivot)

# DataFrame으로 변환 (인덱스/컬럼 매핑)
cosine_sim_df = pd.DataFrame(
    cosine_sim_matrix,
    index=diff_pivot.index,
    columns=diff_pivot.index
)

# 3-3. 유사도 높은 쌍 추출 (함수 재사용 가능하도록 로직 단순화)
def get_cosine_pairs(sim_df, threshold=0.8):
    # 상삼각행렬 마스킹
    mask = np.triu(np.ones(sim_df.shape), k=1).astype(bool)
    sim_series = sim_df.where(mask).stack()

    # 필터링
    high_sim = sim_series[sim_series >= threshold]

    pairs = []
    for idx, val in high_sim.items():
        pairs.append({"item_1": idx[0], "item_2": idx[1], "similarity": val})

    return pd.DataFrame(pairs).sort_values(by="similarity", ascending=False)

# 실행
cosine_pairs = get_cosine_pairs(cosine_sim_df, threshold=0.8)
print(f"방향성이 일치하는 쌍 개수: {len(cosine_pairs)}")


# =========================================
# 4. 시각화: 변화량 산점도 (Gradient Scatter)
# =========================================

def plot_gradient_scatter(pivot_diff, pair_df, num_plots=3):
    if len(pair_df) == 0:
        return

    plt.figure(figsize=(15, 5 * num_plots))

    for i in range(min(len(pair_df), num_plots)):
        row = pair_df.iloc[i]
        item_1 = row['item_1']
        item_2 = row['item_2']
        sim_val = row['similarity']

        # 변화량 데이터 추출
        d1 = pivot_diff.loc[item_1]
        d2 = pivot_diff.loc[item_2]

        # 서브플롯 설정
        plt.subplot(num_plots, 2, i*2 + 1)

        # 1. 시계열 변화량 비교 (Line Plot)
        # 스케일링하여 '타이밍'이 겹치는지 확인
        scaler = StandardScaler()
        plt.plot(d1.index, scaler.fit_transform(d1.values.reshape(-1,1)),
                 label=f'{item_1} Norm. Diff', alpha=0.8)
        plt.plot(d2.index, scaler.fit_transform(d2.values.reshape(-1,1)),
                 label=f'{item_2} Norm. Diff', alpha=0.8, linestyle='--')
        plt.title(f"Time Series of Gradients (Sim: {sim_val:.3f})")
        plt.legend()
        plt.xticks([]) # x축 라벨 생략 (가독성 위함)

        # 2. 변화량 산점도 (Scatter Plot) - 핵심 시각화
        plt.subplot(num_plots, 2, i*2 + 2)
        sns.regplot(x=d1, y=d2, scatter_kws={'alpha':0.5}, line_kws={'color':'red'})
        plt.xlabel(f"Delta of {item_1}")
        plt.ylabel(f"Delta of {item_2}")
        plt.title(f"Gradient Correlation (Reaction Analysis)")
        plt.grid(True)

    plt.tight_layout()
    plt.show()

plot_gradient_scatter(diff_pivot, cosine_pairs, num_plots=2)

### 분석
- 이전 분석법과는 다르게 변화율을 바탕으로 새로운 방법
- 다만 위 시각화와 같이 변화량이 적거나 0에 가까운 값 처리를 하지 못함

# 가설1. 다양한 후보군을 탐색하는 것이 더 올바른 공행성쌍을 찾을 수 있다.
- **다양한 전처리 시도**
    - 원본 데이터
    - Log 변환 + 스무딩
    - 표준화
    - Top-K
        - 각 leader 당 상위 20개만 필터링
- **상관관계 앙상블**
    - Pearson, Spearman, Kendall, Cross-correlation 사용
- **산업 분류 활용 강화**
    - 동일한 hs2의 경우 → 0.15 보너스
    - 동일한 hs4의 경우 → 0.10 추가 보너스
- **추가적인 파라미터 조정**
    - max_lag는 6~8 선에서 조정
    - min_nonzero는 10~20 선에서 조정
    - corr_threshold는 0.20~0.45 선에서 조정
    - top_k_per_leader는 10~20 선에서 조정
    - use_preprocessing는 True/False 교차로 사용
    - use_multi_feature는 True/False 교차로 사용
- 파라미터 조정을 통해 다양한 후보군을 탐색
- 이후 상위 n개를 필터링한다
- 수량보다는 품질에 집중한다.

### 가설1 검증 및 결과

In [ ]:
# =========================================
# 개선된 공행성쌍 탐색 함수
# =========================================
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

# =========================================
# 1. 향상된 피벗 테이블 생성 (hs4, seq 활용)
# =========================================
def create_enhanced_pivot(train):
    """seq와 hs4 정보를 활용한 향상된 피벗 테이블"""

    # hs2 생성 (산업 대분류)
    train['hs2'] = train['hs4'] // 100

    # 기본 월별 집계
    monthly = (
        train
        .groupby(["item_id", "year", "month"], as_index=False)
        .agg({
            'value': 'sum',
            'weight': 'sum',
            'quantity': 'sum',
            'seq': 'mean',  # seq 평균값
            'hs4': 'first',  # hs4는 item_id별로 고정
            'hs2': 'first'
        })
    )

    monthly["ym"] = pd.to_datetime(
        monthly["year"].astype(str) + "-" + monthly["month"].astype(str).str.zfill(2)
    )

    # value 피벗
    pivot_value = (
        monthly
        .pivot(index="item_id", columns="ym", values="value")
        .fillna(0.0)
    )

    # weight 피벗 (추가 정보)
    pivot_weight = (
        monthly
        .pivot(index="item_id", columns="ym", values="weight")
        .fillna(0.0)
    )

    # item_id별 메타 정보
    item_meta = monthly.groupby('item_id').agg({
        'hs4': 'first',
        'hs2': 'first',
        'seq': 'mean'
    }).to_dict('index')

    return pivot_value, pivot_weight, item_meta


# =========================================
# 2. 고급 상관관계 계산 함수들
# =========================================
def safe_corr(x, y):
    """안전한 상관계수 계산"""
    if np.std(x) == 0 or np.std(y) == 0:
        return 0.0, 1.0
    try:
        corr, pval = pearsonr(x, y)
        return float(corr), float(pval)
    except:
        return 0.0, 1.0

def calculate_granger_score(x, y, lag):
    """Granger Causality 스코어 (간소화 버전)"""
    n = len(x)
    if n <= lag + 5:
        return 0.0

    # x의 lag 이전 값들과 y의 현재 값 상관관계
    x_lagged = x[:-lag]
    y_current = y[lag:]

    corr, pval = safe_corr(x_lagged, y_current)

    # p-value가 낮을수록 (통계적으로 유의할수록) 높은 점수
    if pval < 0.05:
        return abs(corr) * (1 - pval)
    return 0.0

def calculate_dtw_distance(x, y):
    """Dynamic Time Warping 거리 (간소화 버전)"""
    from scipy.spatial.distance import euclidean

    # 시계열 정규화
    if np.std(x) > 0:
        x = (x - np.mean(x)) / np.std(x)
    if np.std(y) > 0:
        y = (y - np.mean(y)) / np.std(y)

    # 단순 유클리드 거리로 근사
    if len(x) != len(y):
        return 1000.0

    dist = euclidean(x, y)
    return dist

def calculate_mutual_information(x, y, bins=10):
    """상호정보량 계산"""
    try:
        # 이산화
        x_binned = np.digitize(x, np.histogram(x, bins=bins)[1][:-1])
        y_binned = np.digitize(y, np.histogram(y, bins=bins)[1][:-1])

        # 결합 확률 분포
        joint_hist = np.histogram2d(x_binned, y_binned, bins=bins)[0]
        joint_prob = joint_hist / np.sum(joint_hist)

        # 주변 확률 분포
        x_prob = np.sum(joint_prob, axis=1)
        y_prob = np.sum(joint_prob, axis=0)

        # 상호정보량
        mi = 0.0
        for i in range(len(x_prob)):
            for j in range(len(y_prob)):
                if joint_prob[i, j] > 0:
                    mi += joint_prob[i, j] * np.log(
                        joint_prob[i, j] / (x_prob[i] * y_prob[j] + 1e-10)
                    )

        return max(0.0, mi)
    except:
        return 0.0


# =========================================
# 3. 핵심: 개선된 공행성쌍 탐색
# =========================================
def find_comovement_pairs_advanced(
    pivot,
    pivot_weight=None,
    item_meta=None,
    max_lag=6,
    min_nonzero=12,
    corr_threshold=0.35,  # 기존 0.4보다 낮춤 (다른 지표로 보완)
    min_pvalue=0.1,  # p-value 임계값
    use_granger=True,
    use_mi=True,
    same_hs2_bonus=0.1,  # 같은 산업군일 때 보너스
):
    """
    다중 지표를 활용한 고급 공행성쌍 탐색

    개선 사항:
    1. Pearson + Spearman 상관계수 조합
    2. Granger Causality 스코어
    3. 상호정보량 (Mutual Information)
    4. hs2 산업 분류 활용
    5. p-value 기반 통계적 유의성 검증
    6. 가중치 정보 활용 (weight)
    """

    items = pivot.index.to_list()
    months = pivot.columns.to_list()
    n_months = len(months)

    results = []

    for i, leader in tqdm(enumerate(items), total=len(items), desc="공행성쌍 탐색"):
        x = pivot.loc[leader].values.astype(float)

        # 0이 아닌 값이 충분히 있는지 확인
        if np.count_nonzero(x) < min_nonzero:
            continue

        # leader의 메타 정보
        leader_hs2 = item_meta[leader]['hs2'] if item_meta and leader in item_meta else None

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)
            if np.count_nonzero(y) < min_nonzero:
                continue

            # follower의 메타 정보
            follower_hs2 = item_meta[follower]['hs2'] if item_meta and follower in item_meta else None

            best_lag = None
            best_score = 0.0
            best_corr = 0.0
            best_pval = 1.0
            best_granger = 0.0
            best_mi = 0.0

            # lag 탐색
            for lag in range(1, max_lag + 1):
                if n_months <= lag:
                    continue

                x_lagged = x[:-lag]
                y_shifted = y[lag:]

                # 1) Pearson 상관계수
                pearson_corr, pearson_pval = safe_corr(x_lagged, y_shifted)

                # 2) Spearman 상관계수 (비선형 관계 포착)
                try:
                    spearman_corr, spearman_pval = spearmanr(x_lagged, y_shifted)
                except:
                    spearman_corr, spearman_pval = 0.0, 1.0

                # 3) Granger Causality 스코어
                granger_score = 0.0
                if use_granger:
                    granger_score = calculate_granger_score(x, y, lag)

                # 4) 상호정보량
                mi_score = 0.0
                if use_mi:
                    mi_score = calculate_mutual_information(x_lagged, y_shifted)

                # 종합 점수 계산 (가중 평균)
                combined_score = (
                    0.4 * abs(pearson_corr) +
                    0.3 * abs(spearman_corr) +
                    0.2 * granger_score +
                    0.1 * mi_score
                )

                # hs2가 같은 산업군이면 보너스
                if leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2:
                    combined_score += same_hs2_bonus

                # p-value 체크 (통계적 유의성)
                avg_pval = (pearson_pval + spearman_pval) / 2

                if combined_score > best_score:
                    best_score = combined_score
                    best_lag = lag
                    best_corr = pearson_corr
                    best_pval = avg_pval
                    best_granger = granger_score
                    best_mi = mi_score

            # 채택 조건: 종합 점수 임계값 + p-value
            if (best_lag is not None and
                best_score >= corr_threshold and
                best_pval <= min_pvalue):

                results.append({
                    "leading_item_id": leader,
                    "following_item_id": follower,
                    "best_lag": best_lag,
                    "max_corr": best_corr,
                    "combined_score": best_score,
                    "pvalue": best_pval,
                    "granger_score": best_granger,
                    "mi_score": best_mi,
                    "same_hs2": 1 if (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2) else 0
                })

    pairs = pd.DataFrame(results)

    # 점수 기준 정렬
    if len(pairs) > 0:
        pairs = pairs.sort_values('combined_score', ascending=False)

    return pairs


# =========================================
# 4. 대체 전략: Top-K 필터링
# =========================================
def find_comovement_pairs_topk(
    pivot,
    item_meta=None,
    max_lag=6,
    min_nonzero=12,
    top_k_per_leader=10,  # 각 leader마다 상위 K개만 선택
):
    """
    각 선행 품목당 상위 K개 후행 품목만 선택하는 전략
    과적합 방지 및 품질 향상
    """

    items = pivot.index.to_list()
    months = pivot.columns.to_list()
    n_months = len(months)

    all_results = []

    for leader in tqdm(items, desc="Top-K 공행성쌍 탐색"):
        x = pivot.loc[leader].values.astype(float)

        if np.count_nonzero(x) < min_nonzero:
            continue

        leader_results = []

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)
            if np.count_nonzero(y) < min_nonzero:
                continue

            best_lag = None
            best_corr = 0.0
            best_pval = 1.0

            for lag in range(1, max_lag + 1):
                if n_months <= lag:
                    continue

                corr, pval = safe_corr(x[:-lag], y[lag:])

                if abs(corr) > abs(best_corr):
                    best_corr = corr
                    best_lag = lag
                    best_pval = pval

            if best_lag is not None and best_pval < 0.2:
                leader_results.append({
                    "leading_item_id": leader,
                    "following_item_id": follower,
                    "best_lag": best_lag,
                    "max_corr": best_corr,
                    "pvalue": best_pval,
                })

        # 각 leader당 상위 K개만 선택
        if leader_results:
            leader_df = pd.DataFrame(leader_results)
            leader_df = leader_df.nlargest(top_k_per_leader, 'max_corr')
            all_results.extend(leader_df.to_dict('records'))

    pairs = pd.DataFrame(all_results)
    return pairs


# =========================================
# 5. 사용 예시
# =========================================

# 방법 1: 고급 다중 지표 사용
pivot_value, pivot_weight, item_meta = create_enhanced_pivot(train)

pairs = find_comovement_pairs_advanced(
    pivot_value,
    pivot_weight=pivot_weight,
    item_meta=item_meta,
    max_lag=6,
    min_nonzero=12,
    corr_threshold=0.35,  # 조정 가능
    min_pvalue=0.1,
    use_granger=True,
    use_mi=True,
    same_hs2_bonus=0.1
)

print(f"탐색된 공행성쌍 수: {len(pairs)}")
print(pairs.head(10))


# 방법 2: Top-K 전략 (대안)
# pairs = find_comovement_pairs_topk(
#     pivot_value,
#     item_meta=item_meta,
#     max_lag=6,
#     min_nonzero=12,
#     top_k_per_leader=10
# )


# =========================================
# 6. 앙상블 전략 (최고 성능)
# =========================================
def ensemble_comovement_pairs(pivot, item_meta, train):
    """여러 방법을 조합한 앙상블 전략"""

    pivot_value, pivot_weight, item_meta = create_enhanced_pivot(train)

    # 방법 1: 고급 지표
    pairs1 = find_comovement_pairs_advanced(
        pivot_value, pivot_weight, item_meta,
        corr_threshold=0.35, min_pvalue=0.1
    )
    pairs1['method'] = 'advanced'

    # 방법 2: Top-K
    pairs2 = find_comovement_pairs_topk(
        pivot_value, item_meta,
        top_k_per_leader=15
    )
    pairs2['method'] = 'topk'

    # 방법 3: 보수적 고상관 (baseline 개선)
    pairs3 = find_comovement_pairs_advanced(
        pivot_value, pivot_weight, item_meta,
        corr_threshold=0.5,  # 높은 임계값
        min_pvalue=0.05,
        use_granger=False,
        use_mi=False
    )
    pairs3['method'] = 'conservative'

    # 합치고 중복 제거 (leading_item_id, following_item_id 기준)
    all_pairs = pd.concat([pairs1, pairs2, pairs3], ignore_index=True)

    # 같은 쌍이 여러 방법에서 나온 경우, 가장 높은 상관계수 유지
    all_pairs = all_pairs.sort_values('max_corr', ascending=False)
    all_pairs = all_pairs.drop_duplicates(
        subset=['leading_item_id', 'following_item_id'],
        keep='first'
    )

    return all_pairs

# 앙상블 실행
# pairs_ensemble = ensemble_comovement_pairs(pivot, item_meta, train)

공행성쌍 탐색: 100%|██████████| 100/100 [01:06<00:00,  1.50it/s]

탐색된 공행성쌍 수: 2057
     leading_item_id following_item_id  best_lag  max_corr  combined_score  \
1264        QRKRBYJL          DNMPSKTB         6  0.779804        0.901627   
108         ATLDMDBO          QRKRBYJL         1  0.817712        0.869543   
1253        QRKRBYJL          ATLDMDBO         2  0.780910        0.838904   
94          ATLDMDBO          DNMPSKTB         6  0.677985        0.824037   
1460        SAHWCZNH          LUENUFGA         4  0.714034        0.807373   
1280        QRKRBYJL          QVLMOEYE         2  0.844230        0.787423   
446         DNMPSKTB          QRKRBYJL         4  0.729112        0.784726   
1466        SAHWCZNH          WPQXWHYO         1 -0.676991        0.782717   
2008        ZKENOUDA          DEWLVASR         5  0.799071        0.771442   
1912        XUOIQPFL          QVLMOEYE         6  0.833496        0.767782   

            pvalue  granger_score  mi_score  same_hs2  
1264  3.104157e-08       0.779804  1.057927         1  
108   1.9021

In [ ]:
pairs['max_corr']

,max_corr
1264,0.779804
108,0.817712
1253,0.780910
94,0.677985
1460,0.714034
...,...
623,-0.282247
1044,0.401526
9,-0.330820
1271,-0.333139


In [ ]:
# 절대값 컬럼 생성
pairs['abs_corr'] = pairs['max_corr'].abs()

# 절대값 기준 상위 20% 컷오프
abs_top_20_cutoff = pairs['abs_corr'].quantile(0.80)

# 추출
strong_corr_df = pairs[pairs['abs_corr'] >= abs_top_20_cutoff]
print(f"절대값 기준 상위 20% (강한 관계): {len(strong_corr_df)}개")

절대값 기준 상위 20% (강한 관계): 412개


In [ ]:
pairs

,leading_item_id,following_item_id,best_lag,max_corr,combined_score,pvalue,granger_score,mi_score,same_hs2,abs_corr
1264,QRKRBYJL,DNMPSKTB,6,0.779804,0.901627,3.104157e-08,0.779804,1.057927,1,0.779804
108,ATLDMDBO,QRKRBYJL,1,0.817712,0.869543,1.902198e-05,0.817712,1.016772,1,0.817712
1253,QRKRBYJL,ATLDMDBO,2,0.780910,0.838904,2.830659e-06,0.780910,0.772918,1,0.780910
94,ATLDMDBO,DNMPSKTB,6,0.677985,0.824037,2.117084e-06,0.677983,0.965489,1,0.677985
1460,SAHWCZNH,LUENUFGA,4,0.714034,0.807373,4.369639e-07,0.714033,0.674348,1,0.714034
...,...,...,...,...,...,...,...,...,...,...
623,GKQIJYDH,EVBVXETX,6,-0.282247,0.350433,4.548950e-02,0.000000,0.728113,0,0.282247
1044,NZKBIBNU,AANGBULD,3,0.401526,0.350287,8.944802e-02,0.397419,0.436062,0,0.401526
9,AANGBULD,LUENUFGA,6,-0.330820,0.350228,3.258378e-02,0.315765,0.401632,0,0.330820
1271,QRKRBYJL,JERHKLYW,1,-0.333139,0.350203,2.768048e-02,0.322780,0.482263,0,0.333139


### 상관계수

In [ ]:
import pandas as pd

# 1. 전체 평균 상관계수 구하기
mean_corr = pairs['abs_corr'].mean()
print(f"- 전체 평균 상관계수: {mean_corr:.4f}")

# 2. 분위수(Quantile) 계산
# 상위 20% 기준값 (80% 지점)
top_20_cutoff = pairs['abs_corr'].quantile(0.80)

# 하위 20% 기준값 (20% 지점)
bottom_20_cutoff = pairs['abs_corr'].quantile(0.20)

print(f"- 상위 20% 커트라인: {top_20_cutoff:.4f} 이상")
print(f"- 하위 20% 커트라인: {bottom_20_cutoff:.4f} 이하")

# 3. 데이터 추출 (Filtering)
top_20_df = pairs[pairs['abs_corr'] >= top_20_cutoff]
bottom_20_df = pairs[pairs['abs_corr'] <= bottom_20_cutoff]

# 4. 결과 출력
print(f"\n[상위 20% 데이터] - 총 {len(top_20_df)}개")
# 상위 5개만 확인
print(top_20_df[['leading_item_id', 'following_item_id', 'best_lag', 'abs_corr']].head())

print(f"\n[하위 20% 데이터] - 총 {len(bottom_20_df)}개")
# 하위 5개만 확인
print(bottom_20_df[['leading_item_id', 'following_item_id', 'best_lag', 'abs_corr']].head())

KeyError: 'abs_corr'

### 가설1 분석
- 점수: 0.3337024676
- 공행성 쌍 개수: 2057
- 전체 평균 상관계수: 0.5252
- 상위 20% 커트라인: 0.5708 이상
- 하위 20% 커트라인: 0.4695 이하
---
- 너무 많은 저품질 쌍이 포함되어 평균 성능이 떨어짐
- 평균 상관계수 0.131 (baseline 대비 53% 감소)
- 음의 상관계수까지 포함됨 (25% = -0.358)
- 공행성쌍이 증가한 것에 비해, 저품질쌍이 많이 관찰된 상태여서 평균 성능이 저하됨.
- 수량보다는 품질에 집중한 코드 구성이 적절해 보임

# 가설 2. Baseline 기반으로 다양한 전략을 앙상블하는 것이 최적 결과에 유리하다.
- **쌍 개수를 늘리는 전략**
    - 타당한 근거를 뒷받침하여서 쌍 개수를 늘려야 함
    - 현재 0.3 이상 나온 결과물들의 평균 쌍 개수가 2200개인 것을 고려
- **Baseline 로직 최대한 유지, 미세 튜닝 추가를 목표로 함**
- **6가지 전략 시도 (앙상블 적용)**
    1. baseline 유지
    2. 상관계수 조합 개선 (Pearson + Spearman)
        - Pearson과 Spearman을 교차로 사용하는 전략
    3. hs2 필터 추가
        - use_enhanced_corr: 다른 산업군에 높은 임계값 부여
    4. 임계값 하향 조정 (더 많은 쌍)
        - 공행성쌍을 늘리기 위해 임계값을 낮춤
    5. lag 범위 확대
        1. max_lag=8로 설정하여 더 긴 시차 관계 포착 (공행성쌍 늘림)
    6. min_nonzero 완화
        1. min_nonzero=10으로 완화

### 가설2 검증 및 결과

In [ ]:
# =========================================
# Baseline 기반 미세 개선 (0.4+ 목표)
# 전략: 쌍 개수 유지 + 품질만 살짝 개선
# =========================================
import pandas as pd
import numpy as np
from tqdm import tqdm
from scipy.stats import spearmanr


# =========================================
# 개선 1: 안전한 상관계수 (Baseline 유지)
# =========================================
def safe_corr(x, y):
    if np.std(x) == 0 or np.std(y) == 0:
        return 0.0
    return float(np.corrcoef(x, y)[0, 1])


# =========================================
# 개선 2: 다중 지표 상관계수 (선택적)
# =========================================
def enhanced_corr(x, y, use_spearman=True):
    """
    Pearson + Spearman 조합으로 더 강건한 상관계수
    """
    pearson = safe_corr(x, y)

    if not use_spearman:
        return pearson

    # Spearman 추가
    try:
        spearman, _ = spearmanr(x, y)
        if np.isnan(spearman):
            spearman = 0.0
    except:
        spearman = 0.0

    # 조합: Pearson 70% + Spearman 30%
    combined = 0.7 * pearson + 0.3 * spearman

    return combined


# =========================================
# 개선 3: hs2/hs4 메타 정보 추가
# =========================================
def get_item_meta(train):
    """품목별 메타 정보 추출"""
    train['hs2'] = train['hs4'] // 100

    meta = train.groupby('item_id').agg({
        'hs4': 'first',
        'hs2': 'first',
        'value': 'mean'  # 평균 거래액
    }).to_dict('index')

    return meta


# =========================================
# 핵심: Baseline + 미세 개선
# =========================================
def find_comovement_pairs(
    pivot,
    item_meta=None,
    max_lag=6,
    min_nonzero=12,
    corr_threshold=0.4,
    use_enhanced_corr=True,
    use_hs2_filter=False,
    hs2_same_bonus=0.05,
):
    """
    Baseline 로직 + 선택적 개선

    개선 포인트:
    1. enhanced_corr: Pearson + Spearman 조합
    2. hs2_same_bonus: 같은 산업군에 보너스
    3. use_hs2_filter: 다른 산업군 간 높은 임계값 적용
    """

    items = pivot.index.to_list()
    months = pivot.columns.to_list()
    n_months = len(months)

    results = []

    for i, leader in tqdm(enumerate(items), total=len(items)):
        x = pivot.loc[leader].values.astype(float)
        if np.count_nonzero(x) < min_nonzero:
            continue

        # leader 메타 정보
        leader_hs2 = item_meta.get(leader, {}).get('hs2', None) if item_meta else None

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)
            if np.count_nonzero(y) < min_nonzero:
                continue

            # follower 메타 정보
            follower_hs2 = item_meta.get(follower, {}).get('hs2', None) if item_meta else None

            # hs2 같은지 체크
            same_hs2 = (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2)

            best_lag = None
            best_corr = 0.0

            # lag 탐색
            for lag in range(1, max_lag + 1):
                if n_months <= lag:
                    continue

                # 상관계수 계산
                if use_enhanced_corr:
                    corr = enhanced_corr(x[:-lag], y[lag:], use_spearman=True)
                else:
                    corr = safe_corr(x[:-lag], y[lag:])

                # 같은 산업군이면 보너스
                if same_hs2 and hs2_same_bonus > 0:
                    corr += hs2_same_bonus

                if abs(corr) > abs(best_corr):
                    best_corr = corr
                    best_lag = lag

            # 임계값 적용
            threshold = corr_threshold

            # 다른 산업군이면 더 높은 임계값 요구 (선택적)
            if use_hs2_filter and not same_hs2:
                threshold += 0.05

            # 채택
            if best_lag is not None and abs(best_corr) >= threshold:
                results.append({
                    "leading_item_id": leader,
                    "following_item_id": follower,
                    "best_lag": best_lag,
                    "max_corr": best_corr,
                })

    pairs = pd.DataFrame(results)
    return pairs


# =========================================
# 전략별 시도
# =========================================

# 전략 1: Baseline 그대로 (벤치마크)
print("[전략 1] Baseline")
pairs_baseline = find_comovement_pairs(
    pivot,
    item_meta=None,
    max_lag=6,
    min_nonzero=12,
    corr_threshold=0.4,
    use_enhanced_corr=False,
    use_hs2_filter=False
)
print(f"쌍 수: {len(pairs_baseline)}, 평균 상관: {pairs_baseline['max_corr'].mean():.4f}")


# 전략 2: Enhanced Correlation (Pearson + Spearman)
print("\n[전략 2] Enhanced Correlation")
item_meta = get_item_meta(train)

pairs_enhanced = find_comovement_pairs(
    pivot,
    item_meta=item_meta,
    max_lag=6,
    min_nonzero=12,
    corr_threshold=0.38,  # 약간 낮춤 (보너스로 보상)
    use_enhanced_corr=True,
    use_hs2_filter=False,
    hs2_same_bonus=0.05
)
print(f"쌍 수: {len(pairs_enhanced)}, 평균 상관: {pairs_enhanced['max_corr'].mean():.4f}")


# 전략 3: hs2 필터 추가
print("\n[전략 3] hs2 필터")
pairs_filtered = find_comovement_pairs(
    pivot,
    item_meta=item_meta,
    max_lag=6,
    min_nonzero=12,
    corr_threshold=0.38,
    use_enhanced_corr=True,
    use_hs2_filter=True,  # 다른 산업군에 높은 임계값
    hs2_same_bonus=0.05
)
print(f"쌍 수: {len(pairs_filtered)}, 평균 상관: {pairs_filtered['max_corr'].mean():.4f}")


# 전략 4: 임계값 하향 조정 (더 많은 쌍)
print("\n[전략 4] 낮은 임계값 (0.35)")
pairs_lower = find_comovement_pairs(
    pivot,
    item_meta=item_meta,
    max_lag=6,
    min_nonzero=12,
    corr_threshold=0.35,  # 더 낮춤
    use_enhanced_corr=True,
    use_hs2_filter=False,
    hs2_same_bonus=0.05
)
print(f"쌍 수: {len(pairs_lower)}, 평균 상관: {pairs_lower['max_corr'].mean():.4f}")


# 전략 5: lag 범위 확대
print("\n[전략 5] lag 확대 (최대 8)")
pairs_extended = find_comovement_pairs(
    pivot,
    item_meta=item_meta,
    max_lag=8,  # 8개월까지
    min_nonzero=12,
    corr_threshold=0.38,
    use_enhanced_corr=True,
    use_hs2_filter=False,
    hs2_same_bonus=0.05
)
print(f"쌍 수: {len(pairs_extended)}, 평균 상관: {pairs_extended['max_corr'].mean():.4f}")


# 전략 6: min_nonzero 완화
print("\n[전략 6] min_nonzero 완화 (10)")
pairs_relaxed = find_comovement_pairs(
    pivot,
    item_meta=item_meta,
    max_lag=6,
    min_nonzero=10,  # 10개로 완화
    corr_threshold=0.38,
    use_enhanced_corr=True,
    use_hs2_filter=False,
    hs2_same_bonus=0.05
)
print(f"쌍 수: {len(pairs_relaxed)}, 평균 상관: {pairs_relaxed['max_corr'].mean():.4f}")


# =========================================
# 최종 추천 조합
# =========================================
print("\n" + "="*60)
print("[최종 추천] 여러 개선 조합")
print("="*60)

item_meta = get_item_meta(train)

pairs = find_comovement_pairs(
    pivot,
    item_meta=item_meta,
    max_lag=7,  # 약간 확대
    min_nonzero=11,  # 약간 완화
    corr_threshold=0.37,  # 약간 낮춤
    use_enhanced_corr=True,  # Pearson + Spearman
    use_hs2_filter=False,
    hs2_same_bonus=0.06  # 보너스 증가
)

print(f"\n탐색된 공행성쌍 수: {len(pairs)}")
print(f"평균 상관계수: {pairs['max_corr'].mean():.4f}")
print(f"최소 상관계수: {pairs['max_corr'].min():.4f}")
print(f"최대 상관계수: {pairs['max_corr'].max():.4f}")


# =========================================
# 추가: 앙상블 투표
# =========================================
def ensemble_voting(train, pivot, strategies):
    """
    여러 전략의 교집합/합집합 활용
    """
    item_meta = get_item_meta(train)

    all_pairs_dict = {}

    for name, params in strategies.items():
        print(f"\n{name} 실행 중...")
        pairs = find_comovement_pairs(pivot, item_meta=item_meta, **params)

        for _, row in pairs.iterrows():
            key = (row['leading_item_id'], row['following_item_id'])
            if key not in all_pairs_dict:
                all_pairs_dict[key] = {
                    'leading_item_id': row['leading_item_id'],
                    'following_item_id': row['following_item_id'],
                    'best_lag': row['best_lag'],
                    'max_corr': row['max_corr'],
                    'vote_count': 0,
                    'avg_corr': []
                }

            all_pairs_dict[key]['vote_count'] += 1
            all_pairs_dict[key]['avg_corr'].append(row['max_corr'])

    # 투표 수 기준 필터링 (2개 이상 전략에서 선택된 쌍만)
    final_pairs = []
    for key, info in all_pairs_dict.items():
        if info['vote_count'] >= 2:  # 최소 2개 전략에서 선택
            info['avg_corr'] = np.mean(info['avg_corr'])
            final_pairs.append(info)

    result = pd.DataFrame(final_pairs)

    if len(result) > 0:
        result = result.sort_values('avg_corr', ascending=False)

    return result


# 앙상블 사용 예시
strategies = {
    'strategy1': {
        'max_lag': 6, 'min_nonzero': 12, 'corr_threshold': 0.38,
        'use_enhanced_corr': True, 'hs2_same_bonus': 0.05
    },
    'strategy2': {
        'max_lag': 7, 'min_nonzero': 11, 'corr_threshold': 0.36,
        'use_enhanced_corr': True, 'hs2_same_bonus': 0.06
    },
    'strategy3': {
        'max_lag': 8, 'min_nonzero': 10, 'corr_threshold': 0.35,
        'use_enhanced_corr': True, 'hs2_same_bonus': 0.07
    },
}

pairs_ensemble = ensemble_voting(train, pivot, strategies)

[전략 1] Baseline


100%|██████████| 100/100 [00:05<00:00, 19.86it/s]


쌍 수: 1425, 평균 상관: 0.2789

[전략 2] Enhanced Correlation


100%|██████████| 100/100 [00:24<00:00,  4.09it/s]


쌍 수: 1431, 평균 상관: 0.1980

[전략 3] hs2 필터


...

100%|██████████| 100/100 [00:23<00:00,  4.26it/s]



strategy2 실행 중...


100%|██████████| 100/100 [00:27<00:00,  3.65it/s]



strategy3 실행 중...


100%|██████████| 100/100 [00:31<00:00,  3.16it/s]


### 상관계수

In [ ]:
pairs_ensemble

,leading_item_id,following_item_id,best_lag,max_corr,vote_count,avg_corr
863,QRKRBYJL,DNMPSKTB,6,0.823814,3,0.869666
72,ATLDMDBO,QRKRBYJL,1,0.799637,3,0.809637
856,QRKRBYJL,ATLDMDBO,2,0.789703,3,0.799703
877,QRKRBYJL,QVLMOEYE,2,0.799550,3,0.799550
1339,XUOIQPFL,QVLMOEYE,6,0.793404,3,0.793404
...,...,...,...,...,...,...
328,ELQGMQWE,OKMBFVKS,2,-0.674755,3,-0.674755
1053,SDWAYPIK,ZKENOUDA,3,-0.688699,3,-0.688699
1207,VUAFAIYJ,LSOIUSXD,3,-0.692778,3,-0.692778
453,GYHKIVQT,ZKENOUDA,5,-0.705010,3,-0.705010


In [ ]:
import pandas as pd

pairs_ensemble['abs_corr'] = pairs_ensemble['max_corr'].abs()

# 1. 전체 평균 상관계수 구하기
mean_corr = pairs_ensemble['abs_corr'].mean()
print(f"- 전체 평균 상관계수: {mean_corr:.4f}")

# 2. 분위수(Quantile) 계산
# 상위 20% 기준값 (80% 지점)
top_20_cutoff = pairs_ensemble['abs_corr'].quantile(0.80)

# 하위 20% 기준값 (20% 지점)
bottom_20_cutoff = pairs_ensemble['abs_corr'].quantile(0.20)

print(f"- 상위 20% 커트라인: {top_20_cutoff:.4f} 이상")
print(f"- 하위 20% 커트라인: {bottom_20_cutoff:.4f} 이하")

# 3. 데이터 추출 (Filtering)
top_20_df = pairs_ensemble[pairs_ensemble['abs_corr'] >= top_20_cutoff]
bottom_20_df = pairs_ensemble[pairs_ensemble['abs_corr'] <= bottom_20_cutoff]

# 4. 결과 출력
print(f"\n[상위 20% 데이터] - 총 {len(top_20_df)}개")
# 상위 5개만 확인
print(top_20_df[['leading_item_id', 'following_item_id', 'best_lag', 'abs_corr']].head())

print(f"\n[하위 20% 데이터] - 총 {len(bottom_20_df)}개")
# 하위 5개만 확인
print(bottom_20_df[['leading_item_id', 'following_item_id', 'best_lag', 'abs_corr']].head())

- 전체 평균 상관계수: 0.4502
- 상위 20% 커트라인: 0.5092 이상
- 하위 20% 커트라인: 0.3825 이하

[상위 20% 데이터] - 총 388개
     leading_item_id following_item_id  best_lag  abs_corr
863         QRKRBYJL          DNMPSKTB         6  0.823814
72          ATLDMDBO          QRKRBYJL         1  0.799637
856         QRKRBYJL          ATLDMDBO         2  0.789703
877         QRKRBYJL          QVLMOEYE         2  0.799550
1339        XUOIQPFL          QVLMOEYE         6  0.793404

[하위 20% 데이터] - 총 388개
     leading_item_id following_item_id  best_lag  abs_corr
1596        GYHKIVQT          PYZMVUWD         6  0.374515
1710        OGAFEHLU          WBLJNPZQ         6  0.375972
1931        ZGJXVMNI          ZXERAXWP         3  0.360927
1720        OKMBFVKS          QRKRBYJL         5  0.371088
1687        MBSBZBXA          ZKENOUDA         6  0.364833


### 가설2 분석
- 점수: 0.3380845634
- 공행성 쌍 개수: 1754
- 전체 평균 상관계수: 0.4502
- 상위 20% 커트라인: 0.5092 이상
- 하위 20% 커트라인: 0.3825 이하
---
- 가장 높은 점수를 보였지만, 여전히 0.34 이상의 수치를 보여주지 못함
- 공행성쌍이 1500~3000 사이에서 0.3 이상의 점수를 보일 가능성이 높아졌다.
- 그러나 점수가 급등하지 않는 것으로 보아, 더욱 근본적인 접근이 필요하다고 판단


# 가설3. COVID-19와 계절성이 큰 영향을 줄 것이다.
- **계절성**
    - max_lag를 12까지 크게 잡자.
    - 3개월(분기), 6개월(반기), 12개월(연간) 주기에 보너스
    - 무역 데이터의 계절적 패턴 포착하는 방향으로 접근
- **COVID 19**
    - 무역 데이터가 22~25년까지 이므로, COVID가 줄어드는 시기인 23년 1월을 기점으로 잡자
        - 즉, Post-COVID 데이터(2023~현재)에 집중
    - 정상화된 소비 패턴 반영
- **최근 데이터에 보너스**
    - 최신 트렌드 중시
    - 최근 데이터에 지수적 가중치 (decay_rate=0.95)
- **변동성 매칭**
    - 변동 계수(CV)가 유사한 품목 쌍에 보너스
- **추세 제거**
    - Detrend를 통해 장기 추세 제거
    - 단기 변동 패턴에 집중
- **다중 lag 조합**
    - (1-6), (3-9), (6-12) 범위를 모두 탐색
    - 상관계수 고려, 최적 lag 범위 자동 선택
- **최종) 앙상블 (품질과 다양성 균형)**
    - 최소 2개 전략에서 동의한 쌍만 선택
    - 상관계수 가중 평균으로 최종 점수 계산

### 가설3 검증 및 결과

In [ ]:
# =========================================
# 고급 전략: 계절성 + COVID-19 고려 + 다양한 lag
# 목표: 0.4+ 달성
# =========================================
import pandas as pd
import numpy as np
from tqdm import tqdm
from scipy.stats import spearmanr, kendalltau
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')


# =========================================
# 기본 함수들
# =========================================
def safe_corr(x, y):
    if np.std(x) == 0 or np.std(y) == 0:
        return 0.0
    return float(np.corrcoef(x, y)[0, 1])


def enhanced_corr(x, y, use_spearman=True, use_kendall=False):
    """다중 상관계수 조합"""
    pearson = safe_corr(x, y)

    weights = [0.7]  # Pearson
    scores = [pearson]

    if use_spearman:
        try:
            spearman, _ = spearmanr(x, y)
            if not np.isnan(spearman):
                scores.append(spearman)
                weights.append(0.2)
        except:
            pass

    if use_kendall:
        try:
            kendall, _ = kendalltau(x, y)
            if not np.isnan(kendall):
                scores.append(kendall)
                weights.append(0.1)
        except:
            pass

    # 정규화
    weights = np.array(weights)
    weights = weights / weights.sum()

    combined = sum(s * w for s, w in zip(scores, weights))
    return combined


def get_item_meta(train):
    """품목별 메타 정보"""
    train['hs2'] = train['hs4'] // 100

    meta = train.groupby('item_id').agg({
        'hs4': 'first',
        'hs2': 'first',
        'value': ['mean', 'std']
    }).to_dict('index')

    return meta


# =========================================
# 전략 1: 계절성 고려 (12개월 lag)
# =========================================
def find_comovement_seasonal(
    pivot,
    item_meta=None,
    max_lag=12,
    min_nonzero=12,
    corr_threshold=0.35,
    seasonal_lags=[3, 6, 12],  # 분기, 반기, 연간
    seasonal_bonus=0.03
):
    """
    계절성 lag에 보너스 부여
    3개월(분기), 6개월(반기), 12개월(연간) 주기
    """
    items = pivot.index.to_list()
    months = pivot.columns.to_list()
    n_months = len(months)
    results = []

    for leader in tqdm(items, desc="계절성 전략"):
        x = pivot.loc[leader].values.astype(float)
        if np.count_nonzero(x) < min_nonzero:
            continue

        leader_hs2 = item_meta.get(leader, {}).get('hs2', None) if item_meta else None

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)
            if np.count_nonzero(y) < min_nonzero:
                continue

            follower_hs2 = item_meta.get(follower, {}).get('hs2', None) if item_meta else None
            same_hs2 = (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2)

            best_lag = None
            best_corr = 0.0

            for lag in range(1, min(max_lag + 1, n_months)):
                if n_months <= lag:
                    continue

                corr = enhanced_corr(x[:-lag], y[lag:], use_spearman=True)

                # 계절성 lag에 보너스
                if lag in seasonal_lags:
                    corr += seasonal_bonus

                # 같은 산업군 보너스
                if same_hs2:
                    corr += 0.05

                if abs(corr) > abs(best_corr):
                    best_corr = corr
                    best_lag = lag

            if best_lag is not None and abs(best_corr) >= corr_threshold:
                results.append({
                    "leading_item_id": leader,
                    "following_item_id": follower,
                    "best_lag": best_lag,
                    "max_corr": best_corr,
                })

    return pd.DataFrame(results)


# =========================================
# 전략 2: COVID-19 기간 분리 분석
# =========================================
def find_comovement_covid_aware(
    pivot,
    item_meta=None,
    max_lag=6,
    min_nonzero=8,  # 기간 분리로 완화
    corr_threshold=0.35,
    covid_cutoff='2023-01'
):
    """
    COVID-19 영향 고려
    - 2022~2023년 초: COVID 기간 (특수 소비 패턴)
    - 2023년 이후: Post-COVID 기간 (정상화)
    두 기간을 분리하여 상관관계 계산
    """
    items = pivot.index.to_list()
    months = pivot.columns.to_list()

    # 날짜 분리
    cutoff_date = pd.to_datetime(covid_cutoff)
    covid_mask = [pd.to_datetime(m) < cutoff_date for m in months]
    post_covid_mask = [pd.to_datetime(m) >= cutoff_date for m in months]

    results = []

    for leader in tqdm(items, desc="COVID-aware 전략"):
        x = pivot.loc[leader].values.astype(float)

        leader_hs2 = item_meta.get(leader, {}).get('hs2', None) if item_meta else None

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            follower_hs2 = item_meta.get(follower, {}).get('hs2', None) if item_meta else None
            same_hs2 = (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2)

            best_lag = None
            best_corr = 0.0

            # Post-COVID 기간 중심으로 분석 (더 최근 데이터)
            x_post = x[post_covid_mask]
            y_post = y[post_covid_mask]

            if np.count_nonzero(x_post) < min_nonzero or np.count_nonzero(y_post) < min_nonzero:
                # Post-COVID 데이터가 부족하면 전체 기간 사용
                x_use = x
                y_use = y
            else:
                x_use = x_post
                y_use = y_post

            n_months_use = len(x_use)

            for lag in range(1, min(max_lag + 1, n_months_use)):
                if n_months_use <= lag:
                    continue

                corr = enhanced_corr(x_use[:-lag], y_use[lag:], use_spearman=True)

                # 같은 산업군 보너스
                if same_hs2:
                    corr += 0.05

                if abs(corr) > abs(best_corr):
                    best_corr = corr
                    best_lag = lag

            if best_lag is not None and abs(best_corr) >= corr_threshold:
                results.append({
                    "leading_item_id": leader,
                    "following_item_id": follower,
                    "best_lag": best_lag,
                    "max_corr": best_corr,
                })

    return pd.DataFrame(results)


# =========================================
# 전략 3: 가중 상관계수 (최근 데이터 중시)
# =========================================
def find_comovement_weighted(
    pivot,
    item_meta=None,
    max_lag=8,
    min_nonzero=12,
    corr_threshold=0.35,
    decay_rate=0.95  # 지수 감쇠율
):
    """
    최근 데이터에 더 높은 가중치
    """
    items = pivot.index.to_list()
    months = pivot.columns.to_list()
    n_months = len(months)
    results = []

    for leader in tqdm(items, desc="가중 상관 전략"):
        x = pivot.loc[leader].values.astype(float)
        if np.count_nonzero(x) < min_nonzero:
            continue

        leader_hs2 = item_meta.get(leader, {}).get('hs2', None) if item_meta else None

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)
            if np.count_nonzero(y) < min_nonzero:
                continue

            follower_hs2 = item_meta.get(follower, {}).get('hs2', None) if item_meta else None
            same_hs2 = (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2)

            best_lag = None
            best_corr = 0.0

            for lag in range(1, min(max_lag + 1, n_months)):
                if n_months <= lag:
                    continue

                x_lag = x[:-lag]
                y_shift = y[lag:]

                # 지수 가중치 (최근일수록 높음)
                n = len(x_lag)
                weights = np.array([decay_rate ** (n - i - 1) for i in range(n)])
                weights = weights / weights.sum()

                # 가중 상관계수 계산
                if np.std(x_lag) > 0 and np.std(y_shift) > 0:
                    x_mean = np.average(x_lag, weights=weights)
                    y_mean = np.average(y_shift, weights=weights)

                    cov = np.sum(weights * (x_lag - x_mean) * (y_shift - y_mean))
                    std_x = np.sqrt(np.sum(weights * (x_lag - x_mean)**2))
                    std_y = np.sqrt(np.sum(weights * (y_shift - y_mean)**2))

                    corr = cov / (std_x * std_y + 1e-10)
                else:
                    corr = 0.0

                # 같은 산업군 보너스
                if same_hs2:
                    corr += 0.05

                if abs(corr) > abs(best_corr):
                    best_corr = corr
                    best_lag = lag

            if best_lag is not None and abs(best_corr) >= corr_threshold:
                results.append({
                    "leading_item_id": leader,
                    "following_item_id": follower,
                    "best_lag": best_lag,
                    "max_corr": best_corr,
                })

    return pd.DataFrame(results)


# =========================================
# 전략 4: 변동성 매칭
# =========================================
def find_comovement_volatility_match(
    pivot,
    item_meta=None,
    max_lag=6,
    min_nonzero=12,
    corr_threshold=0.35,
    vol_similarity_bonus=0.05
):
    """
    변동성이 유사한 품목 쌍에 보너스
    """
    items = pivot.index.to_list()
    months = pivot.columns.to_list()
    n_months = len(months)
    results = []

    for leader in tqdm(items, desc="변동성 매칭 전략"):
        x = pivot.loc[leader].values.astype(float)
        if np.count_nonzero(x) < min_nonzero:
            continue

        cv_x = np.std(x) / (np.mean(x) + 1e-8)  # 변동 계수
        leader_hs2 = item_meta.get(leader, {}).get('hs2', None) if item_meta else None

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)
            if np.count_nonzero(y) < min_nonzero:
                continue

            cv_y = np.std(y) / (np.mean(y) + 1e-8)
            vol_similarity = 1.0 / (1.0 + abs(cv_x - cv_y))  # 유사도

            follower_hs2 = item_meta.get(follower, {}).get('hs2', None) if item_meta else None
            same_hs2 = (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2)

            best_lag = None
            best_corr = 0.0

            for lag in range(1, min(max_lag + 1, n_months)):
                if n_months <= lag:
                    continue

                corr = enhanced_corr(x[:-lag], y[lag:], use_spearman=True)

                # 변동성 유사도 보너스
                corr += vol_similarity_bonus * vol_similarity

                # 같은 산업군 보너스
                if same_hs2:
                    corr += 0.05

                if abs(corr) > abs(best_corr):
                    best_corr = corr
                    best_lag = lag

            if best_lag is not None and abs(best_corr) >= corr_threshold:
                results.append({
                    "leading_item_id": leader,
                    "following_item_id": follower,
                    "best_lag": best_lag,
                    "max_corr": best_corr,
                })

    return pd.DataFrame(results)


# =========================================
# 전략 5: 추세 제거 후 상관분석
# =========================================
def find_comovement_detrended(
    pivot,
    item_meta=None,
    max_lag=6,
    min_nonzero=12,
    corr_threshold=0.35
):
    """
    추세를 제거한 후 상관관계 분석
    단기 변동 패턴에 집중
    """
    from scipy.signal import detrend

    items = pivot.index.to_list()
    months = pivot.columns.to_list()
    n_months = len(months)
    results = []

    for leader in tqdm(items, desc="추세 제거 전략"):
        x = pivot.loc[leader].values.astype(float)
        if np.count_nonzero(x) < min_nonzero:
            continue

        # 추세 제거
        try:
            x_detrend = detrend(x)
        except:
            x_detrend = x

        leader_hs2 = item_meta.get(leader, {}).get('hs2', None) if item_meta else None

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)
            if np.count_nonzero(y) < min_nonzero:
                continue

            try:
                y_detrend = detrend(y)
            except:
                y_detrend = y

            follower_hs2 = item_meta.get(follower, {}).get('hs2', None) if item_meta else None
            same_hs2 = (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2)

            best_lag = None
            best_corr = 0.0

            for lag in range(1, min(max_lag + 1, n_months)):
                if n_months <= lag:
                    continue

                corr = enhanced_corr(x_detrend[:-lag], y_detrend[lag:], use_spearman=True)

                if same_hs2:
                    corr += 0.05

                if abs(corr) > abs(best_corr):
                    best_corr = corr
                    best_lag = lag

            if best_lag is not None and abs(best_corr) >= corr_threshold:
                results.append({
                    "leading_item_id": leader,
                    "following_item_id": follower,
                    "best_lag": best_lag,
                    "max_corr": best_corr,
                })

    return pd.DataFrame(results)


# =========================================
# 전략 6: 다중 lag 조합
# =========================================
def find_comovement_multi_lag(
    pivot,
    item_meta=None,
    lag_ranges=[(1, 6), (3, 9), (6, 12)],
    min_nonzero=12,
    corr_threshold=0.35
):
    """
    여러 lag 범위에서 최적값 탐색
    """
    all_results = []

    for lag_min, lag_max in lag_ranges:
        items = pivot.index.to_list()
        months = pivot.columns.to_list()
        n_months = len(months)

        for leader in tqdm(items, desc=f"Multi-lag ({lag_min}-{lag_max})"):
            x = pivot.loc[leader].values.astype(float)
            if np.count_nonzero(x) < min_nonzero:
                continue

            leader_hs2 = item_meta.get(leader, {}).get('hs2', None) if item_meta else None

            for follower in items:
                if follower == leader:
                    continue

                y = pivot.loc[follower].values.astype(float)
                if np.count_nonzero(y) < min_nonzero:
                    continue

                follower_hs2 = item_meta.get(follower, {}).get('hs2', None) if item_meta else None
                same_hs2 = (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2)

                best_lag = None
                best_corr = 0.0

                for lag in range(lag_min, min(lag_max + 1, n_months)):
                    if n_months <= lag:
                        continue

                    corr = enhanced_corr(x[:-lag], y[lag:], use_spearman=True)

                    if same_hs2:
                        corr += 0.05

                    if abs(corr) > abs(best_corr):
                        best_corr = corr
                        best_lag = lag

                if best_lag is not None and abs(best_corr) >= corr_threshold:
                    all_results.append({
                        "leading_item_id": leader,
                        "following_item_id": follower,
                        "best_lag": best_lag,
                        "max_corr": best_corr,
                    })

    df = pd.DataFrame(all_results)
    # 중복 제거 (같은 쌍은 최고 점수만)
    if len(df) > 0:
        df = df.sort_values('max_corr', ascending=False)
        df = df.drop_duplicates(subset=['leading_item_id', 'following_item_id'], keep='first')

    return df


# =========================================
# 메인: 모든 전략 실행 및 앙상블
# =========================================
def run_all_strategies(train, pivot):
    """
    모든 전략 실행 및 비교
    """
    item_meta = get_item_meta(train)

    print("="*70)
    print("다양한 전략 실행 중...")
    print("="*70)

    strategies_results = {}

    # 전략 1: 계절성 (12개월 lag)
    print("\n[전략 1] 계절성 고려 (lag 최대 12)")
    pairs1 = find_comovement_seasonal(pivot, item_meta, max_lag=12, corr_threshold=0.35)
    strategies_results['seasonal'] = pairs1
    print(f"쌍 수: {len(pairs1)}, 평균 상관: {pairs1['max_corr'].mean() if len(pairs1) > 0 else 0}")

    # 전략 2: COVID-aware
    print("\n[전략 2] COVID-19 기간 분리")
    pairs2 = find_comovement_covid_aware(pivot, item_meta, corr_threshold=0.35)
    strategies_results['covid_aware'] = pairs2
    print(f"쌍 수: {len(pairs2)}, 평균 상관: {pairs2['max_corr'].mean() if len(pairs2) > 0 else 0}")

    # 전략 3: 가중 상관
    print("\n[전략 3] 가중 상관계수 (최근 중시)")
    pairs3 = find_comovement_weighted(pivot, item_meta, max_lag=8, corr_threshold=0.35)
    strategies_results['weighted'] = pairs3
    print(f"쌍 수: {len(pairs3)}, 평균 상관: {pairs3['max_corr'].mean() if len(pairs3) > 0 else 0}")

    # 전략 4: 변동성 매칭
    print("\n[전략 4] 변동성 매칭")
    pairs4 = find_comovement_volatility_match(pivot, item_meta, corr_threshold=0.35)
    strategies_results['volatility'] = pairs4
    print(f"쌍 수: {len(pairs4)}, 평균 상관: {pairs4['max_corr'].mean() if len(pairs4) > 0 else 0}")

    # 전략 5: 추세 제거
    print("\n[전략 5] 추세 제거")
    pairs5 = find_comovement_detrended(pivot, item_meta, corr_threshold=0.35)
    strategies_results['detrended'] = pairs5
    print(f"쌍 수: {len(pairs5)}, 평균 상관: {pairs5['max_corr'].mean() if len(pairs5) > 0 else 0}")

    # 전략 6: 다중 lag
    print("\n[전략 6] 다중 lag 조합")
    pairs6 = find_comovement_multi_lag(pivot, item_meta, lag_ranges=[(1, 6), (3, 9), (6, 12)], corr_threshold=0.35)
    strategies_results['multi_lag'] = pairs6
    print(f"쌍 수: {len(pairs6)}, 평균 상관: {pairs6['max_corr'].mean() if len(pairs6) > 0 else 0}")

    return strategies_results


# =========================================
# 고급 앙상블: 투표 + 가중 평균
# =========================================
def advanced_ensemble(strategies_results, min_votes=2, weight_by_corr=True):
    """
    여러 전략의 결과를 투표 방식으로 앙상블 (수정됨)
    """
    all_pairs_dict = {}

    for strategy_name, pairs_df in strategies_results.items():
        if len(pairs_df) == 0:
            continue

        for _, row in pairs_df.iterrows():
            key = (row['leading_item_id'], row['following_item_id'])

            if key not in all_pairs_dict:
                all_pairs_dict[key] = {
                    'leading_item_id': row['leading_item_id'],
                    'following_item_id': row['following_item_id'],
                    'lags': [],
                    'corrs': [],
                    'strategies': [],
                    'vote_count': 0
                }

            all_pairs_dict[key]['lags'].append(row['best_lag'])
            all_pairs_dict[key]['corrs'].append(row['max_corr'])
            all_pairs_dict[key]['strategies'].append(strategy_name)
            all_pairs_dict[key]['vote_count'] += 1

    # 필터링 및 최종 값 계산
    final_pairs = []
    for key, info in all_pairs_dict.items():
        if info['vote_count'] >= min_votes:
            # 가장 많이 선택된 lag
            best_lag = max(set(info['lags']), key=info['lags'].count)

            # 상관계수 평균 (가중 평균 옵션)
            if weight_by_corr:
                raw_corrs = np.array(info['corrs'])
                # 수정: 가중치는 '절댓값'을 기준으로 해야 함 (강한 상관관계일수록 높은 비중)
                weights = np.abs(raw_corrs)

                # 모든 상관계수가 0인 경우 방지 (거의 없겠지만)
                if weights.sum() == 0:
                    avg_corr = 0.0
                else:
                    weights = weights / weights.sum()
                    avg_corr = np.sum(raw_corrs * weights)
            else:
                avg_corr = np.mean(info['corrs'])

            final_pairs.append({
                'leading_item_id': info['leading_item_id'],
                'following_item_id': info['following_item_id'],
                'best_lag': best_lag,
                'max_corr': avg_corr,
                'vote_count': info['vote_count'],
                'strategies': ','.join(info['strategies'])
            })

    result = pd.DataFrame(final_pairs)

    if len(result) > 0:
        result = result.sort_values('max_corr', ascending=False)

    return result


# =========================================
# 실행
# =========================================
item_meta = get_item_meta(train)

# 모든 전략 실행
strategies_results = run_all_strategies(train, pivot)

# 앙상블 (최소 2개 전략 동의)
print("\n" + "="*70)
print("앙상블 결과 (최소 2개 전략 동의)")
print("="*70)

pairs_ensemble = advanced_ensemble(strategies_results, min_votes=2, weight_by_corr=True)
print(f"\n최종 쌍 수: {len(pairs_ensemble)}")
if len(pairs_ensemble) > 0:
    print(f"평균 상관: {pairs_ensemble['max_corr'].mean():.4f}")
    print(f"최소 상관: {pairs_ensemble['max_corr'].min():.4f}")
    print(f"최대 상관: {pairs_ensemble['max_corr'].max():.4f}")
    print("\nlag 분포:")
    print(pairs_ensemble['best_lag'].value_counts().sort_index())

# 최종 선택: 가장 좋은 단일 전략 또는 앙상블
pairs = pairs_ensemble

print("\n" + "="*70)
print("각 전략별 쌍 개수 요약:")
for name, df in strategies_results.items():
    print(f"{name}: {len(df)}개")
print(f"앙상블: {len(pairs_ensemble)}개")
print("="*70)

다양한 전략 실행 중...

[전략 1] 계절성 고려 (lag 최대 12)


계절성 전략: 100%|██████████| 100/100 [01:38<00:00,  1.02it/s]


쌍 수: 3432, 평균 상관: 0.192488949326394

[전략 2] COVID-19 기간 분리


COVID-aware 전략: 100%|██████████| 100/100 [01:00<00:00,  1.67it/s]


쌍 수: 3419, 평균 상관: 0.10062273547469375

[전략 3] 가중 상관계수 (최근 중시)


...

쌍 수: 2080, 평균 상관: 0.2924525970102296

[전략 5] 추세 제거


추세 제거 전략: 100%|██████████| 100/100 [00:51<00:00,  1.93it/s]


쌍 수: 1558, 평균 상관: 0.14131246976051784

[전략 6] 다중 lag 조합


Multi-lag (6-12): 100%|██████████| 100/100 [00:59<00:00,  1.67it/s]


쌍 수: 3339, 평균 상관: 0.18272009406940581

앙상블 결과 (최소 2개 전략 동의)

최종 쌍 수: 4051
평균 상관: 0.1645
최소 상관: -0.6887
최대 상관: 0.7904

lag 분포:
best_lag
1     470
2     422
3     415
4     362
5     377
6     420
7     190
8     314
9     260
10    256
11    255
12    310
Name: count, dtype: int64

각 전략별 쌍 개수 요약:
seasonal: 3432개
covid_aware: 3419개
weighted: 2972개
volatility: 2080개
detrended: 1558개
multi_lag: 3339개
앙상블: 4051개


### 상관계수

In [ ]:
pairs_ensemble

,leading_item_id,following_item_id,best_lag,max_corr,vote_count,strategies,abs_corr
2129,QRKRBYJL,QVLMOEYE,2,0.790375,5,"seasonal,weighted,volatility,detrended,multi_lag",0.790375
1755,NAQIHUKZ,LLHREMKS,3,0.783002,6,"seasonal,covid_aware,weighted,volatility,detre...",0.783002
3043,XIIEJNEE,DJBLNPNC,5,0.781865,5,"seasonal,weighted,volatility,detrended,multi_lag",0.781865
1750,NAQIHUKZ,FTSVTTSR,1,0.779850,6,"seasonal,covid_aware,weighted,volatility,detre...",0.779850
3329,ZKENOUDA,DEWLVASR,5,0.770472,6,"seasonal,covid_aware,weighted,volatility,detre...",0.770472
...,...,...,...,...,...,...,...
743,ELQGMQWE,OKMBFVKS,2,-0.659715,6,"seasonal,covid_aware,weighted,volatility,detre...",0.659715
1086,GYHKIVQT,ZKENOUDA,5,-0.660159,6,"seasonal,covid_aware,weighted,volatility,detre...",0.660159
701,DNMPSKTB,ZKENOUDA,6,-0.662696,5,"seasonal,covid_aware,weighted,volatility,multi...",0.662696
1046,GYHKIVQT,EVBVXETX,6,-0.685771,6,"seasonal,covid_aware,weighted,volatility,detre...",0.685771


In [ ]:
import pandas as pd

pairs_ensemble['abs_corr'] = pairs_ensemble['max_corr'].abs()

# 1. 전체 평균 상관계수 구하기
mean_corr = pairs_ensemble['abs_corr'].mean()
print(f"- 전체 평균 상관계수: {mean_corr:.4f}")

# 2. 분위수(Quantile) 계산
# 상위 20% 기준값 (80% 지점)
top_20_cutoff = pairs_ensemble['abs_corr'].quantile(0.80)

# 하위 20% 기준값 (20% 지점)
bottom_20_cutoff = pairs_ensemble['abs_corr'].quantile(0.20)

print(f"- 상위 20% 커트라인: {top_20_cutoff:.4f} 이상")
print(f"- 하위 20% 커트라인: {bottom_20_cutoff:.4f} 이하")

# 3. 데이터 추출 (Filtering)
top_20_df = pairs_ensemble[pairs_ensemble['abs_corr'] >= top_20_cutoff]
bottom_20_df = pairs_ensemble[pairs_ensemble['abs_corr'] <= bottom_20_cutoff]

# 4. 결과 출력
print(f"\n[상위 20% 데이터] - 총 {len(top_20_df)}개")
# 상위 5개만 확인
print(top_20_df[['leading_item_id', 'following_item_id', 'best_lag', 'abs_corr']].head())

print(f"\n[하위 20% 데이터] - 총 {len(bottom_20_df)}개")
# 하위 5개만 확인
print(bottom_20_df[['leading_item_id', 'following_item_id', 'best_lag', 'abs_corr']].head())

- 전체 평균 상관계수: 0.4050
- 상위 20% 커트라인: 0.4894 이상
- 하위 20% 커트라인: 0.3657 이하

[상위 20% 데이터] - 총 811개
     leading_item_id following_item_id  best_lag  abs_corr
2129        QRKRBYJL          QVLMOEYE         2  0.790375
1755        NAQIHUKZ          LLHREMKS         3  0.783002
3043        XIIEJNEE          DJBLNPNC         5  0.781865
1750        NAQIHUKZ          FTSVTTSR         1  0.779850
3329        ZKENOUDA          DEWLVASR         5  0.770472

[하위 20% 데이터] - 총 811개
     leading_item_id following_item_id  best_lag  abs_corr
3680        LSOIUSXD          BTMOEMEP         5  0.365746
1231        JBVHSUWY          UXSPKBJR        11  0.365683
1332        JSLXRQOK          JERHKLYW        12  0.365660
798         FCYBOAXC          ATLDMDBO         5  0.365608
814         FCYBOAXC          PYZMVUWD         8  0.365607


### 가설3 분석
- 점수: 0.325993085
- 공행성 쌍 개수: 4052
- 전체 평균 상관계수: 0.4050
- 상위 20% 커트라인: 0.4894 이상
- 하위 20% 커트라인: 0.3657 이하
---
- 공행성쌍이 이전에 비해 급등했고, 점수는 0.3 이상이지만 baseline 점수와 별반 차이가 없다.
- 정답 공행성쌍을 다수 찾았지만, 그만큼 오답 공행성쌍도 함께 발견된 것으로 판단
- 최대한 파라미터의 조정 및 비교를 동원해서 결과를 더욱 다양히 봐야함
- 간혹 거래가 되지 않아 value=0인 거래들이 존재하는데, 이에 대한 처리도 필요하지 않을까?

# 가설4. 파라미터 조정 범위를 늘리고, value=0인 거래에 대한 처리가 결과의 핵심일 것이다.
- **Value=0인 거래에 대한 처리**
    - `keep`: 그대로 유지
    - `interpolate`: 선형 보간
    - `forward_fill`: 앞 값으로 채움
    - `mean_fill`: 평균값으로 채움
    - `small_value`: 아주 작은 값으로 대체
    - 다른 방법) 그래프 상에서 튀는 값이 존재하여 해당 항목 제거 → 오히려 성능 저하
- **데이터 변환**
    - `none`: 변환 없음
    - `log`: log(x+1) 변환
    - `sqrt`: 제곱근 변환
    - `standardize`: 표준화
    - `minmax`: 정규화
- **상관계수**
    - `pearson`: 기본
    - `spearman`: 순위 기반
    - `pearson_spearman`: 조합
    - `abs_pearson`: 절댓값
- **튜닝 최적화**
    - 빠른 그리드 서치
        - 핵심 파라미터 조합 테스트
    - 정밀 튜닝
        - 최적값 주변 ±α 범위 탐색
    - 앙상블
        - 상위 5개 조합의 교집합
    - GPU 가속 (CuPy)
        - Colab 환경의 A100에서 실행 (2시간 가량 소요)
    - 멀티프로세싱
    - 메모리 최적화

### 가설4 검증 및 결과

In [ ]:
# =========================================
# GPU 가속 하이퍼파라미터 그리드 서치
# CuPy + 병렬 처리
# =========================================
import pandas as pd
import numpy as np
from tqdm import tqdm
from scipy.stats import spearmanr
from itertools import product
import warnings
warnings.filterwarnings('ignore')

# GPU 사용 가능 여부 확인 및 설정
try:
    import cupy as cp
    GPU_AVAILABLE = True
    print("✓ GPU(CuPy) 사용 가능")
except ImportError:
    print("✗ CuPy 없음 - CPU로 실행됩니다")
    print("  설치: pip install cupy-cuda11x (또는 cuda12x)")
    GPU_AVAILABLE = False
    cp = np

# 멀티프로세싱
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
import multiprocessing as mp


# =========================================
# GPU 가속 함수들
# =========================================
def to_gpu(array):
    """NumPy 배열을 GPU로 전송"""
    if GPU_AVAILABLE:
        return cp.asarray(array)
    return array


def to_cpu(array):
    """GPU 배열을 CPU로 전송"""
    if GPU_AVAILABLE and isinstance(array, cp.ndarray):
        return cp.asnumpy(array)
    return array


def gpu_corrcoef(x, y):
    """GPU 가속 상관계수 계산"""
    if GPU_AVAILABLE:
        x = cp.asarray(x)
        y = cp.asarray(y)

        if cp.std(x) == 0 or cp.std(y) == 0:
            return 0.0

        corr = cp.corrcoef(x, y)[0, 1]
        return float(cp.asnumpy(corr))
    else:
        if np.std(x) == 0 or np.std(y) == 0:
            return 0.0
        return float(np.corrcoef(x, y)[0, 1])


def gpu_batch_correlations(X_batch, Y_batch, lags):
    """
    배치 단위로 여러 상관계수를 한번에 계산 (GPU)
    X_batch: (n_pairs, n_months)
    Y_batch: (n_pairs, n_months)
    lags: (n_pairs,)
    """
    if not GPU_AVAILABLE:
        # CPU 폴백
        results = []
        for i in range(len(X_batch)):
            x = X_batch[i]
            y = Y_batch[i]
            lag = lags[i]
            if len(x) > lag:
                corr = gpu_corrcoef(x[:-lag], y[lag:])
                results.append(corr)
            else:
                results.append(0.0)
        return results

    # GPU 계산
    X_batch = cp.asarray(X_batch)
    Y_batch = cp.asarray(Y_batch)

    results = []
    for i in range(len(X_batch)):
        x = X_batch[i]
        y = Y_batch[i]
        lag = lags[i]

        if len(x) > lag:
            x_lag = x[:-lag]
            y_shift = y[lag:]

            if cp.std(x_lag) > 0 and cp.std(y_shift) > 0:
                corr = cp.corrcoef(x_lag, y_shift)[0, 1]
                results.append(float(cp.asnumpy(corr)))
            else:
                results.append(0.0)
        else:
            results.append(0.0)

    return results


# =========================================
# 0값 처리 및 변환 (벡터화)
# =========================================
def preprocess_zero_values(series, method='keep'):
    """0값 처리"""
    series = np.array(series, dtype=float)

    if method == 'keep':
        return series

    elif method == 'interpolate':
        series_temp = series.copy()
        series_temp[series_temp == 0] = np.nan
        mask = ~np.isnan(series_temp)
        if mask.sum() > 1:
            indices = np.arange(len(series_temp))
            series_temp = np.interp(indices, indices[mask], series_temp[mask])
        return series_temp

    elif method == 'forward_fill':
        series_temp = series.copy()
        mask = series_temp != 0
        idx = np.where(mask, np.arange(len(series_temp)), 0)
        np.maximum.accumulate(idx, out=idx)
        return series_temp[idx]

    elif method == 'mean_fill':
        mean_val = series[series > 0].mean() if np.sum(series > 0) > 0 else 0
        series_temp = series.copy()
        series_temp[series_temp == 0] = mean_val
        return series_temp

    return series


def apply_transformation(series, transform='none'):
    """데이터 변환"""
    series = np.array(series, dtype=float)

    if transform == 'none':
        return series
    elif transform == 'log':
        return np.log1p(series)
    elif transform == 'sqrt':
        return np.sqrt(np.abs(series))
    elif transform == 'standardize':
        if np.std(series) > 0:
            return (series - np.mean(series)) / np.std(series)
        return series

    return series


def get_item_meta(train):
    """품목별 메타 정보"""
    train['hs2'] = train['hs4'] // 100
    meta = train.groupby('item_id').agg({
        'hs4': 'first',
        'hs2': 'first'
    }).to_dict('index')
    return meta


# =========================================
# GPU 가속 공행성쌍 탐색 (병렬화)
# =========================================
def find_comovement_single_leader(args):
    """
    단일 leader에 대한 공행성쌍 탐색 (병렬 처리용)
    """
    leader, pivot_dict, items, item_meta, params = args

    max_lag = params['max_lag']
    min_nonzero = params['min_nonzero']
    corr_threshold = params['corr_threshold']
    zero_handling = params['zero_handling']
    transform = params['transform']
    hs2_same_bonus = params['hs2_same_bonus']

    results = []

    # Leader 데이터 처리
    x = np.array(pivot_dict[leader], dtype=float)
    x = preprocess_zero_values(x, method=zero_handling)
    x = apply_transformation(x, transform=transform)

    if np.count_nonzero(x) < min_nonzero:
        return results

    leader_hs2 = item_meta.get(leader, {}).get('hs2', None) if item_meta else None
    n_months = len(x)

    # GPU로 전송 (가능하면)
    x_gpu = to_gpu(x)

    for follower in items:
        if follower == leader:
            continue

        y = np.array(pivot_dict[follower], dtype=float)
        y = preprocess_zero_values(y, method=zero_handling)
        y = apply_transformation(y, transform=transform)

        if np.count_nonzero(y) < min_nonzero:
            continue

        follower_hs2 = item_meta.get(follower, {}).get('hs2', None) if item_meta else None
        same_hs2 = (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2)

        y_gpu = to_gpu(y)

        best_lag = None
        best_corr = 0.0

        # 모든 lag에 대해 계산
        for lag in range(1, min(max_lag + 1, n_months)):
            if n_months <= lag:
                continue

            corr = gpu_corrcoef(x_gpu[:-lag], y_gpu[lag:])

            # Spearman 추가 (선택적)
            if params.get('use_spearman', True):
                try:
                    x_cpu = to_cpu(x_gpu[:-lag])
                    y_cpu = to_cpu(y_gpu[lag:])
                    spearman, _ = spearmanr(x_cpu, y_cpu)
                    if not np.isnan(spearman):
                        corr = 0.7 * corr + 0.3 * spearman
                except:
                    pass

            if same_hs2:
                corr += hs2_same_bonus

            if abs(corr) > abs(best_corr):
                best_corr = corr
                best_lag = lag

        if best_lag is not None and abs(best_corr) >= corr_threshold:
            results.append({
                "leading_item_id": leader,
                "following_item_id": follower,
                "best_lag": best_lag,
                "max_corr": best_corr,
            })

    return results


def find_comovement_pairs_gpu(
    pivot,
    item_meta=None,
    params=None,
    n_jobs=-1
):
    """
    GPU + 멀티프로세싱으로 가속화된 공행성쌍 탐색
    """

    if params is None:
        params = {
            'max_lag': 6,
            'min_nonzero': 12,
            'corr_threshold': 0.4,
            'zero_handling': 'keep',
            'transform': 'none',
            'hs2_same_bonus': 0.05,
            'use_spearman': True,
        }

    items = pivot.index.to_list()

    # pivot을 dict로 변환 (직렬화 가능)
    pivot_dict = {item: pivot.loc[item].values for item in items}

    # 병렬 처리 준비
    if n_jobs == -1:
        n_jobs = mp.cpu_count()

    args_list = [
        (leader, pivot_dict, items, item_meta, params)
        for leader in items
    ]

    # 멀티프로세싱 실행
    all_results = []

    with ProcessPoolExecutor(max_workers=n_jobs) as executor:
        # tqdm으로 진행상황 표시
        for results in tqdm(
            executor.map(find_comovement_single_leader, args_list),
            total=len(args_list),
            desc="GPU 가속 탐색"
        ):
            all_results.extend(results)

    return pd.DataFrame(all_results)


# =========================================
# GPU 가속 그리드 서치
# =========================================
def grid_search_gpu(train, pivot, param_grid=None, top_k=10, n_jobs=-1):
    """
    GPU 가속 그리드 서치
    """

    if param_grid is None:
        param_grid = {
            'max_lag': [6, 8, 10],
            'min_nonzero': [11, 12],
            'corr_threshold': [0.36, 0.38, 0.40],
            'zero_handling': ['keep', 'interpolate'],
            'transform': ['none', 'log'],
            'hs2_same_bonus': [0.05, 0.06],
            'use_spearman': [True],
        }

    item_meta = get_item_meta(train)

    # 모든 조합 생성
    keys = list(param_grid.keys())
    values = list(param_grid.values())
    combinations = list(product(*values))

    print(f"총 {len(combinations)}개 조합 테스트")
    print(f"CPU 코어: {mp.cpu_count()}개")
    print(f"GPU: {'사용' if GPU_AVAILABLE else '미사용'}")
    print("="*70)

    results = []

    for i, combo in enumerate(tqdm(combinations, desc="그리드 서치")):
        params = dict(zip(keys, combo))

        try:
            pairs = find_comovement_pairs_gpu(
                pivot,
                item_meta=item_meta,
                params=params,
                n_jobs=n_jobs
            )

            if len(pairs) > 0:
                result_info = {
                    'combo_id': i,
                    'num_pairs': len(pairs),
                    'mean_corr': pairs['max_corr'].mean(),
                    'min_corr': pairs['max_corr'].min(),
                    'max_corr': pairs['max_corr'].max(),
                    'std_corr': pairs['max_corr'].std(),
                    'pairs': pairs,
                    **params
                }
                results.append(result_info)

                # 중간 결과 출력
                print(f"\n조합 {i}: 쌍={len(pairs)}, 평균상관={pairs['max_corr'].mean():.4f}")

        except Exception as e:
            print(f"\n조합 {i} 실패: {e}")
            continue

    # 결과 정리
    results_df = pd.DataFrame([
        {k: v for k, v in r.items() if k != 'pairs'}
        for r in results
    ])

    print("\n" + "="*70)
    print(f"성공한 조합 수: {len(results_df)}")
    print("="*70)

    if len(results_df) > 0:
        results_df['score'] = results_df['num_pairs'] * results_df['mean_corr']
        results_df = results_df.sort_values('score', ascending=False)

        print(f"\n상위 {top_k}개 조합:")
        print("-"*70)
        for idx in range(min(top_k, len(results_df))):
            row = results_df.iloc[idx]
            print(f"\n순위 {idx+1}:")
            print(f"  쌍 개수: {row['num_pairs']}")
            print(f"  평균 상관: {row['mean_corr']:.4f}")
            print(f"  점수: {row['score']:.2f}")
            print(f"  파라미터:")
            print(f"    max_lag={row['max_lag']}, min_nonzero={row['min_nonzero']}")
            print(f"    corr_threshold={row['corr_threshold']}")
            print(f"    zero_handling={row['zero_handling']}")
            print(f"    transform={row['transform']}")

    return results, results_df


# =========================================
# 대규모 그리드 서치 (GPU 버전)
# =========================================
def large_grid_search_gpu(train, pivot, n_jobs=-1):
    """
    대규모 파라미터 조합 탐색
    """

    param_grid = {
        'max_lag': [6, 7, 8, 9, 10, 12],
        'min_nonzero': [10, 11, 12],
        'corr_threshold': [0.33, 0.35, 0.37, 0.38, 0.40, 0.42],
        'zero_handling': ['keep', 'interpolate', 'forward_fill', 'mean_fill'],
        'transform': ['none', 'log', 'sqrt'],
        'hs2_same_bonus': [0.0, 0.05, 0.07],
        'use_spearman': [True, False],
    }

    print(f"총 조합 수: {np.prod([len(v) for v in param_grid.values()])}개")

    return grid_search_gpu(train, pivot, param_grid, top_k=20, n_jobs=n_jobs)


# =========================================
# 실행
# =========================================

print("="*70)
print("GPU 가속 하이퍼파라미터 그리드 서치")
print("="*70)

# 빠른 그리드 서치
print("\n[빠른 그리드 서치]")
results, results_df = grid_search_gpu(train, pivot, n_jobs=-1)

# 최적 조합 선택
if len(results) > 0:
    best_combo = results[0]
    pairs = best_combo['pairs']

    print("\n" + "="*70)
    print("[최적 조합]")
    print("="*70)
    print(f"쌍 개수: {len(pairs)}")
    print(f"평균 상관: {pairs['max_corr'].mean():.4f}")
    print("\n통계:")
    print(pairs['max_corr'].describe())
    print("\nlag 분포:")
    print(pairs['best_lag'].value_counts().sort_index())

else:
    print("결과 없음")
    pairs = pd.DataFrame()


# =========================================
# 대규모 그리드 서치 (주석 해제하여 실행)
# =========================================
print("\n" + "="*70)
print("[옵션] 대규모 그리드 서치")
print("="*70)
print("더 많은 조합 테스트:")
print("# large_results, large_df = large_grid_search_gpu(train, pivot, n_jobs=-1)")


# =========================================
# GPU 메모리 정리
# =========================================
if GPU_AVAILABLE:
    print("\nGPU 메모리 정리...")
    cp.get_default_memory_pool().free_all_blocks()
    print("완료")

✗ CuPy 없음 - CPU로 실행됩니다
  설치: pip install cupy-cuda11x (또는 cuda12x)
GPU 가속 하이퍼파라미터 그리드 서치

[빠른 그리드 서치]
총 144개 조합 테스트
CPU 코어: 24개
GPU: 미사용


그리드 서치:   1%|          | 1/144 [00:09<22:02,  9.25s/it]


조합 0: 쌍=1743, 평균상관=0.1825



그리드 서치:   1%|▏         | 2/144 [00:18<22:14,  9.39s/it]


조합 1: 쌍=1754, 평균상관=0.1853


...


조합 141: 쌍=2414, 평균상관=0.1881



그리드 서치:  99%|█████████▉| 143/144 [32:06<00:17, 17.40s/it]


조합 142: 쌍=2430, 평균상관=0.0713



그리드 서치: 100%|██████████| 144/144 [32:24<00:00, 13.51s/it]


조합 143: 쌍=2437, 평균상관=0.0740

성공한 조합 수: 144

상위 10개 조합:
----------------------------------------------------------------------

순위 1:
  쌍 개수: 3344
  평균 상관: 0.1553
  점수: 519.46
  파라미터:
    max_lag=10, min_nonzero=11
    corr_threshold=0.36
    zero_handling=interpolate
    transform=none

순위 2:
  쌍 개수: 3344
  평균 상관: 0.1553
  점수: 519.46
  파라미터:
    max_lag=10, min_nonzero=12
    corr_threshold=0.36
    zero_handling=interpolate
    transform=none

순위 3:
  쌍 개수: 3334
  평균 상관: 0.1526
  점수: 508.72
  파라미터:
    max_lag=10, min_nonzero=12
    corr_threshold=0.36
    zero_handling=interpolate
    transform=none

순위 4:
  쌍 개수: 3334
  평균 상관: 0.1526
  점수: 508.72
  파라미터:
    max_lag=10, min_nonzero=11
    corr_threshold=0.36
    zero_handling=interpolate
    transform=none

순위 5:
  쌍 개수: 2841
  평균 상관: 0.1712
  점수: 486.47
  파라미터:
    max_lag=10, min_nonzero=11
    corr_threshold=0.38
    zero_handling=interpolate
    transform=none

순위 6:
  쌍 개수: 2841
  평균 상관: 0.1712
  점수: 486.47
  파라미터:
    max_lag

### 상관계수

In [ ]:
pairs

,leading_item_id,following_item_id,best_lag,max_corr,combined_score,pvalue,granger_score,mi_score,same_hs2
1264,QRKRBYJL,DNMPSKTB,6,0.779804,0.901627,3.104157e-08,0.779804,1.057927,1
108,ATLDMDBO,QRKRBYJL,1,0.817712,0.869543,1.902198e-05,0.817712,1.016772,1
1253,QRKRBYJL,ATLDMDBO,2,0.780910,0.838904,2.830659e-06,0.780910,0.772918,1
94,ATLDMDBO,DNMPSKTB,6,0.677985,0.824037,2.117084e-06,0.677983,0.965489,1
1460,SAHWCZNH,LUENUFGA,4,0.714034,0.807373,4.369639e-07,0.714033,0.674348,1
...,...,...,...,...,...,...,...,...,...
623,GKQIJYDH,EVBVXETX,6,-0.282247,0.350433,4.548950e-02,0.000000,0.728113,0
1044,NZKBIBNU,AANGBULD,3,0.401526,0.350287,8.944802e-02,0.397419,0.436062,0
9,AANGBULD,LUENUFGA,6,-0.330820,0.350228,3.258378e-02,0.315765,0.401632,0
1271,QRKRBYJL,JERHKLYW,1,-0.333139,0.350203,2.768048e-02,0.322780,0.482263,0


In [ ]:
import pandas as pd

pairs['abs_corr'] = pairs['max_corr'].abs()

# 1. 전체 평균 상관계수 구하기
mean_corr = pairs['abs_corr'].mean()
print(f"- 전체 평균 상관계수: {mean_corr:.4f}")

# 2. 분위수(Quantile) 계산
# 상위 20% 기준값 (80% 지점)
top_20_cutoff = pairs['abs_corr'].quantile(0.80)

# 하위 20% 기준값 (20% 지점)
bottom_20_cutoff = pairs['abs_corr'].quantile(0.20)

print(f"- 상위 20% 커트라인: {top_20_cutoff:.4f} 이상")
print(f"- 하위 20% 커트라인: {bottom_20_cutoff:.4f} 이하")

# 3. 데이터 추출 (Filtering)
top_20_df = pairs[pairs['abs_corr'] >= top_20_cutoff]
bottom_20_df = pairs[pairs['abs_corr'] <= bottom_20_cutoff]

# 4. 결과 출력
print(f"\n[상위 20% 데이터] - 총 {len(top_20_df)}개")
# 상위 5개만 확인
print(top_20_df[['leading_item_id', 'following_item_id', 'best_lag', 'abs_corr']].head())

print(f"\n[하위 20% 데이터] - 총 {len(bottom_20_df)}개")
# 하위 5개만 확인
print(bottom_20_df[['leading_item_id', 'following_item_id', 'best_lag', 'abs_corr']].head())

- 전체 평균 상관계수: 0.4353
- 상위 20% 커트라인: 0.5092 이상
- 하위 20% 커트라인: 0.3517 이하

[상위 20% 데이터] - 총 412개
     leading_item_id following_item_id  best_lag  abs_corr
1264        QRKRBYJL          DNMPSKTB         6  0.779804
108         ATLDMDBO          QRKRBYJL         1  0.817712
1253        QRKRBYJL          ATLDMDBO         2  0.780910
94          ATLDMDBO          DNMPSKTB         6  0.677985
1460        SAHWCZNH          LUENUFGA         4  0.714034

[하위 20% 데이터] - 총 412개
     leading_item_id following_item_id  best_lag  abs_corr
1289        QRKRBYJL          VWMBASNE         5  0.348222
2025        ZKENOUDA          QRKRBYJL         3  0.351588
458         DNMPSKTB          WPQXWHYO         1  0.318028
719         HXYSSRXE          QRKRBYJL         5  0.350903
1889        XMKRPGLB          XIPPENFQ         5  0.341127


### 가설4 분석
- 점수: 0.3384513693
- 공행성 쌍 개수: 1744
- 전체 평균 상관계수: 0.4353
- 상위 20% 커트라인: 0.5092 이상
- 하위 20% 커트라인: 0.3517 이하
---
- 가설의 일부에 대한 그리드 서칭에 걸린 시간이 오래 걸렸고, 전체 그리드 서치를 하는데는 더욱 걸릴 것으로 보임
- GPU를 사용하여 들인 컴퓨팅 시간에 비해 점수 상승폭이 매우 낮음
- COVID-19에 의한 요인에 집중했는데, 오히려 여기에 연관이 적은 건 아닐까?
    - hs2와 hs4에서 해당 요인에 의해 영향을 받을 만한 품목이 있는가
- 다른 외부적 요인 혹은 내부적 탐색 알고리즘은 없을까?

# 가설5. COVID-19 요인에 대해 실효성을 분석하고, 새로운 외부적 요인/내부 알고리즘을 찾아야 성능이 오를 것이다.
- **COVID-19에 영향을 받은 hs코드**
    - **HS 30**: 의약품 (Pharmaceuticals) - 백신, 치료제
    - **HS 38**: 화학제품 - 소독제, 손세정제
    - **HS 40**: 고무제품 - 장갑
    - **HS 48**: 종이제품 - 종이 마스크
    - **HS 63**: 섬유제품 - 보호복, 마스크
    - **HS 90**: 의료기기 - 인공호흡기, 체온계, 진단기기
- **그러나 위 품목들은 22~23년 초에만 급증하고, 이후 정상화**
    - COVID 효과는 이미 2년 이상 지남
    - 2025년 8월 예측이 목표인 만큼, 현재로썬 유의미하지 않다고 판단
- **점수가 급등하지 않는 이유 분석**
    - 문제 1: 선형 상관계수의 한계
        - 현재 사용한 Pearson correlation은 선형 관계만 포착 → 무역 데이터는 비선형, 복잡한 패턴 사용
    - 문제 2: 단순 Lag 탐색
        - 고정된 lag 값 사용 → 시간에 따라 변하는 관계 못 잡음
    - 문제 3: 쌍별 독립 분석
        - A→B만 보고 A→C→B 같은 연쇄 관계는 고려하지 않음
- **새로운 전략**
    - Dynamic Time Warping (DTW)
        - ‘시간 왜곡을 허용한 형태 유사도’를 통해 시간 지연 패턴 포착
    - 모든 lag에서 상관계수 계산
    - Granger Causality
        - X가 Y 예측에 실제로 **도움이 되는지** 측정
        - 단순 상관 vs 인과관계 구분
    - Shape-based Matching
        - 추세와 변동 패턴의 **형태 유사성**
        - 변화 방향의 일치도
    - 연쇄 효과
        - A→C→B 등의 관계
    - 위 방법들에 대한 앙상블 진행

### 가설5 검증 및 결과

In [ ]:
# =========================================
# 혁신적 접근: 네트워크 + 비선형 + 동적 lag
# 목표: 0.4+ 달성
# =========================================
import pandas as pd
import numpy as np
from tqdm import tqdm
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')


# =========================================
# 전략 1: 비선형 유사도 측정 (DTW)
# =========================================
def dtw_distance(x, y, window=3):
    """
    Dynamic Time Warping 거리
    시계열의 형태 유사성 측정 (시간 왜곡 허용)
    """
    n, m = len(x), len(y)

    # 정규화
    x = (x - np.mean(x)) / (np.std(x) + 1e-8)
    y = (y - np.mean(y)) / (np.std(y) + 1e-8)

    # DTW 행렬
    dtw = np.full((n + 1, m + 1), np.inf)
    dtw[0, 0] = 0

    for i in range(1, n + 1):
        for j in range(max(1, i - window), min(m + 1, i + window + 1)):
            cost = abs(x[i-1] - y[j-1])
            dtw[i, j] = cost + min(dtw[i-1, j], dtw[i, j-1], dtw[i-1, j-1])

    return dtw[n, m]


def dtw_similarity(x, y, window=3):
    """DTW 거리를 유사도로 변환 (0~1)"""
    dist = dtw_distance(x, y, window)
    # 거리를 유사도로: 1 / (1 + dist)
    return 1.0 / (1.0 + dist)


# =========================================
# 전략 2: 교차 상관함수 (CCF)
# =========================================
def cross_correlation_function(x, y, max_lag=12):
    """
    모든 lag에 대한 상관관계 계산
    가장 높은 상관계수와 해당 lag 반환
    """
    x = (x - np.mean(x)) / (np.std(x) + 1e-8)
    y = (y - np.mean(y)) / (np.std(y) + 1e-8)

    correlations = []

    for lag in range(1, min(max_lag + 1, len(x))):
        if len(x) <= lag:
            break

        corr = np.corrcoef(x[:-lag], y[lag:])[0, 1]
        correlations.append((lag, corr))

    if not correlations:
        return 0, 0.0

    # 절댓값이 가장 큰 상관계수 선택
    best_lag, best_corr = max(correlations, key=lambda t: abs(t[1]))

    return best_lag, best_corr


# =========================================
# 전략 3: Granger Causality Test (간소화)
# =========================================
def granger_causality_score(x, y, max_lag=6):
    """
    X가 Y를 예측하는데 도움이 되는지 측정
    """
    best_score = 0.0
    best_lag = 1

    for lag in range(1, min(max_lag + 1, len(x) - 5)):
        if len(x) <= lag + 2:
            continue

        # Y의 자기회귀 모델
        y_current = y[lag:]
        y_lagged = y[:-lag]

        # X의 lag 값 추가
        x_lagged = x[:-lag]

        # 단순 상관 기반 Granger score
        corr_y_self = abs(np.corrcoef(y_current, y_lagged)[0, 1])
        corr_x_y = abs(np.corrcoef(x_lagged, y_current)[0, 1])

        # X가 추가 정보를 제공하는 정도
        score = max(0, corr_x_y - corr_y_self * 0.5)

        if score > best_score:
            best_score = score
            best_lag = lag

    return best_lag, best_score


# =========================================
# 전략 4: 네트워크 기반 간접 관계
# =========================================
def find_indirect_relationships(pivot, direct_pairs, max_intermediate=3):
    """
    A→C→B 같은 간접 관계 탐색
    직접 관계가 약해도 중간 매개가 강하면 추가
    """
    items = pivot.index.to_list()

    # 직접 관계를 그래프로 구성
    graph = {}
    for _, row in direct_pairs.iterrows():
        leader = row['leading_item_id']
        follower = row['following_item_id']

        if leader not in graph:
            graph[leader] = []
        graph[leader].append((follower, row['max_corr']))

    indirect_pairs = []

    # 각 아이템 쌍에 대해 간접 경로 탐색
    for leader in tqdm(items, desc="간접 관계 탐색"):
        if leader not in graph:
            continue

        for follower in items:
            if leader == follower:
                continue

            # BFS로 경로 탐색
            queue = [(leader, [leader], 1.0)]
            visited = {leader}
            found_paths = []

            while queue and len(found_paths) < max_intermediate:
                current, path, score = queue.pop(0)

                if current == follower and len(path) > 2:
                    # 간접 경로 발견
                    found_paths.append((path, score))
                    continue

                if len(path) >= max_intermediate + 1:
                    continue

                if current in graph:
                    for next_item, edge_corr in graph[current]:
                        if next_item not in visited:
                            visited.add(next_item)
                            new_score = score * edge_corr  # 곱으로 경로 강도 계산
                            queue.append((next_item, path + [next_item], new_score))

            # 강한 간접 경로가 있으면 추가
            if found_paths:
                best_path, best_score = max(found_paths, key=lambda t: t[1])

                if best_score > 0.15:  # 임계값
                    # 평균 lag 계산 (경로상 lag 합)
                    total_lag = 0
                    for i in range(len(best_path) - 1):
                        pair = direct_pairs[
                            (direct_pairs['leading_item_id'] == best_path[i]) &
                            (direct_pairs['following_item_id'] == best_path[i+1])
                        ]
                        if not pair.empty:
                            total_lag += pair.iloc[0]['best_lag']

                    avg_lag = total_lag // (len(best_path) - 1)

                    indirect_pairs.append({
                        'leading_item_id': leader,
                        'following_item_id': follower,
                        'best_lag': avg_lag,
                        'max_corr': best_score,
                        'indirect': True,
                        'path': '→'.join(best_path)
                    })

    return pd.DataFrame(indirect_pairs)


# =========================================
# 전략 5: 변동 패턴 매칭 (Shape-based)
# =========================================
def shape_similarity(x, y, window_size=6):
    """
    시계열의 형태 유사도
    추세, 변동 패턴의 유사성
    """
    # 이동 평균으로 스무딩
    x_smooth = pd.Series(x).rolling(window=window_size, min_periods=1).mean().values
    y_smooth = pd.Series(y).rolling(window=window_size, min_periods=1).mean().values

    # 변화율 계산
    x_diff = np.diff(x_smooth)
    y_diff = np.diff(y_smooth)

    # 변화 방향의 일치도
    if len(x_diff) > 0 and len(y_diff) > 0:
        direction_match = np.mean(np.sign(x_diff) == np.sign(y_diff))
    else:
        direction_match = 0

    # DTW 유사도와 결합
    dtw_sim = dtw_similarity(x_smooth, y_smooth, window=3)

    combined = 0.5 * direction_match + 0.5 * dtw_sim

    return combined


# =========================================
# 전략 6: 다중 지표 앙상블
# =========================================
def find_comovement_advanced(
    pivot,
    item_meta=None,
    max_lag=12,
    min_nonzero=11,
    methods=['ccf', 'dtw', 'granger', 'shape'],
    corr_threshold=0.30,  # 낮은 임계값
):
    """
    여러 비선형 지표를 종합한 공행성 탐색
    """
    items = pivot.index.to_list()
    results = []

    for leader in tqdm(items, desc="고급 공행성 탐색"):
        x = pivot.loc[leader].values.astype(float)

        if np.count_nonzero(x) < min_nonzero:
            continue

        leader_hs2 = item_meta.get(leader, {}).get('hs2', None) if item_meta else None

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.count_nonzero(y) < min_nonzero:
                continue

            follower_hs2 = item_meta.get(follower, {}).get('hs2', None) if item_meta else None
            same_hs2 = (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2)

            scores = {}
            lags = {}

            # 1. Cross-Correlation Function
            if 'ccf' in methods:
                lag_ccf, corr_ccf = cross_correlation_function(x, y, max_lag)
                scores['ccf'] = abs(corr_ccf)
                lags['ccf'] = lag_ccf

            # 2. DTW 유사도
            if 'dtw' in methods:
                # 여러 lag 시도
                best_dtw = 0
                best_lag_dtw = 1
                for lag in range(1, min(max_lag + 1, len(x))):
                    if len(x) <= lag:
                        break
                    sim = dtw_similarity(x[:-lag], y[lag:], window=3)
                    if sim > best_dtw:
                        best_dtw = sim
                        best_lag_dtw = lag

                scores['dtw'] = best_dtw
                lags['dtw'] = best_lag_dtw

            # 3. Granger Causality
            if 'granger' in methods:
                lag_granger, score_granger = granger_causality_score(x, y, max_lag)
                scores['granger'] = score_granger
                lags['granger'] = lag_granger

            # 4. Shape Similarity
            if 'shape' in methods:
                best_shape = 0
                best_lag_shape = 1
                for lag in range(1, min(max_lag + 1, len(x))):
                    if len(x) <= lag:
                        break
                    sim = shape_similarity(x[:-lag], y[lag:], window_size=3)
                    if sim > best_shape:
                        best_shape = sim
                        best_lag_shape = lag

                scores['shape'] = best_shape
                lags['shape'] = best_lag_shape

            # 종합 점수
            if scores:
                avg_score = np.mean(list(scores.values()))

                # hs2 보너스
                if same_hs2:
                    avg_score += 0.05

                # 가장 많이 선택된 lag
                lag_values = list(lags.values())
                if lag_values:
                    best_lag = max(set(lag_values), key=lag_values.count)
                else:
                    best_lag = 1

                if avg_score >= corr_threshold:
                    results.append({
                        'leading_item_id': leader,
                        'following_item_id': follower,
                        'best_lag': best_lag,
                        'max_corr': avg_score,
                        **scores  # 개별 점수도 저장
                    })

    return pd.DataFrame(results)


# =========================================
# 최종 통합 전략
# =========================================
def get_item_meta(train):
    train['hs2'] = train['hs4'] // 100
    meta = train.groupby('item_id').agg({
        'hs4': 'first',
        'hs2': 'first'
    }).to_dict('index')
    return meta


print("="*70)
print("혁신적 접근: 비선형 + 네트워크 기반 공행성 탐색")
print("="*70)

item_meta = get_item_meta(train)

# 1단계: 고급 방법으로 직접 관계 탐색
print("\n[1단계] 다중 지표 기반 직접 관계 탐색")
pairs_direct = find_comovement_advanced(
    pivot,
    item_meta=item_meta,
    max_lag=12,
    min_nonzero=11,
    methods=['ccf', 'dtw', 'granger', 'shape'],
    corr_threshold=0.28  # 낮은 임계값
)

print(f"직접 관계: {len(pairs_direct)}개")
if len(pairs_direct) > 0:
    print(f"평균 점수: {pairs_direct['max_corr'].mean():.4f}")

# 2단계: 네트워크 기반 간접 관계
print("\n[2단계] 네트워크 기반 간접 관계 탐색")
if len(pairs_direct) > 50:  # 충분한 직접 관계가 있을 때만
    pairs_indirect = find_indirect_relationships(
        pivot,
        pairs_direct.head(500),  # 상위 500개만 사용
        max_intermediate=3
    )
    print(f"간접 관계: {len(pairs_indirect)}개")

    # 결합
    pairs = pd.concat([pairs_direct, pairs_indirect], ignore_index=True)

    # 중복 제거
    pairs = pairs.sort_values('max_corr', ascending=False)
    pairs = pairs.drop_duplicates(subset=['leading_item_id', 'following_item_id'], keep='first')
else:
    pairs = pairs_direct

print(f"\n최종 쌍 수: {len(pairs)}")
if len(pairs) > 0:
    print(f"평균 상관: {pairs['max_corr'].mean():.4f}")
    print(f"최소 상관: {pairs['max_corr'].min():.4f}")
    print(f"최대 상관: {pairs['max_corr'].max():.4f}")
    print("\nlag 분포:")
    print(pairs['best_lag'].value_counts().sort_index())


# =========================================
# 추가: 상위 N개만 선택 (품질 관리)
# =========================================
print("\n" + "="*70)
print("[옵션] 상위 품질만 선택")
print("="*70)

# 상위 1500~2000개 선택
if len(pairs) > 2000:
    pairs_top = pairs.nlargest(2000, 'max_corr')
    print(f"상위 2000개 선택: 평균 상관 {pairs_top['max_corr'].mean():.4f}")
elif len(pairs) > 1500:
    pairs_top = pairs.nlargest(1500, 'max_corr')
    print(f"상위 1500개 선택: 평균 상관 {pairs_top['max_corr'].mean():.4f}")
else:
    pairs_top = pairs

# pairs = pairs_top  # 사용 시 주석 해제

혁신적 접근: 비선형 + 네트워크 기반 공행성 탐색

[1단계] 다중 지표 기반 직접 관계 탐색


고급 공행성 탐색: 100%|██████████| 100/100 [03:10<00:00,  1.90s/it]


직접 관계: 2527개
평균 점수: 0.3324

[2단계] 네트워크 기반 간접 관계 탐색


간접 관계 탐색: 100%|██████████| 100/100 [00:00<00:00, 698.83it/s]

간접 관계: 27개

최종 쌍 수: 2551
평균 상관: 0.3309
최소 상관: 0.1506
최대 상관: 0.5592

lag 분포:
best_lag
1     146
2     159
3     165
4     179
5     146
6     167
7     161
8     312
9     269
10    292
11    233
12    322
Name: count, dtype: int64

[옵션] 상위 품질만 선택
상위 2000개 선택: 평균 상관 0.3444


### 상관계수

In [ ]:
pairs

,leading_item_id,following_item_id,best_lag,max_corr,ccf,dtw,granger,shape,indirect,path
2276,XIIEJNEE,DJBLNPNC,5,0.559174,0.926700,0.089594,0.908258,0.312145,NaN,NaN
1328,NAQIHUKZ,LLHREMKS,3,0.555431,0.894903,0.123215,0.881644,0.321963,NaN,NaN
1322,NAQIHUKZ,FTSVTTSR,1,0.535685,0.905093,0.088561,0.842925,0.306159,NaN,NaN
1558,QRKRBYJL,DNMPSKTB,7,0.534639,0.842065,0.079085,0.585835,0.431573,NaN,NaN
725,FTSVTTSR,LLHREMKS,2,0.534274,0.884762,0.088801,0.837838,0.325695,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
2535,BEZYMBBT,XIPPENFQ,11,0.154563,NaN,NaN,NaN,NaN,True,BEZYMBBT→AXULOHBQ→XIPPENFQ
2549,DJBLNPNC,AHMDUILJ,5,0.154391,NaN,NaN,NaN,NaN,True,DJBLNPNC→BSRMSVTC→AHMDUILJ
2529,AHMDUILJ,XIPPENFQ,10,0.151801,NaN,NaN,NaN,NaN,True,AHMDUILJ→AXULOHBQ→XIPPENFQ
2542,BTMOEMEP,LSOIUSXD,8,0.151699,NaN,NaN,NaN,NaN,True,BTMOEMEP→BSRMSVTC→LSOIUSXD


In [ ]:
import pandas as pd

pairs['abs_corr'] = pairs['max_corr'].abs()

# 1. 전체 평균 상관계수 구하기
mean_corr = pairs['abs_corr'].mean()
print(f"- 전체 평균 상관계수: {mean_corr:.4f}")

# 2. 분위수(Quantile) 계산
# 상위 20% 기준값 (80% 지점)
top_20_cutoff = pairs['abs_corr'].quantile(0.80)

# 하위 20% 기준값 (20% 지점)
bottom_20_cutoff = pairs['abs_corr'].quantile(0.20)

print(f"- 상위 20% 커트라인: {top_20_cutoff:.4f} 이상")
print(f"- 하위 20% 커트라인: {bottom_20_cutoff:.4f} 이하")

# 3. 데이터 추출 (Filtering)
top_20_df = pairs[pairs['abs_corr'] >= top_20_cutoff]
bottom_20_df = pairs[pairs['abs_corr'] <= bottom_20_cutoff]

# 4. 결과 출력
print(f"\n[상위 20% 데이터] - 총 {len(top_20_df)}개")
# 상위 5개만 확인
print(top_20_df[['leading_item_id', 'following_item_id', 'best_lag', 'abs_corr']].head())

print(f"\n[하위 20% 데이터] - 총 {len(bottom_20_df)}개")
# 하위 5개만 확인
print(bottom_20_df[['leading_item_id', 'following_item_id', 'best_lag', 'abs_corr']].head())

- 전체 평균 상관계수: 0.3309
- 상위 20% 커트라인: 0.3651 이상
- 하위 20% 커트라인: 0.2932 이하

[상위 20% 데이터] - 총 511개
     leading_item_id following_item_id  best_lag  abs_corr
2276        XIIEJNEE          DJBLNPNC         5  0.559174
1328        NAQIHUKZ          LLHREMKS         3  0.555431
1322        NAQIHUKZ          FTSVTTSR         1  0.535685
1558        QRKRBYJL          DNMPSKTB         7  0.534639
725         FTSVTTSR          LLHREMKS         2  0.534274

[하위 20% 데이터] - 총 511개
     leading_item_id following_item_id  best_lag  abs_corr
1155        LPHPPJUG          VMAQSTJE         7  0.293209
1939        UGEQLMXM          LRVGFDFM        10  0.293203
996         JSLXRQOK          ELQGMQWE        10  0.293189
1030        KAGJCHMR          FQCLOEXA         5  0.293179
1150        LPHPPJUG          PYZMVUWD         2  0.293151


### 가설5 분석
- 점수: 0.3422865227
- 공행성 쌍 개수: 2552
- 전체 평균 상관계수: 0.3309
- 상위 20% 커트라인: 0.3651 이상
- 하위 20% 커트라인: 0.2932 이하
---
- 별도의 문제점은 발견 못 했음
- 점수 급등, 지금의 방향성으로 접근 필요
- 다양한 외부적 요인 혹은 내부적 탐색 알고리즘을 모두 고려해야 함


# 가설6. 다양한 외부적 요인 혹은 내부적 탐색 알고리즘을 모두 적용시켜야 성능이 개선될 것이다.
- **계절성 선행 (계절성 패턴을 미리 포착하자)**
    - 3/6/12개월 주기 lag에 보너스
    - 분기별 평균 패턴 분석
    - 선행 지표 효과
- **모멘텀 전이 (A의 성장률 → B의 성장률 전이가 있지 않을까?)**
    - 변화율 기반 상관관계
    - 추세 전염 효과
    - 증가 패턴 복제
- **누적 효과 (A의 누적량이 B에 미치는 영향이 있을 것이다.)**
    - 장기 축적 효과
    - 재고/공급망 연쇄
    - 시장 포화 신호
- **이상치 동조화 (충격(급등/급락)의 여파가 있을 수 있다.)**
    - 이상 변동 시점 매칭
    - Jaccard 유사도 사용
    - 위기/호황 동조화
- **주파수 영역 (주기적 패턴 유사성을 찾아보자.)**
    - FFT 파워 스펙트럼 분석
    - 시간 + 주파수 결합
    - 숨은 주기 발견
- **클러스터 기반 (같은 산업군 내 강한 연결이 있을 것이다.)**
    - hs2 그룹 내 우선 탐색
    - 클러스터 보너스 0.08
    - 낮은 임계값 (0.25)
- 당연히 최종적으로 앙상블 기법 사용

### 가설6 검증 및 결과

In [ ]:
# =========================================
# 혁신적 다중 전략 모음
# 각 전략을 독립적으로 테스트 가능
# =========================================
import pandas as pd
import numpy as np
from tqdm import tqdm
from scipy.stats import spearmanr
from scipy.signal import correlate
import warnings
warnings.filterwarnings('ignore')


def get_item_meta(train):
    train['hs2'] = train['hs4'] // 100
    meta = train.groupby('item_id').agg({
        'hs4': 'first',
        'hs2': 'first'
    }).to_dict('index')
    return meta


# =========================================
# 전략 1: 계절성 강화 (Front-Running)
# =========================================
def strategy_seasonal_frontrun(pivot, item_meta, max_lag=12):
    """
    계절성 패턴 선행 포착
    - 특정 월에 강한 품목 쌍 탐색
    - 계절 주기: 3개월, 6개월, 12개월
    """
    items = pivot.index.to_list()
    months = pivot.columns.to_list()
    n_months = len(months)
    results = []

    # 월별 인덱스 계산
    month_indices = [pd.to_datetime(m).month for m in months]

    for leader in tqdm(items, desc="계절성 선행 전략"):
        x = pivot.loc[leader].values.astype(float)

        if np.count_nonzero(x) < 11:
            continue

        # 각 계절별 평균 계산
        seasonal_avg_leader = {}
        for season in [1, 2, 3, 4]:  # 분기
            mask = [(m-1)//3 == season-1 for m in month_indices]
            seasonal_avg_leader[season] = np.mean(x[mask]) if sum(mask) > 0 else 0

        leader_hs2 = item_meta.get(leader, {}).get('hs2', None)

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.count_nonzero(y) < 11:
                continue

            follower_hs2 = item_meta.get(follower, {}).get('hs2', None)
            same_hs2 = (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2)

            best_lag = None
            best_score = 0.0

            # 계절 주기 lag 우선 탐색
            priority_lags = [3, 6, 9, 12] + list(range(1, max_lag+1))

            for lag in priority_lags:
                if lag >= n_months or lag < 1:
                    continue

                # 기본 상관
                if np.std(x[:-lag]) > 0 and np.std(y[lag:]) > 0:
                    corr = np.corrcoef(x[:-lag], y[lag:])[0, 1]
                else:
                    corr = 0

                # 계절 주기 보너스
                if lag in [3, 6, 12]:
                    corr += 0.05

                if same_hs2:
                    corr += 0.05

                if abs(corr) > abs(best_score):
                    best_score = corr
                    best_lag = lag

            if best_lag and abs(best_score) >= 0.30:
                results.append({
                    'leading_item_id': leader,
                    'following_item_id': follower,
                    'best_lag': best_lag,
                    'max_corr': best_score
                })

    return pd.DataFrame(results)


# =========================================
# 전략 2: 변화율 기반 (Momentum Transfer)
# =========================================
def strategy_momentum_transfer(pivot, item_meta, max_lag=8):
    """
    변화율(모멘텀) 전이 패턴
    - A의 증가율이 B의 증가율과 상관
    - 추세 전염 효과
    """
    items = pivot.index.to_list()
    n_months = len(pivot.columns)
    results = []

    for leader in tqdm(items, desc="모멘텀 전이 전략"):
        x = pivot.loc[leader].values.astype(float)

        if np.count_nonzero(x) < 12:
            continue

        # 변화율 계산
        x_pct = np.zeros(len(x))
        for i in range(1, len(x)):
            if x[i-1] != 0:
                x_pct[i] = (x[i] - x[i-1]) / x[i-1]

        leader_hs2 = item_meta.get(leader, {}).get('hs2', None)

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.count_nonzero(y) < 12:
                continue

            # 변화율 계산
            y_pct = np.zeros(len(y))
            for i in range(1, len(y)):
                if y[i-1] != 0:
                    y_pct[i] = (y[i] - y[i-1]) / y[i-1]

            follower_hs2 = item_meta.get(follower, {}).get('hs2', None)
            same_hs2 = (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2)

            best_lag = None
            best_corr = 0.0

            for lag in range(1, min(max_lag+1, n_months)):
                if n_months <= lag:
                    continue

                # 변화율 간 상관
                x_lag = x_pct[:-lag]
                y_shift = y_pct[lag:]

                if np.std(x_lag) > 0 and np.std(y_shift) > 0:
                    corr = np.corrcoef(x_lag, y_shift)[0, 1]

                    if same_hs2:
                        corr += 0.05

                    if abs(corr) > abs(best_corr):
                        best_corr = corr
                        best_lag = lag

            if best_lag and abs(best_corr) >= 0.28:
                results.append({
                    'leading_item_id': leader,
                    'following_item_id': follower,
                    'best_lag': best_lag,
                    'max_corr': best_corr
                })

    return pd.DataFrame(results)


# =========================================
# 전략 3: 누적 효과 (Cumulative Impact)
# =========================================
def strategy_cumulative_impact(pivot, item_meta, max_lag=10):
    """
    누적 영향 측정
    - A의 누적 거래량이 B에 미치는 영향
    - 장기 축적 효과
    """
    items = pivot.index.to_list()
    n_months = len(pivot.columns)
    results = []

    for leader in tqdm(items, desc="누적 효과 전략"):
        x = pivot.loc[leader].values.astype(float)

        if np.count_nonzero(x) < 11:
            continue

        # 누적 합계 (이동 누적)
        x_cumsum = np.cumsum(x)

        leader_hs2 = item_meta.get(leader, {}).get('hs2', None)

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.count_nonzero(y) < 11:
                continue

            follower_hs2 = item_meta.get(follower, {}).get('hs2', None)
            same_hs2 = (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2)

            best_lag = None
            best_corr = 0.0

            for lag in range(1, min(max_lag+1, n_months)):
                if n_months <= lag:
                    continue

                # 누적 vs 현재
                x_cum_lag = x_cumsum[:-lag]
                y_shift = y[lag:]

                if np.std(x_cum_lag) > 0 and np.std(y_shift) > 0:
                    corr = np.corrcoef(x_cum_lag, y_shift)[0, 1]

                    if same_hs2:
                        corr += 0.05

                    if abs(corr) > abs(best_corr):
                        best_corr = corr
                        best_lag = lag

            if best_lag and abs(best_corr) >= 0.28:
                results.append({
                    'leading_item_id': leader,
                    'following_item_id': follower,
                    'best_lag': best_lag,
                    'max_corr': best_corr
                })

    return pd.DataFrame(results)


# =========================================
# 전략 4: 이상치 동조화 (Outlier Synchronization)
# =========================================
def strategy_outlier_sync(pivot, item_meta, max_lag=8):
    """
    이상치(급등/급락) 동조화
    - A의 이상 변동이 B의 이상 변동과 연동
    - 충격 전파 효과
    """
    items = pivot.index.to_list()
    n_months = len(pivot.columns)
    results = []

    def detect_outliers(series, threshold=1.5):
        """Z-score 기반 이상치 검출"""
        if np.std(series) == 0:
            return np.zeros(len(series), dtype=bool)
        z_scores = np.abs((series - np.mean(series)) / np.std(series))
        return z_scores > threshold

    for leader in tqdm(items, desc="이상치 동조 전략"):
        x = pivot.loc[leader].values.astype(float)

        if np.count_nonzero(x) < 11:
            continue

        # 이상치 감지
        x_outliers = detect_outliers(x)

        if np.sum(x_outliers) < 2:  # 이상치가 너무 적으면 스킵
            continue

        leader_hs2 = item_meta.get(leader, {}).get('hs2', None)

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.count_nonzero(y) < 11:
                continue

            y_outliers = detect_outliers(y)

            if np.sum(y_outliers) < 2:
                continue

            follower_hs2 = item_meta.get(follower, {}).get('hs2', None)
            same_hs2 = (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2)

            best_lag = None
            best_score = 0.0

            for lag in range(1, min(max_lag+1, n_months)):
                if n_months <= lag:
                    continue

                # 이상치 발생 시점의 동조화
                x_out_lag = x_outliers[:-lag]
                y_out_shift = y_outliers[lag:]

                # Jaccard 유사도
                intersection = np.sum(x_out_lag & y_out_shift)
                union = np.sum(x_out_lag | y_out_shift)

                if union > 0:
                    jaccard = intersection / union

                    if same_hs2:
                        jaccard += 0.05

                    if jaccard > best_score:
                        best_score = jaccard
                        best_lag = lag

            if best_lag and best_score >= 0.20:
                results.append({
                    'leading_item_id': leader,
                    'following_item_id': follower,
                    'best_lag': best_lag,
                    'max_corr': best_score
                })

    return pd.DataFrame(results)


# =========================================
# 전략 5: 교차 스펙트럼 분석 (Frequency Domain)
# =========================================
def strategy_frequency_domain(pivot, item_meta, max_lag=10):
    """
    주파수 영역 분석
    - 주기적 패턴의 유사성
    - FFT 기반 주기 동조화
    """
    items = pivot.index.to_list()
    n_months = len(pivot.columns)
    results = []

    for leader in tqdm(items, desc="주파수 영역 전략"):
        x = pivot.loc[leader].values.astype(float)

        if np.count_nonzero(x) < 12:
            continue

        # FFT
        x_fft = np.fft.fft(x)
        x_power = np.abs(x_fft)**2

        leader_hs2 = item_meta.get(leader, {}).get('hs2', None)

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.count_nonzero(y) < 12:
                continue

            y_fft = np.fft.fft(y)
            y_power = np.abs(y_fft)**2

            # 파워 스펙트럼 상관
            if np.std(x_power) > 0 and np.std(y_power) > 0:
                spectrum_corr = np.corrcoef(x_power, y_power)[0, 1]
            else:
                spectrum_corr = 0

            follower_hs2 = item_meta.get(follower, {}).get('hs2', None)
            same_hs2 = (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2)

            # 시간 영역에서도 최적 lag 찾기
            best_lag = None
            best_corr = 0.0

            for lag in range(1, min(max_lag+1, n_months)):
                if n_months <= lag:
                    continue

                if np.std(x[:-lag]) > 0 and np.std(y[lag:]) > 0:
                    time_corr = np.corrcoef(x[:-lag], y[lag:])[0, 1]

                    # 주파수 + 시간 영역 결합
                    combined = 0.6 * time_corr + 0.4 * spectrum_corr

                    if same_hs2:
                        combined += 0.05

                    if abs(combined) > abs(best_corr):
                        best_corr = combined
                        best_lag = lag

            if best_lag and abs(best_corr) >= 0.28:
                results.append({
                    'leading_item_id': leader,
                    'following_item_id': follower,
                    'best_lag': best_lag,
                    'max_corr': best_corr
                })

    return pd.DataFrame(results)


# =========================================
# 전략 6: 품목군 클러스터 기반
# =========================================
def strategy_cluster_based(pivot, item_meta, train):
    """
    hs2 기반 클러스터 내 강한 연결
    - 같은 산업군 내 리더-팔로워 쌍
    - 클러스터 중심성 활용
    """
    items = pivot.index.to_list()
    results = []

    # hs2별 그룹화
    hs2_groups = {}
    for item in items:
        hs2 = item_meta.get(item, {}).get('hs2', None)
        if hs2:
            if hs2 not in hs2_groups:
                hs2_groups[hs2] = []
            hs2_groups[hs2].append(item)

    # 각 클러스터 내에서 탐색
    for hs2, group_items in tqdm(hs2_groups.items(), desc="클러스터 기반 전략"):
        if len(group_items) < 2:
            continue

        for leader in group_items:
            x = pivot.loc[leader].values.astype(float)

            if np.count_nonzero(x) < 11:
                continue

            for follower in group_items:
                if follower == leader:
                    continue

                y = pivot.loc[follower].values.astype(float)

                if np.count_nonzero(y) < 11:
                    continue

                best_lag = None
                best_corr = 0.0

                for lag in range(1, min(13, len(x))):
                    if len(x) <= lag:
                        continue

                    if np.std(x[:-lag]) > 0 and np.std(y[lag:]) > 0:
                        corr = np.corrcoef(x[:-lag], y[lag:])[0, 1]

                        # 같은 클러스터 내 보너스
                        corr += 0.08

                        if abs(corr) > abs(best_corr):
                            best_corr = corr
                            best_lag = lag

                # 클러스터 내에서는 낮은 임계값
                if best_lag and abs(best_corr) >= 0.25:
                    results.append({
                        'leading_item_id': leader,
                        'following_item_id': follower,
                        'best_lag': best_lag,
                        'max_corr': best_corr
                    })

    return pd.DataFrame(results)


# =========================================
# 실행: 모든 전략 테스트
# =========================================
print("="*70)
print("6가지 혁신 전략 실행")
print("="*70)

item_meta = get_item_meta(train)

strategies = {}

print("\n[전략 1] 계절성 선행 (Front-Running)")
strategies['seasonal'] = strategy_seasonal_frontrun(pivot, item_meta, max_lag=12)
print(f"쌍 수: {len(strategies['seasonal'])}")

print("\n[전략 2] 모멘텀 전이")
strategies['momentum'] = strategy_momentum_transfer(pivot, item_meta, max_lag=8)
print(f"쌍 수: {len(strategies['momentum'])}")

print("\n[전략 3] 누적 효과")
strategies['cumulative'] = strategy_cumulative_impact(pivot, item_meta, max_lag=10)
print(f"쌍 수: {len(strategies['cumulative'])}")

print("\n[전략 4] 이상치 동조화")
strategies['outlier'] = strategy_outlier_sync(pivot, item_meta, max_lag=8)
print(f"쌍 수: {len(strategies['outlier'])}")

print("\n[전략 5] 주파수 영역")
strategies['frequency'] = strategy_frequency_domain(pivot, item_meta, max_lag=10)
print(f"쌍 수: {len(strategies['frequency'])}")

print("\n[전략 6] 클러스터 기반")
strategies['cluster'] = strategy_cluster_based(pivot, item_meta, train)
print(f"쌍 수: {len(strategies['cluster'])}")


# =========================================
# 앙상블: 여러 전략 결합
# =========================================
print("\n" + "="*70)
print("전략 앙상블")
print("="*70)

all_pairs_dict = {}

for strategy_name, pairs_df in strategies.items():
    if len(pairs_df) == 0:
        continue

    for _, row in pairs_df.iterrows():
        key = (row['leading_item_id'], row['following_item_id'])

        if key not in all_pairs_dict:
            all_pairs_dict[key] = {
                'leading_item_id': row['leading_item_id'],
                'following_item_id': row['following_item_id'],
                'lags': [],
                'corrs': [],
                'strategies': [],
                'vote_count': 0
            }

        all_pairs_dict[key]['lags'].append(row['best_lag'])
        all_pairs_dict[key]['corrs'].append(row['max_corr'])
        all_pairs_dict[key]['strategies'].append(strategy_name)
        all_pairs_dict[key]['vote_count'] += 1

# 2개 이상 전략에서 선택된 쌍
final_pairs = []
for key, info in all_pairs_dict.items():
    if info['vote_count'] >= 2:  # 최소 2개 전략 동의
        best_lag = max(set(info['lags']), key=info['lags'].count)
        avg_corr = np.mean(info['corrs'])

        final_pairs.append({
            'leading_item_id': info['leading_item_id'],
            'following_item_id': info['following_item_id'],
            'best_lag': best_lag,
            'max_corr': avg_corr,
            'vote_count': info['vote_count'],
            'strategies': ','.join(info['strategies'])
        })

pairs = pd.DataFrame(final_pairs)

if len(pairs) > 0:
    pairs = pairs.sort_values('max_corr', ascending=False)

print(f"\n최종 쌍 수: {len(pairs)}")
if len(pairs) > 0:
    print(f"평균 상관: {pairs['max_corr'].mean():.4f}")
    print(f"최소 상관: {pairs['max_corr'].min():.4f}")
    print(f"최대 상관: {pairs['max_corr'].max():.4f}")


# =========================================
# 개별 전략 사용 (선택)
# =========================================
print("\n" + "="*70)
print("개별 전략 선택 옵션")
print("="*70)
print("가장 좋은 단일 전략 선택:")

for name, df in strategies.items():
    if len(df) > 0:
        print(f"{name}: {len(df)}개, 평균상관 {df['max_corr'].mean():.4f}")

# 최고 성능 단일 전략 자동 선택
best_strategy = max(
    [(name, df) for name, df in strategies.items() if len(df) > 0],
    key=lambda x: len(x[1]) * x[1]['max_corr'].mean(),
    default=(None, pd.DataFrame())
)

if best_strategy[0]:
    print(f"\n최고 전략: {best_strategy[0]}")
    # pairs = best_strategy[1]  # 사용 시 주석 해제

6가지 혁신 전략 실행

[전략 1] 계절성 선행 (Front-Running)


계절성 선행 전략: 100%|██████████| 100/100 [00:50<00:00,  1.98it/s]


쌍 수: 5344

[전략 2] 모멘텀 전이


모멘텀 전이 전략: 100%|██████████| 100/100 [00:14<00:00,  7.05it/s]


쌍 수: 4136

[전략 3] 누적 효과


...

쌍 수: 3222

[전략 5] 주파수 영역


주파수 영역 전략: 100%|██████████| 100/100 [00:18<00:00,  5.49it/s]


쌍 수: 8011

[전략 6] 클러스터 기반


클러스터 기반 전략: 100%|██████████| 39/39 [00:01<00:00, 31.17it/s]


쌍 수: 531

전략 앙상블

최종 쌍 수: 7602
평균 상관: 0.2576
최소 상관: -0.6514
최대 상관: 0.7937

개별 전략 선택 옵션
가장 좋은 단일 전략 선택:
seasonal: 5344개, 평균상관 0.2096
momentum: 4136개, 평균상관 0.3175
cumulative: 5243개, 평균상관 -0.1914
outlier: 3222개, 평균상관 0.3009
frequency: 8011개, 평균상관 0.5352
cluster: 531개, 평균상관 0.3063

최고 전략: frequency


### 상관계수

In [ ]:
pairs

,leading_item_id,following_item_id,best_lag,max_corr,vote_count,strategies,abs_corr
2129,QRKRBYJL,QVLMOEYE,2,0.790375,5,"seasonal,weighted,volatility,detrended,multi_lag",0.790375
1755,NAQIHUKZ,LLHREMKS,3,0.783002,6,"seasonal,covid_aware,weighted,volatility,detre...",0.783002
3043,XIIEJNEE,DJBLNPNC,5,0.781865,5,"seasonal,weighted,volatility,detrended,multi_lag",0.781865
1750,NAQIHUKZ,FTSVTTSR,1,0.779850,6,"seasonal,covid_aware,weighted,volatility,detre...",0.779850
3329,ZKENOUDA,DEWLVASR,5,0.770472,6,"seasonal,covid_aware,weighted,volatility,detre...",0.770472
...,...,...,...,...,...,...,...
743,ELQGMQWE,OKMBFVKS,2,-0.659715,6,"seasonal,covid_aware,weighted,volatility,detre...",0.659715
1086,GYHKIVQT,ZKENOUDA,5,-0.660159,6,"seasonal,covid_aware,weighted,volatility,detre...",0.660159
701,DNMPSKTB,ZKENOUDA,6,-0.662696,5,"seasonal,covid_aware,weighted,volatility,multi...",0.662696
1046,GYHKIVQT,EVBVXETX,6,-0.685771,6,"seasonal,covid_aware,weighted,volatility,detre...",0.685771


In [ ]:
import pandas as pd

pairs['abs_corr'] = pairs['max_corr'].abs()

# 1. 전체 평균 상관계수 구하기
mean_corr = pairs['abs_corr'].mean()
print(f"- 전체 평균 상관계수: {mean_corr:.4f}")

# 2. 분위수(Quantile) 계산
# 상위 20% 기준값 (80% 지점)
top_20_cutoff = pairs['abs_corr'].quantile(0.80)

# 하위 20% 기준값 (20% 지점)
bottom_20_cutoff = pairs['abs_corr'].quantile(0.20)

print(f"- 상위 20% 커트라인: {top_20_cutoff:.4f} 이상")
print(f"- 하위 20% 커트라인: {bottom_20_cutoff:.4f} 이하")

# 3. 데이터 추출 (Filtering)
top_20_df = pairs[pairs['abs_corr'] >= top_20_cutoff]
bottom_20_df = pairs[pairs['abs_corr'] <= bottom_20_cutoff]

# 4. 결과 출력
print(f"\n[상위 20% 데이터] - 총 {len(top_20_df)}개")
# 상위 5개만 확인
print(top_20_df[['leading_item_id', 'following_item_id', 'best_lag', 'abs_corr']].head())

print(f"\n[하위 20% 데이터] - 총 {len(bottom_20_df)}개")
# 하위 5개만 확인
print(bottom_20_df[['leading_item_id', 'following_item_id', 'best_lag', 'abs_corr']].head())

- 전체 평균 상관계수: 0.2807
- 상위 20% 커트라인: 0.4400 이상
- 하위 20% 커트라인: 0.1176 이하

[상위 20% 데이터] - 총 1521개
     leading_item_id following_item_id  best_lag  abs_corr
2131        JSLXRQOK          NAQIHUKZ         2  0.793657
5237        ZKENOUDA          DEWLVASR         5  0.778466
2818        NAQIHUKZ          LLHREMKS         3  0.763982
5631        FDXPMYGF          WHPUAOID        10  0.753171
6017        OGAFEHLU          WHPUAOID         8  0.738044

[하위 20% 데이터] - 총 1521개
     leading_item_id following_item_id  best_lag  abs_corr
2351        LLHREMKS          OJIFIHMZ        12  0.117621
3516        RAWUKQMJ          WHPUAOID         2  0.117581
3996        SUOYXCHP          OXKURKXR        10  0.117558
1564        FWUCPMMW          DUCMGGNW         4  0.117517
6750        DUCMGGNW          BTMOEMEP         1  0.117473


### 가설6 분석
- 점수: 0.2130530332
- 공행성 쌍 개수: 7603
- 전체 평균 상관계수: 0.2807
- 상위 20% 커트라인: 0.4400 이상
- 하위 20% 커트라인: 0.1176 이하
---
- 거래 품목의 기간에 대한 관계성을 포착하려고 다양한 가설 및 접근을 했지만, 성능이 오히려 떨어졌다.
- 즉, 너무 복잡하게 갔다.
- 0.342를 달성했던 이전 방식으로 돌아가, 미세 튜닝에 집중해야 한다.

# 가설7. 기존의 전략에서 미세 튜닝을 진행하는 것이 공행성쌍을 효과적으로 탐색할 것이다.
- **전략6**
    - 원래 성공한 코드 그대로
    - CCF + DTW + Granger + Shape
    - 임계값 0.28
- **TopK 조정**
    - 임계값 **0.25로 낮춤** (더 많은 후보)
    - Leader당 **상위 15개만** 선택
    - 품질 유지 + 수량 증가
- **가중치 조정**
    - CCF: 35%, DTW: 30%, Granger: 20%, Shape: 15%
    - 더 신뢰성 높은 지표에 큰 비중
- **Kendall Tau 추가**
    - 순위 상관 계수의 한 종류로, 이상치에 강함
    - 5가지 지표 (기존 4개 + Kendall)
- **다중 임계값 앙상블**
    - 0.30, 0.28, 0.26 세 가지로 탐색
    - 2개 이상에서 선택된 쌍만 사용

### 가설7 검증 및 결과

In [ ]:
# =========================================
# 0.342 기반 개선 버전들
# 각각 독립적으로 테스트 가능
# =========================================
import pandas as pd
import numpy as np
from tqdm import tqdm
from scipy.stats import spearmanr, kendalltau
from scipy.spatial.distance import euclidean
import warnings
warnings.filterwarnings('ignore')


def get_item_meta(train):
    train['hs2'] = train['hs4'] // 100
    meta = train.groupby('item_id').agg({
        'hs4': 'first',
        'hs2': 'first'
    }).to_dict('index')
    return meta


# =========================================
# 기본 함수들 (0.342 코드에서)
# =========================================
def dtw_distance(x, y, window=3):
    """DTW 거리"""
    n, m = len(x), len(y)
    x = (x - np.mean(x)) / (np.std(x) + 1e-8)
    y = (y - np.mean(y)) / (np.std(y) + 1e-8)

    dtw = np.full((n + 1, m + 1), np.inf)
    dtw[0, 0] = 0

    for i in range(1, n + 1):
        for j in range(max(1, i - window), min(m + 1, i + window + 1)):
            cost = abs(x[i-1] - y[j-1])
            dtw[i, j] = cost + min(dtw[i-1, j], dtw[i, j-1], dtw[i-1, j-1])

    return dtw[n, m]


def dtw_similarity(x, y, window=3):
    """DTW 유사도"""
    dist = dtw_distance(x, y, window)
    return 1.0 / (1.0 + dist)


def cross_correlation_function(x, y, max_lag=12):
    """CCF"""
    x = (x - np.mean(x)) / (np.std(x) + 1e-8)
    y = (y - np.mean(y)) / (np.std(y) + 1e-8)

    correlations = []
    for lag in range(1, min(max_lag + 1, len(x))):
        if len(x) <= lag:
            break
        corr = np.corrcoef(x[:-lag], y[lag:])[0, 1]
        correlations.append((lag, corr))

    if not correlations:
        return 0, 0.0

    best_lag, best_corr = max(correlations, key=lambda t: abs(t[1]))
    return best_lag, best_corr


def granger_causality_score(x, y, max_lag=6):
    """Granger Causality"""
    best_score = 0.0
    best_lag = 1

    for lag in range(1, min(max_lag + 1, len(x) - 5)):
        if len(x) <= lag + 2:
            continue

        y_current = y[lag:]
        y_lagged = y[:-lag]
        x_lagged = x[:-lag]

        corr_y_self = abs(np.corrcoef(y_current, y_lagged)[0, 1])
        corr_x_y = abs(np.corrcoef(x_lagged, y_current)[0, 1])

        score = max(0, corr_x_y - corr_y_self * 0.5)

        if score > best_score:
            best_score = score
            best_lag = lag

    return best_lag, best_score


def shape_similarity(x, y, window_size=6):
    """Shape 유사도"""
    x_smooth = pd.Series(x).rolling(window=window_size, min_periods=1).mean().values
    y_smooth = pd.Series(y).rolling(window=window_size, min_periods=1).mean().values

    x_diff = np.diff(x_smooth)
    y_diff = np.diff(y_smooth)

    if len(x_diff) > 0 and len(y_diff) > 0:
        direction_match = np.mean(np.sign(x_diff) == np.sign(y_diff))
    else:
        direction_match = 0

    dtw_sim = dtw_similarity(x_smooth, y_smooth, window=3)

    return 0.5 * direction_match + 0.5 * dtw_sim


# =========================================
# 버전 1: 기본 (0.342 달성 코드 재현)
# =========================================
def version1_baseline(pivot, item_meta, max_lag=12, min_nonzero=11,
                      methods=['ccf', 'dtw', 'granger', 'shape'],
                      corr_threshold=0.28):
    """
    0.342 달성 버전 (기준선)
    """
    items = pivot.index.to_list()
    results = []

    for leader in tqdm(items, desc="V1-기준선"):
        x = pivot.loc[leader].values.astype(float)

        if np.count_nonzero(x) < min_nonzero:
            continue

        leader_hs2 = item_meta.get(leader, {}).get('hs2', None)

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.count_nonzero(y) < min_nonzero:
                continue

            follower_hs2 = item_meta.get(follower, {}).get('hs2', None)
            same_hs2 = (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2)

            scores = {}
            lags = {}

            if 'ccf' in methods:
                lag_ccf, corr_ccf = cross_correlation_function(x, y, max_lag)
                scores['ccf'] = abs(corr_ccf)
                lags['ccf'] = lag_ccf

            if 'dtw' in methods:
                best_dtw = 0
                best_lag_dtw = 1
                for lag in range(1, min(max_lag + 1, len(x))):
                    if len(x) <= lag:
                        break
                    sim = dtw_similarity(x[:-lag], y[lag:], window=3)
                    if sim > best_dtw:
                        best_dtw = sim
                        best_lag_dtw = lag
                scores['dtw'] = best_dtw
                lags['dtw'] = best_lag_dtw

            if 'granger' in methods:
                lag_granger, score_granger = granger_causality_score(x, y, max_lag)
                scores['granger'] = score_granger
                lags['granger'] = lag_granger

            if 'shape' in methods:
                best_shape = 0
                best_lag_shape = 1
                for lag in range(1, min(max_lag + 1, len(x))):
                    if len(x) <= lag:
                        break
                    sim = shape_similarity(x[:-lag], y[lag:], window_size=3)
                    if sim > best_shape:
                        best_shape = sim
                        best_lag_shape = lag
                scores['shape'] = best_shape
                lags['shape'] = best_lag_shape

            if scores:
                avg_score = np.mean(list(scores.values()))

                if same_hs2:
                    avg_score += 0.05

                lag_values = list(lags.values())
                best_lag = max(set(lag_values), key=lag_values.count) if lag_values else 1

                if avg_score >= corr_threshold:
                    results.append({
                        'leading_item_id': leader,
                        'following_item_id': follower,
                        'best_lag': best_lag,
                        'max_corr': avg_score
                    })

    return pd.DataFrame(results)


# =========================================
# 버전 2: 임계값 하향 + 상위 K개 선택
# =========================================
def version2_topk(pivot, item_meta, max_lag=12, min_nonzero=11,
                  corr_threshold=0.25, top_k_per_leader=15):
    """
    낮은 임계값으로 많이 찾고, leader당 상위 K개만
    품질 보장 + 수량 확보
    """
    items = pivot.index.to_list()
    all_results = []

    for leader in tqdm(items, desc="V2-TopK"):
        x = pivot.loc[leader].values.astype(float)

        if np.count_nonzero(x) < min_nonzero:
            continue

        leader_hs2 = item_meta.get(leader, {}).get('hs2', None)
        leader_results = []

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.count_nonzero(y) < min_nonzero:
                continue

            follower_hs2 = item_meta.get(follower, {}).get('hs2', None)
            same_hs2 = (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2)

            # 4가지 지표
            lag_ccf, corr_ccf = cross_correlation_function(x, y, max_lag)

            best_dtw = 0
            best_lag_dtw = 1
            for lag in range(1, min(max_lag + 1, len(x))):
                if len(x) <= lag:
                    break
                sim = dtw_similarity(x[:-lag], y[lag:], window=3)
                if sim > best_dtw:
                    best_dtw = sim
                    best_lag_dtw = lag

            lag_granger, score_granger = granger_causality_score(x, y, max_lag)

            best_shape = 0
            best_lag_shape = 1
            for lag in range(1, min(max_lag + 1, len(x))):
                if len(x) <= lag:
                    break
                sim = shape_similarity(x[:-lag], y[lag:], window_size=3)
                if sim > best_shape:
                    best_shape = sim
                    best_lag_shape = lag

            avg_score = np.mean([abs(corr_ccf), best_dtw, score_granger, best_shape])

            if same_hs2:
                avg_score += 0.05

            lags = [lag_ccf, best_lag_dtw, lag_granger, best_lag_shape]
            best_lag = max(set(lags), key=lags.count)

            if avg_score >= corr_threshold:
                leader_results.append({
                    'leading_item_id': leader,
                    'following_item_id': follower,
                    'best_lag': best_lag,
                    'max_corr': avg_score
                })

        # 상위 K개만 선택
        if leader_results:
            leader_df = pd.DataFrame(leader_results)
            leader_df = leader_df.nlargest(top_k_per_leader, 'max_corr')
            all_results.extend(leader_df.to_dict('records'))

    return pd.DataFrame(all_results)


# =========================================
# 버전 3: 가중치 조정
# =========================================
def version3_weighted(pivot, item_meta, max_lag=12, min_nonzero=11,
                      corr_threshold=0.28,
                      weights={'ccf': 0.35, 'dtw': 0.30, 'granger': 0.20, 'shape': 0.15}):
    """
    각 지표에 다른 가중치
    CCF와 DTW에 더 높은 비중
    """
    items = pivot.index.to_list()
    results = []

    for leader in tqdm(items, desc="V3-가중치"):
        x = pivot.loc[leader].values.astype(float)

        if np.count_nonzero(x) < min_nonzero:
            continue

        leader_hs2 = item_meta.get(leader, {}).get('hs2', None)

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.count_nonzero(y) < min_nonzero:
                continue

            follower_hs2 = item_meta.get(follower, {}).get('hs2', None)
            same_hs2 = (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2)

            scores = {}
            lags = {}

            lag_ccf, corr_ccf = cross_correlation_function(x, y, max_lag)
            scores['ccf'] = abs(corr_ccf)
            lags['ccf'] = lag_ccf

            best_dtw = 0
            best_lag_dtw = 1
            for lag in range(1, min(max_lag + 1, len(x))):
                if len(x) <= lag:
                    break
                sim = dtw_similarity(x[:-lag], y[lag:], window=3)
                if sim > best_dtw:
                    best_dtw = sim
                    best_lag_dtw = lag
            scores['dtw'] = best_dtw
            lags['dtw'] = best_lag_dtw

            lag_granger, score_granger = granger_causality_score(x, y, max_lag)
            scores['granger'] = score_granger
            lags['granger'] = lag_granger

            best_shape = 0
            best_lag_shape = 1
            for lag in range(1, min(max_lag + 1, len(x))):
                if len(x) <= lag:
                    break
                sim = shape_similarity(x[:-lag], y[lag:], window_size=3)
                if sim > best_shape:
                    best_shape = sim
                    best_lag_shape = lag
            scores['shape'] = best_shape
            lags['shape'] = best_lag_shape

            # 가중 평균
            weighted_score = sum(scores[k] * weights[k] for k in scores.keys())

            if same_hs2:
                weighted_score += 0.05

            lag_values = list(lags.values())
            best_lag = max(set(lag_values), key=lag_values.count)

            if weighted_score >= corr_threshold:
                results.append({
                    'leading_item_id': leader,
                    'following_item_id': follower,
                    'best_lag': best_lag,
                    'max_corr': weighted_score
                })

    return pd.DataFrame(results)


# =========================================
# 버전 4: Kendall Tau 추가
# =========================================
def version4_kendall(pivot, item_meta, max_lag=12, min_nonzero=11,
                     corr_threshold=0.28):
    """
    Kendall Tau 순위 상관 추가 (5가지 지표)
    더 강건한 측정
    """
    items = pivot.index.to_list()
    results = []

    for leader in tqdm(items, desc="V4-Kendall"):
        x = pivot.loc[leader].values.astype(float)

        if np.count_nonzero(x) < min_nonzero:
            continue

        leader_hs2 = item_meta.get(leader, {}).get('hs2', None)

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.count_nonzero(y) < min_nonzero:
                continue

            follower_hs2 = item_meta.get(follower, {}).get('hs2', None)
            same_hs2 = (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2)

            scores = []
            lags_list = []

            # CCF
            lag_ccf, corr_ccf = cross_correlation_function(x, y, max_lag)
            scores.append(abs(corr_ccf))
            lags_list.append(lag_ccf)

            # DTW
            best_dtw = 0
            best_lag_dtw = 1
            for lag in range(1, min(max_lag + 1, len(x))):
                if len(x) <= lag:
                    break
                sim = dtw_similarity(x[:-lag], y[lag:], window=3)
                if sim > best_dtw:
                    best_dtw = sim
                    best_lag_dtw = lag
            scores.append(best_dtw)
            lags_list.append(best_lag_dtw)

            # Granger
            lag_granger, score_granger = granger_causality_score(x, y, max_lag)
            scores.append(score_granger)
            lags_list.append(lag_granger)

            # Shape
            best_shape = 0
            best_lag_shape = 1
            for lag in range(1, min(max_lag + 1, len(x))):
                if len(x) <= lag:
                    break
                sim = shape_similarity(x[:-lag], y[lag:], window_size=3)
                if sim > best_shape:
                    best_shape = sim
                    best_lag_shape = lag
            scores.append(best_shape)
            lags_list.append(best_lag_shape)

            # Kendall Tau (새로 추가)
            best_kendall = 0
            best_lag_kendall = 1
            for lag in range(1, min(max_lag + 1, len(x))):
                if len(x) <= lag:
                    break
                try:
                    tau, _ = kendalltau(x[:-lag], y[lag:])
                    if not np.isnan(tau) and abs(tau) > abs(best_kendall):
                        best_kendall = tau
                        best_lag_kendall = lag
                except:
                    pass
            scores.append(abs(best_kendall))
            lags_list.append(best_lag_kendall)

            avg_score = np.mean(scores)

            if same_hs2:
                avg_score += 0.05

            best_lag = max(set(lags_list), key=lags_list.count)

            if avg_score >= corr_threshold:
                results.append({
                    'leading_item_id': leader,
                    'following_item_id': follower,
                    'best_lag': best_lag,
                    'max_corr': avg_score
                })

    return pd.DataFrame(results)


# =========================================
# 버전 5: 다중 임계값 앙상블
# =========================================
def version5_multi_threshold(pivot, item_meta, max_lag=12, min_nonzero=11):
    """
    여러 임계값으로 탐색 후 합치기
    - 0.30 (보수적)
    - 0.28 (균형)
    - 0.26 (공격적)
    교집합으로 신뢰성 확보
    """
    thresholds = [0.30, 0.28, 0.26]
    all_pairs = []

    for threshold in thresholds:
        print(f"  임계값 {threshold} 탐색 중...")
        pairs = version1_baseline(pivot, item_meta, max_lag, min_nonzero,
                                  methods=['ccf', 'dtw', 'granger', 'shape'],
                                  corr_threshold=threshold)
        all_pairs.append(pairs)

    # 투표 방식
    vote_dict = {}

    for pairs_df in all_pairs:
        for _, row in pairs_df.iterrows():
            key = (row['leading_item_id'], row['following_item_id'])

            if key not in vote_dict:
                vote_dict[key] = {
                    'leading_item_id': row['leading_item_id'],
                    'following_item_id': row['following_item_id'],
                    'lags': [],
                    'corrs': [],
                    'votes': 0
                }

            vote_dict[key]['lags'].append(row['best_lag'])
            vote_dict[key]['corrs'].append(row['max_corr'])
            vote_dict[key]['votes'] += 1

    # 2개 이상 임계값에서 선택된 쌍
    final = []
    for key, info in vote_dict.items():
        if info['votes'] >= 2:
            final.append({
                'leading_item_id': info['leading_item_id'],
                'following_item_id': info['following_item_id'],
                'best_lag': max(set(info['lags']), key=info['lags'].count),
                'max_corr': np.mean(info['corrs'])
            })

    return pd.DataFrame(final)


# =========================================
# 실행
# =========================================
print("="*70)
print("5가지 개선 버전 테스트")
print("="*70)

item_meta = get_item_meta(train)

print("\n[버전 1] 기준선 (0.342 재현)")
pairs_v1 = version1_baseline(pivot, item_meta)
print(f"쌍 수: {len(pairs_v1)}")
if len(pairs_v1) > 0:
    print(f"평균 상관: {pairs_v1['max_corr'].mean():.4f}")

print("\n[버전 2] TopK 전략")
pairs_v2 = version2_topk(pivot, item_meta, corr_threshold=0.25, top_k_per_leader=15)
print(f"쌍 수: {len(pairs_v2)}")
if len(pairs_v2) > 0:
    print(f"평균 상관: {pairs_v2['max_corr'].mean():.4f}")

print("\n[버전 3] 가중치 조정")
pairs_v3 = version3_weighted(pivot, item_meta)
print(f"쌍 수: {len(pairs_v3)}")
if len(pairs_v3) > 0:
    print(f"평균 상관: {pairs_v3['max_corr'].mean():.4f}")

print("\n[버전 4] Kendall Tau 추가")
pairs_v4 = version4_kendall(pivot, item_meta)
print(f"쌍 수: {len(pairs_v4)}")
if len(pairs_v4) > 0:
    print(f"평균 상관: {pairs_v4['max_corr'].mean():.4f}")

print("\n[버전 5] 다중 임계값 앙상블")
pairs_v5 = version5_multi_threshold(pivot, item_meta)
print(f"쌍 수: {len(pairs_v5)}")
if len(pairs_v5) > 0:
    print(f"평균 상관: {pairs_v5['max_corr'].mean():.4f}")

# 최종 선택 (가장 좋은 버전 사용)
print("\n" + "="*70)
print("각 버전별 요약")
print("="*70)

versions = {
    'v1_baseline': pairs_v1,
    'v2_topk': pairs_v2,
    'v3_weighted': pairs_v3,
    'v4_kendall': pairs_v4,
    'v5_ensemble': pairs_v5
}

for name, df in versions.items():
    if len(df) > 0:
        print(f"{name}: {len(df)}개, 평균={df['max_corr'].mean():.4f}")

# 기본값은 v1 (0.342 재현)
pairs = pairs_v1

5가지 개선 버전 테스트

[버전 1] 기준선 (0.342 재현)


V1-기준선: 100%|██████████| 100/100 [02:54<00:00,  1.75s/it]


쌍 수: 2527
평균 상관: 0.3324

[버전 2] TopK 전략


V2-TopK: 100%|██████████| 100/100 [02:47<00:00,  1.68s/it]


쌍 수: 1365
평균 상관: 0.3529

[버전 3] 가중치 조정


...

  임계값 0.28 탐색 중...


V1-기준선: 100%|██████████| 100/100 [03:00<00:00,  1.81s/it]


  임계값 0.26 탐색 중...


V1-기준선: 100%|██████████| 100/100 [02:58<00:00,  1.78s/it]


쌍 수: 2527
평균 상관: 0.3324

각 버전별 요약
v1_baseline: 2527개, 평균=0.3324
v2_topk: 1365개, 평균=0.3529
v3_weighted: 2245개, 평균=0.3362
v4_kendall: 2489개, 평균=0.3307
v5_ensemble: 2527개, 평균=0.3324


### 상관계수

In [ ]:
pairs

,leading_item_id,following_item_id,best_lag,max_corr
0,AANGBULD,APQGTRMF,8,0.310952
1,AANGBULD,BEZYMBBT,10,0.289829
2,AANGBULD,BLANHGYY,11,0.374989
3,AANGBULD,DEWLVASR,6,0.377643
4,AANGBULD,ELQGMQWE,10,0.314311
...,...,...,...,...
2522,ZXERAXWP,SAAYMURU,12,0.303267
2523,ZXERAXWP,UIFPPCLR,1,0.344870
2524,ZXERAXWP,VBYCLTYZ,8,0.283813
2525,ZXERAXWP,WHPUAOID,6,0.363354


In [ ]:
import pandas as pd

pairs['abs_corr'] = pairs['max_corr'].abs()

# 1. 전체 평균 상관계수 구하기
mean_corr = pairs['abs_corr'].mean()
print(f"- 전체 평균 상관계수: {mean_corr:.4f}")

# 2. 분위수(Quantile) 계산
# 상위 20% 기준값 (80% 지점)
top_20_cutoff = pairs['abs_corr'].quantile(0.80)

# 하위 20% 기준값 (20% 지점)
bottom_20_cutoff = pairs['abs_corr'].quantile(0.20)

print(f"- 상위 20% 커트라인: {top_20_cutoff:.4f} 이상")
print(f"- 하위 20% 커트라인: {bottom_20_cutoff:.4f} 이하")

# 3. 데이터 추출 (Filtering)
top_20_df = pairs[pairs['abs_corr'] >= top_20_cutoff]
bottom_20_df = pairs[pairs['abs_corr'] <= bottom_20_cutoff]

# 4. 결과 출력
print(f"\n[상위 20% 데이터] - 총 {len(top_20_df)}개")
# 상위 5개만 확인
print(top_20_df[['leading_item_id', 'following_item_id', 'best_lag', 'abs_corr']].head())

print(f"\n[하위 20% 데이터] - 총 {len(bottom_20_df)}개")
# 하위 5개만 확인
print(bottom_20_df[['leading_item_id', 'following_item_id', 'best_lag', 'abs_corr']].head())

- 전체 평균 상관계수: 0.3324
- 상위 20% 커트라인: 0.3654 이상
- 하위 20% 커트라인: 0.2936 이하

[상위 20% 데이터] - 총 506개
   leading_item_id following_item_id  best_lag  abs_corr
2         AANGBULD          BLANHGYY        11  0.374989
3         AANGBULD          DEWLVASR         6  0.377643
8         AANGBULD          FWUCPMMW        11  0.412499
9         AANGBULD          GKQIJYDH         6  0.377858
11        AANGBULD          KJNSOAHR        11  0.387577

[하위 20% 데이터] - 총 506개
   leading_item_id following_item_id  best_lag  abs_corr
1         AANGBULD          BEZYMBBT        10  0.289829
6         AANGBULD          FITUEHWN         7  0.286795
17        AANGBULD          VUAFAIYJ         2  0.287466
22        AHMDUILJ          APQGTRMF         6  0.282095
30        AHMDUILJ          IGDVVKUD         6  0.281740


### 가설7 분석
- 점수: 0.3442646172
- 공행성 쌍 개수: 2528
- 전체 평균 상관계수: 0.3324
- 상위 20% 커트라인: 0.3654 이상
- 하위 20% 커트라인: 0.2936 이하
---
- 노이즈 품목 포함
    - 거래가 적거나 불안정한 품목 존재
- 균일한 처리
    - 모든 품목을 동일하게 취급한 것
- 오래된 데이터 영향
    - 2022년 데이터가 2025년 예측에 영향
- 기존의 방식보다 점수가 오름
- 0.4에 가까워지기 위한 접근 방식 필요
- 다음의 전략이 바람직한 방향이라고 추정됨
    - 비선형 지표 분석 (DTW, CCF, Granger, Shape)
    - 앙상블 기법 (다중 임계값 투표)
    - [품질 > 수량] 중심

# 가설 8. 품질과 최근 데이터에 더욱 집중하자.
- **데이터 기반 필터링**
    - 고품질 품목만 선택 (상위 80개 품목에 집중)
    - 품질 점수 = 거래량 + 안정성 + 최근활성
- **동적 lag 범위 사용**
    - 변동성 낮은 쌍: lag 1-15
    - 변동성 중간: lag 1-10
    - 변동성 높은 쌍: lag 1-7
    - 품목 특성에 맞는 최적화
- **최근 데이터 강조**
    - 전체 기간 40% + 최근 12개월 60%
    - 2025년 예측에 최근 패턴이 더 중요하다고 판단
- **상관관계가 강한 쌍만 선택**
    - 임계값 0.35 (높음)
    - 품질 > 수량 전략
- **회귀 모델 변경**
    - RandomForest + Optuna

### 가설8 검증 및 결과

In [ ]:
# =========================================
# 0.4 돌파 전략
# 핵심: 실제 데이터의 숨은 패턴 발굴
# =========================================
import pandas as pd
import numpy as np
from tqdm import tqdm
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')


def get_item_meta(train):
    train['hs2'] = train['hs4'] // 100
    meta = train.groupby('item_id').agg({
        'hs4': 'first',
        'hs2': 'first'
    }).to_dict('index')
    return meta


# =========================================
# 핵심 아이디어 1: 실제 데이터 분석
# =========================================
def analyze_actual_patterns(train):
    """
    실제 데이터에서 강한 공행성을 보이는 패턴 발굴
    """
    print("="*70)
    print("실제 데이터 패턴 분석")
    print("="*70)

    # 1. hs2별 평균 거래량
    hs2_summary = train.groupby('hs2').agg({
        'value': ['sum', 'mean', 'std', 'count']
    }).round(2)

    print("\n[HS2별 거래 규모]")
    hs2_summary_sorted = hs2_summary.sort_values(('value', 'sum'), ascending=False)
    print(hs2_summary_sorted.head(10))

    # 2. 시간별 추세
    monthly = train.groupby(['year', 'month'])['value'].sum()
    print(f"\n[시간별 추세]")
    print(f"전체 평균: {monthly.mean():.0f}")
    print(f"최근 6개월 평균: {monthly.tail(6).mean():.0f}")
    print(f"증가율: {(monthly.tail(6).mean() / monthly.head(6).mean() - 1) * 100:.1f}%")

    # 3. 품목별 변동성
    item_volatility = train.groupby('item_id')['value'].agg(['mean', 'std'])
    item_volatility['cv'] = item_volatility['std'] / (item_volatility['mean'] + 1)

    print(f"\n[품목 변동성 분포]")
    print(f"저변동 (<0.5): {(item_volatility['cv'] < 0.5).sum()}개")
    print(f"중변동 (0.5-1.5): {((item_volatility['cv'] >= 0.5) & (item_volatility['cv'] < 1.5)).sum()}개")
    print(f"고변동 (>=1.5): {(item_volatility['cv'] >= 1.5).sum()}개")

    return hs2_summary_sorted, item_volatility


# =========================================
# 전략 1: 데이터 기반 필터링
# =========================================
def strategy_data_driven_filter(pivot, train, item_meta):
    """
    실제 데이터 특성에 맞는 필터링
    - 고거래량 품목 우선
    - 안정적 품목 우선
    - 최근 활성 품목 우선
    """
    from scipy.stats import spearmanr

    # 품목별 특성 계산
    item_stats = {}
    for item_id in pivot.index:
        values = pivot.loc[item_id].values

        item_stats[item_id] = {
            'total': np.sum(values),
            'mean': np.mean(values),
            'recent_mean': np.mean(values[-6:]),  # 최근 6개월
            'cv': np.std(values) / (np.mean(values) + 1),
            'nonzero_ratio': np.count_nonzero(values) / len(values),
            'trend': np.polyfit(range(len(values)), values, 1)[0]  # 추세
        }

    # 품질 점수 계산
    for item_id in item_stats:
        stats = item_stats[item_id]

        # 점수 = 거래량 + 안정성 + 최근활성
        score = 0
        score += min(stats['total'] / 1e9, 1.0) * 0.3  # 총 거래량 (정규화)
        score += (1 - min(stats['cv'], 2.0) / 2) * 0.3  # 안정성
        score += stats['nonzero_ratio'] * 0.2  # 데이터 풍부도
        score += min(stats['recent_mean'] / 1e8, 1.0) * 0.2  # 최근 활성도

        item_stats[item_id]['quality_score'] = score

    # 상위 품목만 사용
    sorted_items = sorted(item_stats.items(), key=lambda x: x[1]['quality_score'], reverse=True)
    top_items = [item[0] for item in sorted_items[:80]]  # 상위 80개

    print(f"고품질 품목: {len(top_items)}개 선택")

    # 이 품목들로만 공행성 탐색
    items = top_items
    results = []

    for leader in tqdm(items, desc="데이터 기반 필터"):
        x = pivot.loc[leader].values.astype(float)

        if np.count_nonzero(x) < 11:
            continue

        leader_hs2 = item_meta.get(leader, {}).get('hs2', None)

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.count_nonzero(y) < 11:
                continue

            follower_hs2 = item_meta.get(follower, {}).get('hs2', None)
            same_hs2 = (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2)

            best_lag = None
            best_corr = 0.0

            for lag in range(1, min(13, len(x))):
                if len(x) <= lag:
                    continue

                if np.std(x[:-lag]) > 0 and np.std(y[lag:]) > 0:
                    # Pearson + Spearman 조합
                    pearson = np.corrcoef(x[:-lag], y[lag:])[0, 1]
                    try:
                        spearman, _ = spearmanr(x[:-lag], y[lag:])
                        spearman = spearman if not np.isnan(spearman) else 0
                    except:
                        spearman = 0

                    corr = 0.7 * pearson + 0.3 * spearman

                    # 품질 점수 보너스
                    quality_bonus = (item_stats[leader]['quality_score'] +
                                   item_stats[follower]['quality_score']) / 2 * 0.1
                    corr += quality_bonus

                    if same_hs2:
                        corr += 0.06

                    if abs(corr) > abs(best_corr):
                        best_corr = corr
                        best_lag = lag

            if best_lag and abs(best_corr) >= 0.30:
                results.append({
                    'leading_item_id': leader,
                    'following_item_id': follower,
                    'best_lag': best_lag,
                    'max_corr': best_corr
                })

    return pd.DataFrame(results)


# =========================================
# 전략 2: 동적 lag 범위
# =========================================
def strategy_adaptive_lag(pivot, item_meta, min_lag=1, max_lag=15):
    """
    품목 쌍마다 최적 lag 범위가 다름
    - 변동성 높은 쌍: 짧은 lag (1-6)
    - 안정적 쌍: 긴 lag도 가능 (1-15)
    """
    items = pivot.index.to_list()
    results = []

    for leader in tqdm(items, desc="동적 lag"):
        x = pivot.loc[leader].values.astype(float)

        if np.count_nonzero(x) < 11:
            continue

        x_cv = np.std(x) / (np.mean(x) + 1)
        leader_hs2 = item_meta.get(leader, {}).get('hs2', None)

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.count_nonzero(y) < 11:
                continue

            y_cv = np.std(y) / (np.mean(y) + 1)
            follower_hs2 = item_meta.get(follower, {}).get('hs2', None)
            same_hs2 = (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2)

            # 변동성에 따라 lag 범위 조정
            avg_cv = (x_cv + y_cv) / 2

            if avg_cv < 0.8:  # 안정적
                lag_range = range(1, min(max_lag + 1, len(x)))
            elif avg_cv < 1.5:  # 중간
                lag_range = range(1, min(10, len(x)))
            else:  # 변동적
                lag_range = range(1, min(7, len(x)))

            best_lag = None
            best_corr = 0.0

            for lag in lag_range:
                if len(x) <= lag:
                    continue

                if np.std(x[:-lag]) > 0 and np.std(y[lag:]) > 0:
                    corr = np.corrcoef(x[:-lag], y[lag:])[0, 1]

                    if same_hs2:
                        corr += 0.06

                    if abs(corr) > abs(best_corr):
                        best_corr = corr
                        best_lag = lag

            if best_lag and abs(best_corr) >= 0.30:
                results.append({
                    'leading_item_id': leader,
                    'following_item_id': follower,
                    'best_lag': best_lag,
                    'max_corr': best_corr
                })

    return pd.DataFrame(results)


# =========================================
# 전략 3: 복합 시그널 (최근 강조)
# =========================================
def strategy_recent_emphasis(pivot, item_meta, recent_months=12):
    """
    최근 데이터에 더 높은 가중치
    - 전체 기간 상관 40%
    - 최근 N개월 상관 60%
    """
    items = pivot.index.to_list()
    n_months = len(pivot.columns)
    results = []

    for leader in tqdm(items, desc="최근 강조"):
        x = pivot.loc[leader].values.astype(float)

        if np.count_nonzero(x) < 11:
            continue

        leader_hs2 = item_meta.get(leader, {}).get('hs2', None)

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.count_nonzero(y) < 11:
                continue

            follower_hs2 = item_meta.get(follower, {}).get('hs2', None)
            same_hs2 = (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2)

            best_lag = None
            best_score = 0.0

            for lag in range(1, min(13, n_months)):
                if n_months <= lag:
                    continue

                # 전체 기간 상관
                if np.std(x[:-lag]) > 0 and np.std(y[lag:]) > 0:
                    corr_full = np.corrcoef(x[:-lag], y[lag:])[0, 1]
                else:
                    corr_full = 0

                # 최근 기간 상관
                if n_months > recent_months + lag:
                    x_recent = x[-(recent_months+lag):-lag]
                    y_recent = y[-recent_months:]

                    if np.std(x_recent) > 0 and np.std(y_recent) > 0:
                        corr_recent = np.corrcoef(x_recent, y_recent)[0, 1]
                    else:
                        corr_recent = corr_full
                else:
                    corr_recent = corr_full

                # 조합
                combined = 0.4 * corr_full + 0.6 * corr_recent

                if same_hs2:
                    combined += 0.06

                if abs(combined) > abs(best_score):
                    best_score = combined
                    best_lag = lag

            if best_lag and abs(best_score) >= 0.30:
                results.append({
                    'leading_item_id': leader,
                    'following_item_id': follower,
                    'best_lag': best_lag,
                    'max_corr': best_score
                })

    return pd.DataFrame(results)


# =========================================
# 전략 4: 강한 쌍 집중 (Top Pairs)
# =========================================
def strategy_strong_pairs_only(pivot, item_meta, min_corr=0.35):
    """
    매우 높은 임계값으로 강한 쌍만
    수량 < 품질
    """
    items = pivot.index.to_list()
    results = []

    for leader in tqdm(items, desc="강한 쌍"):
        x = pivot.loc[leader].values.astype(float)

        if np.count_nonzero(x) < 12:
            continue

        leader_hs2 = item_meta.get(leader, {}).get('hs2', None)

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.count_nonzero(y) < 12:
                continue

            follower_hs2 = item_meta.get(follower, {}).get('hs2', None)
            same_hs2 = (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2)

            best_lag = None
            best_corr = 0.0

            for lag in range(1, min(13, len(x))):
                if len(x) <= lag:
                    continue

                if np.std(x[:-lag]) > 0 and np.std(y[lag:]) > 0:
                    corr = np.corrcoef(x[:-lag], y[lag:])[0, 1]

                    # Spearman 추가
                    try:
                        spear, _ = spearmanr(x[:-lag], y[lag:])
                        if not np.isnan(spear):
                            corr = 0.7 * corr + 0.3 * spear
                    except:
                        pass

                    if same_hs2:
                        corr += 0.07

                    if abs(corr) > abs(best_corr):
                        best_corr = corr
                        best_lag = lag

            # 높은 임계값
            if best_lag and abs(best_corr) >= min_corr:
                results.append({
                    'leading_item_id': leader,
                    'following_item_id': follower,
                    'best_lag': best_lag,
                    'max_corr': best_corr
                })

    return pd.DataFrame(results)


# =========================================
# 실행 및 앙상블
# =========================================
print("="*70)
print("0.4 돌파 전략")
print("="*70)

item_meta = get_item_meta(train)

# 먼저 실제 데이터 분석
hs2_summary, item_vol = analyze_actual_patterns(train)

strategies = {}

print("\n[전략 1] 데이터 기반 필터링")
strategies['data_driven'] = strategy_data_driven_filter(pivot, train, item_meta)
print(f"쌍 수: {len(strategies['data_driven'])}")

print("\n[전략 2] 동적 lag")
strategies['adaptive_lag'] = strategy_adaptive_lag(pivot, item_meta, max_lag=15)
print(f"쌍 수: {len(strategies['adaptive_lag'])}")

print("\n[전략 3] 최근 강조")
strategies['recent'] = strategy_recent_emphasis(pivot, item_meta, recent_months=12)
print(f"쌍 수: {len(strategies['recent'])}")

print("\n[전략 4] 강한 쌍만")
strategies['strong'] = strategy_strong_pairs_only(pivot, item_meta, min_corr=0.35)
print(f"쌍 수: {len(strategies['strong'])}")

# 앙상블
print("\n" + "="*70)
print("전략 앙상블")
print("="*70)

vote_dict = {}

for name, df in strategies.items():
    if len(df) == 0:
        continue

    for _, row in df.iterrows():
        key = (row['leading_item_id'], row['following_item_id'])

        if key not in vote_dict:
            vote_dict[key] = {
                'leading_item_id': row['leading_item_id'],
                'following_item_id': row['following_item_id'],
                'lags': [],
                'corrs': [],
                'votes': 0
            }

        vote_dict[key]['lags'].append(row['best_lag'])
        vote_dict[key]['corrs'].append(row['max_corr'])
        vote_dict[key]['votes'] += 1

# 2표 이상
final = []
for key, info in vote_dict.items():
    if info['votes'] >= 2:
        final.append({
            'leading_item_id': info['leading_item_id'],
            'following_item_id': info['following_item_id'],
            'best_lag': max(set(info['lags']), key=info['lags'].count),
            'max_corr': np.mean(info['corrs'])
        })

pairs = pd.DataFrame(final)

if len(pairs) > 0:
    pairs = pairs.sort_values('max_corr', ascending=False)

print(f"\n최종 쌍 수: {len(pairs)}")
if len(pairs) > 0:
    print(f"평균 상관: {pairs['max_corr'].mean():.4f}")
    print(f"최소 상관: {pairs['max_corr'].min():.4f}")
    print(f"최대 상관: {pairs['max_corr'].max():.4f}")

# 개별 전략도 테스트 가능
print("\n개별 전략 선택:")
for name, df in strategies.items():
    if len(df) > 0:
        print(f"{name}: {len(df)}개, 평균={df['max_corr'].mean():.4f}")

0.4 돌파 전략
실제 데이터 패턴 분석

[HS2별 거래 규모]
            value                               
              sum        mean          std count
hs2                                             
38   6.002055e+09  5810314.79  12107087.36  1033
28   4.461838e+09  1753866.97   5908115.24  2544
85   2.496308e+09  2658475.26   5022862.95   939
31   1.509013e+09  5871646.19   6645617.99   257
62   1.081998e+09  8387578.77   4137829.53   129
94   7.404158e+08  5739657.26   1145087.98   129
72   5.236698e+08  1961310.19   2481963.56   267
84   3.096500e+08   892363.08   1887098.03   347
39   2.808027e+08  1088382.47   1098791.51   258
32   2.441937e+08   634269.28    832090.38   385

[시간별 추세]
전체 평균: 438339405
최근 6개월 평균: 399800190
증가율: -32.0%

[품목 변동성 분포]
저변동 (<0.5): 16개
중변동 (0.5-1.5): 52개
고변동 (>=1.5): 32개

[전략 1] 데이터 기반 필터링
고품질 품목: 80개 선택


데이터 기반 필터: 100%|██████████| 80/80 [01:11<00:00,  1.11it/s]


쌍 수: 4116

[전략 2] 동적 lag


동적 lag: 100%|██████████| 100/100 [00:19<00:00,  5.20it/s]


쌍 수: 5018

[전략 3] 최근 강조


최근 강조: 100%|██████████| 100/100 [00:35<00:00,  2.79it/s]


쌍 수: 7534

[전략 4] 강한 쌍만


강한 쌍: 100%|██████████| 100/100 [01:31<00:00,  1.09it/s]


쌍 수: 3298

전략 앙상블

최종 쌍 수: 5646
평균 상관: 0.1701
최소 상관: -0.6932
최대 상관: 0.8554

개별 전략 선택:
data_driven: 4116개, 평균=0.2454
adaptive_lag: 5018개, 평균=0.1468
recent: 7534개, 평균=0.1292
strong: 3298개, 평균=0.1630


### 상관계수

In [ ]:
pairs

,leading_item_id,following_item_id,best_lag,max_corr
1777,QRKRBYJL,DNMPSKTB,7,0.855357
4878,NAQIHUKZ,FTSVTTSR,1,0.832526
80,ATLDMDBO,QRKRBYJL,1,0.828813
4881,NAQIHUKZ,LLHREMKS,3,0.824692
513,GKQIJYDH,VMAQSTJE,10,0.784189
...,...,...,...,...
2635,LSOIUSXD,VUAFAIYJ,9,-0.673110
259,GYHKIVQT,ZKENOUDA,5,-0.673561
1004,OKMBFVKS,FQCLOEXA,10,-0.681848
220,GYHKIVQT,KJNSOAHR,8,-0.692947


In [ ]:
import pandas as pd

pairs['abs_corr'] = pairs['max_corr'].abs()

# 1. 전체 평균 상관계수 구하기
mean_corr = pairs['abs_corr'].mean()
print(f"- 전체 평균 상관계수: {mean_corr:.4f}")

# 2. 분위수(Quantile) 계산
# 상위 20% 기준값 (80% 지점)
top_20_cutoff = pairs['abs_corr'].quantile(0.80)

# 하위 20% 기준값 (20% 지점)
bottom_20_cutoff = pairs['abs_corr'].quantile(0.20)

print(f"- 상위 20% 커트라인: {top_20_cutoff:.4f} 이상")
print(f"- 하위 20% 커트라인: {bottom_20_cutoff:.4f} 이하")

# 3. 데이터 추출 (Filtering)
top_20_df = pairs[pairs['abs_corr'] >= top_20_cutoff]
bottom_20_df = pairs[pairs['abs_corr'] <= bottom_20_cutoff]

# 4. 결과 출력
print(f"\n[상위 20% 데이터] - 총 {len(top_20_df)}개")
# 상위 5개만 확인
print(top_20_df[['leading_item_id', 'following_item_id', 'best_lag', 'abs_corr']].head())

print(f"\n[하위 20% 데이터] - 총 {len(bottom_20_df)}개")
# 하위 5개만 확인
print(bottom_20_df[['leading_item_id', 'following_item_id', 'best_lag', 'abs_corr']].head())

- 전체 평균 상관계수: 0.3635
- 상위 20% 커트라인: 0.4825 이상
- 하위 20% 커트라인: 0.2002 이하

[상위 20% 데이터] - 총 1130개
     leading_item_id following_item_id  best_lag  abs_corr
1777        QRKRBYJL          DNMPSKTB         7  0.855357
4878        NAQIHUKZ          FTSVTTSR         1  0.832526
80          ATLDMDBO          QRKRBYJL         1  0.828813
4881        NAQIHUKZ          LLHREMKS         3  0.824692
513         GKQIJYDH          VMAQSTJE        10  0.784189

[하위 20% 데이터] - 총 1130개
     leading_item_id following_item_id  best_lag  abs_corr
2165        DBWLZWNK          WQMVCOEM         9  0.200214
2207        BJALXPFS          WBLJNPZQ         5  0.199864
2508        FITUEHWN          STZDBITS         3  0.199489
5122        ROACSLMG          IGDVVKUD         2  0.199139
505         GKQIJYDH          ELQGMQWE         2  0.198877


### 가설8 분석
- 점수: 0.3500838476
- 공행성 쌍 개수: 2528
- 전체 평균 상관계수: 0.3635
- 상위 20% 커트라인: 0.4825 이상
- 하위 20% 커트라인: 0.2002 이하
---
- feature 개수가 5개로(b_t, b_t_1, a_t_lag, max_corr, best_lag) 적음
  - 늘릴 필요 있다고 판
- 점수가 0.35대로 상승, 동일한 방향성의 정교한 가설 필요

# 가설9. 외부적 요인을 추가로 탐색하고, feature을 조정하는 것이 적절한 방식이다.
- **외부적 요인 추가 탐색**
    - 대체재/보완재
        - 경쟁 vs 시너지 관계
    - 글로벌 이벤트
        - 2022 인플레이션, 2023 금리 인상 동조
    - 공급망 연쇄
        - 원자재 → 중간재 → 완제품
    - 계절성 리더십
        - 계절 변화 선행 감지
    - 거래량 영향력
        - 대량 거래 품목의 시장 지배력
- **feature 개수 증가**
    - Lag 피처 (b_t_2, b_t_3, a_t_lag_1, a_t_lag_2)
    - 변화율 (3개)
    - 이동평균 (3개)
    - 변동성 (2개)
    - 비율 (2개)
    - 교차 (2개)
    - 추세 (1개)
    - 시간 (3개)
    - 상호작용 (2개)
- **n_estimators은 150 으로 증가**

### 가설9 검증 및 결과

In [ ]:
# =========================================
# 0.4 돌파: 창의적 가설 + 회귀 피처 강화
# =========================================
import pandas as pd
import numpy as np
from tqdm import tqdm
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')


def get_item_meta(train):
    train['hs2'] = train['hs4'] // 100
    train['hs4_group'] = train['hs4'] // 10  # hs4 중분류

    meta = train.groupby('item_id').agg({
        'hs4': 'first',
        'hs2': 'first',
        'value': ['mean', 'std', 'sum'],
        'weight': 'mean',
        'quantity': 'mean'
    })

    # 컬럼명 단순화
    meta.columns = ['_'.join(col).strip() if col[1] else col[0]
                    for col in meta.columns.values]

    return meta.to_dict('index')


# =========================================
# 가설 1: "대체재/보완재 관계"
# =========================================
def hypothesis_substitute_complement(pivot, item_meta):
    """
    가설: 특정 품목 쌍은 대체재 또는 보완재 관계
    - 대체재: A↑ → B↓ (음의 상관)
    - 보완재: A↑ → B↑ (양의 상관, 동시 lag)

    특징:
    - 같은 hs2 내에서 대체재 가능성
    - 다른 hs2 간 보완재 가능성
    """
    items = pivot.index.to_list()
    results = []

    for leader in tqdm(items, desc="대체재/보완재 가설"):
        x = pivot.loc[leader].values.astype(float)

        if np.count_nonzero(x) < 11:
            continue

        leader_hs2 = item_meta.get(leader, {}).get('hs2_first', None)

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.count_nonzero(y) < 11:
                continue

            follower_hs2 = item_meta.get(follower, {}).get('hs2_first', None)
            same_hs2 = (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2)

            best_lag = None
            best_corr = 0.0
            relationship_type = 'unknown'

            for lag in range(1, min(13, len(x))):
                if len(x) <= lag:
                    continue

                if np.std(x[:-lag]) > 0 and np.std(y[lag:]) > 0:
                    corr = np.corrcoef(x[:-lag], y[lag:])[0, 1]

                    # 관계 유형 판별
                    if corr > 0.3:  # 강한 양의 상관
                        if lag <= 2 and not same_hs2:
                            # 보완재 (동시 이동, 다른 산업)
                            corr += 0.08
                            relationship_type = 'complement'
                        elif same_hs2:
                            # 같은 산업 내 선후행
                            corr += 0.05
                            relationship_type = 'same_industry'

                    elif corr < -0.2 and same_hs2:  # 음의 상관 + 같은 산업
                        # 대체재 가능성
                        corr = abs(corr) + 0.10  # 음의 상관도 유효
                        relationship_type = 'substitute'

                    if abs(corr) > abs(best_corr):
                        best_corr = corr
                        best_lag = lag

            if best_lag and abs(best_corr) >= 0.28:
                results.append({
                    'leading_item_id': leader,
                    'following_item_id': follower,
                    'best_lag': best_lag,
                    'max_corr': best_corr,
                    'relationship': relationship_type
                })

    return pd.DataFrame(results)


# =========================================
# 가설 2: "글로벌 이벤트 동조화"
# =========================================
def hypothesis_global_event_sync(pivot, item_meta):
    """
    가설: 특정 기간의 글로벌 이벤트에 같이 반응하는 품목들
    - 2022년 인플레이션 → 원자재 동시 상승
    - 2023년 금리 인상 → 내구재 동시 하락

    방법: 특정 기간의 변동 패턴 유사도
    """
    items = pivot.index.to_list()
    months = pivot.columns.to_list()
    n_months = len(months)

    # 주요 이벤트 기간 정의 (인덱스)
    event_periods = {
        'inflation_2022': list(range(0, 12)),  # 2022년 전체
        'rate_hike_2023': list(range(12, 24)),  # 2023년
        'recent': list(range(-12, 0))  # 최근 12개월
    }

    results = []

    for leader in tqdm(items, desc="글로벌 이벤트 가설"):
        x = pivot.loc[leader].values.astype(float)

        if np.count_nonzero(x) < 11:
            continue

        leader_hs2 = item_meta.get(leader, {}).get('hs2_first', None)

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.count_nonzero(y) < 11:
                continue

            follower_hs2 = item_meta.get(follower, {}).get('hs2_first', None)
            same_hs2 = (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2)

            # 각 이벤트 기간의 유사도 계산
            event_scores = []

            for event_name, period_idx in event_periods.items():
                try:
                    x_period = x[period_idx]
                    y_period = y[period_idx]

                    if np.std(x_period) > 0 and np.std(y_period) > 0:
                        event_corr = np.corrcoef(x_period, y_period)[0, 1]
                        event_scores.append(abs(event_corr))
                except:
                    pass

            if not event_scores:
                continue

            # 평균 이벤트 동조화 점수
            avg_event_score = np.mean(event_scores)

            # 일반 lag 상관도 계산
            best_lag = None
            best_corr = 0.0

            for lag in range(1, min(13, n_months)):
                if n_months <= lag:
                    continue

                if np.std(x[:-lag]) > 0 and np.std(y[lag:]) > 0:
                    corr = np.corrcoef(x[:-lag], y[lag:])[0, 1]

                    # 이벤트 동조화 보너스
                    combined = abs(corr) + 0.3 * avg_event_score

                    if same_hs2:
                        combined += 0.05

                    if combined > best_corr:
                        best_corr = combined
                        best_lag = lag

            if best_lag and best_corr >= 0.30:
                results.append({
                    'leading_item_id': leader,
                    'following_item_id': follower,
                    'best_lag': best_lag,
                    'max_corr': best_corr,
                    'event_sync_score': avg_event_score
                })

    return pd.DataFrame(results)


# =========================================
# 가설 3: "공급망 연쇄 (Supply Chain Cascade)"
# =========================================
def hypothesis_supply_chain(pivot, item_meta):
    """
    가설: 원자재 → 중간재 → 완제품 순차 영향

    HS 분류 기반 추정:
    - HS 25-27: 광물, 에너지 (원자재)
    - HS 72-83: 철강, 금속 (중간재)
    - HS 84-90: 기계, 전자 (완제품)

    원자재 → 중간재/완제품 lag이 더 길 것
    """
    items = pivot.index.to_list()

    # HS2 기반 분류
    raw_materials = [25, 26, 27]  # 광물, 에너지
    intermediates = list(range(72, 84))  # 금속
    final_goods = list(range(84, 91))  # 기계, 전자

    def get_supply_chain_level(hs2):
        if hs2 in raw_materials:
            return 'raw'
        elif hs2 in intermediates:
            return 'intermediate'
        elif hs2 in final_goods:
            return 'final'
        return 'other'

    results = []

    for leader in tqdm(items, desc="공급망 가설"):
        x = pivot.loc[leader].values.astype(float)

        if np.count_nonzero(x) < 11:
            continue

        leader_hs2 = item_meta.get(leader, {}).get('hs2_first', None)
        leader_level = get_supply_chain_level(leader_hs2) if leader_hs2 else 'other'

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.count_nonzero(y) < 11:
                continue

            follower_hs2 = item_meta.get(follower, {}).get('hs2_first', None)
            follower_level = get_supply_chain_level(follower_hs2) if follower_hs2 else 'other'

            # 공급망 단계 확인
            is_supply_chain = False
            bonus = 0.0
            preferred_lag_range = range(1, 13)

            if leader_level == 'raw' and follower_level in ['intermediate', 'final']:
                is_supply_chain = True
                bonus = 0.10
                preferred_lag_range = range(3, 10)  # 긴 lag 선호

            elif leader_level == 'intermediate' and follower_level == 'final':
                is_supply_chain = True
                bonus = 0.08
                preferred_lag_range = range(2, 8)

            best_lag = None
            best_corr = 0.0

            for lag in preferred_lag_range:
                if len(x) <= lag:
                    continue

                if np.std(x[:-lag]) > 0 and np.std(y[lag:]) > 0:
                    corr = np.corrcoef(x[:-lag], y[lag:])[0, 1]

                    if is_supply_chain:
                        corr += bonus

                    if abs(corr) > abs(best_corr):
                        best_corr = corr
                        best_lag = lag

            if best_lag and abs(best_corr) >= 0.28:
                results.append({
                    'leading_item_id': leader,
                    'following_item_id': follower,
                    'best_lag': best_lag,
                    'max_corr': best_corr,
                    'supply_chain': is_supply_chain
                })

    return pd.DataFrame(results)


# =========================================
# 가설 4: "계절성 리더십"
# =========================================
def hypothesis_seasonal_leadership(pivot, item_meta):
    """
    가설: 특정 품목은 계절 변화를 먼저 감지

    예: 패션 → 섬유 원자재
        여행 성수기 → 항공 관련 품목

    방법: 계절 패턴 분해 후 리더십 측정
    """
    items = pivot.index.to_list()
    months = pivot.columns.to_list()

    # 월별 인덱스
    month_nums = [pd.to_datetime(m).month for m in months]

    results = []

    for leader in tqdm(items, desc="계절성 리더십"):
        x = pivot.loc[leader].values.astype(float)

        if np.count_nonzero(x) < 11:
            continue

        # 계절 패턴 추출 (푸리에 변환 간소화 버전)
        x_seasonal = np.zeros(len(x))
        for i, month in enumerate(month_nums):
            x_seasonal[i] = np.mean([x[j] for j in range(len(x))
                                    if month_nums[j] == month])

        leader_hs2 = item_meta.get(leader, {}).get('hs2_first', None)

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.count_nonzero(y) < 11:
                continue

            y_seasonal = np.zeros(len(y))
            for i, month in enumerate(month_nums):
                y_seasonal[i] = np.mean([y[j] for j in range(len(y))
                                        if month_nums[j] == month])

            follower_hs2 = item_meta.get(follower, {}).get('hs2_first', None)
            same_hs2 = (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2)

            best_lag = None
            best_corr = 0.0

            # 계절 패턴 우선, 특히 3, 6, 12개월 lag
            priority_lags = [3, 6, 12] + list(range(1, 13))

            for lag in priority_lags:
                if len(x) <= lag:
                    continue

                # 원본 상관
                if np.std(x[:-lag]) > 0 and np.std(y[lag:]) > 0:
                    corr_orig = np.corrcoef(x[:-lag], y[lag:])[0, 1]
                else:
                    corr_orig = 0

                # 계절 패턴 상관
                if np.std(x_seasonal[:-lag]) > 0 and np.std(y_seasonal[lag:]) > 0:
                    corr_seasonal = np.corrcoef(x_seasonal[:-lag], y_seasonal[lag:])[0, 1]
                else:
                    corr_seasonal = 0

                # 조합
                combined = 0.6 * abs(corr_orig) + 0.4 * abs(corr_seasonal)

                # 계절 주기 lag 보너스
                if lag in [3, 6, 12]:
                    combined += 0.05

                if same_hs2:
                    combined += 0.05

                if combined > best_corr:
                    best_corr = combined
                    best_lag = lag

            if best_lag and best_corr >= 0.28:
                results.append({
                    'leading_item_id': leader,
                    'following_item_id': follower,
                    'best_lag': best_lag,
                    'max_corr': best_corr
                })

    return pd.DataFrame(results)


# =========================================
# 가설 5: "거래량 기반 영향력"
# =========================================
def hypothesis_volume_influence(pivot, item_meta):
    """
    가설: 대량 거래 품목이 소량 거래 품목에 영향

    시장 지배력: 거래량 큰 품목 → 작은 품목
    """
    items = pivot.index.to_list()
    results = []

    # 품목별 평균 거래량
    item_volumes = {item: item_meta.get(item, {}).get('value_mean', 0)
                   for item in items}

    for leader in tqdm(items, desc="거래량 영향력"):
        x = pivot.loc[leader].values.astype(float)

        if np.count_nonzero(x) < 11:
            continue

        leader_volume = item_volumes.get(leader, 0)
        leader_hs2 = item_meta.get(leader, {}).get('hs2_first', None)

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.count_nonzero(y) < 11:
                continue

            follower_volume = item_volumes.get(follower, 0)
            follower_hs2 = item_meta.get(follower, {}).get('hs2_first', None)
            same_hs2 = (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2)

            # 거래량 비율
            volume_ratio = leader_volume / (follower_volume + 1e-6)

            # 대량 → 소량 영향일 때 보너스
            influence_bonus = 0.0
            if volume_ratio > 2.0:  # leader가 2배 이상 큰 거래
                influence_bonus = 0.08
            elif volume_ratio > 1.5:
                influence_bonus = 0.05

            best_lag = None
            best_corr = 0.0

            for lag in range(1, min(13, len(x))):
                if len(x) <= lag:
                    continue

                if np.std(x[:-lag]) > 0 and np.std(y[lag:]) > 0:
                    corr = np.corrcoef(x[:-lag], y[lag:])[0, 1]

                    corr += influence_bonus

                    if same_hs2:
                        corr += 0.05

                    if abs(corr) > abs(best_corr):
                        best_corr = corr
                        best_lag = lag

            if best_lag and abs(best_corr) >= 0.28:
                results.append({
                    'leading_item_id': leader,
                    'following_item_id': follower,
                    'best_lag': best_lag,
                    'max_corr': best_corr,
                    'volume_ratio': volume_ratio
                })

    return pd.DataFrame(results)


# =========================================
# 실행 및 앙상블
# =========================================
print("="*70)
print("창의적 가설 기반 공행성 탐색")
print("="*70)

item_meta = get_item_meta(train)

hypotheses = {}

print("\n[가설 1] 대체재/보완재")
hypotheses['substitute'] = hypothesis_substitute_complement(pivot, item_meta)
print(f"쌍 수: {len(hypotheses['substitute'])}")

print("\n[가설 2] 글로벌 이벤트 동조화")
hypotheses['event'] = hypothesis_global_event_sync(pivot, item_meta)
print(f"쌍 수: {len(hypotheses['event'])}")

print("\n[가설 3] 공급망 연쇄")
hypotheses['supply_chain'] = hypothesis_supply_chain(pivot, item_meta)
print(f"쌍 수: {len(hypotheses['supply_chain'])}")

print("\n[가설 4] 계절성 리더십")
hypotheses['seasonal'] = hypothesis_seasonal_leadership(pivot, item_meta)
print(f"쌍 수: {len(hypotheses['seasonal'])}")

print("\n[가설 5] 거래량 영향력")
hypotheses['volume'] = hypothesis_volume_influence(pivot, item_meta)
print(f"쌍 수: {len(hypotheses['volume'])}")

# 앙상블
print("\n" + "="*70)
print("가설 앙상블")
print("="*70)

vote_dict = {}

for name, df in hypotheses.items():
    if len(df) == 0:
        continue

    for _, row in df.iterrows():
        key = (row['leading_item_id'], row['following_item_id'])

        if key not in vote_dict:
            vote_dict[key] = {
                'leading_item_id': row['leading_item_id'],
                'following_item_id': row['following_item_id'],
                'lags': [],
                'corrs': [],
                'votes': 0
            }

        vote_dict[key]['lags'].append(row['best_lag'])
        vote_dict[key]['corrs'].append(row['max_corr'])
        vote_dict[key]['votes'] += 1

# 2표 이상
final = []
for key, info in vote_dict.items():
    if info['votes'] >= 2:
        final.append({
            'leading_item_id': info['leading_item_id'],
            'following_item_id': info['following_item_id'],
            'best_lag': max(set(info['lags']), key=info['lags'].count),
            'max_corr': np.mean(info['corrs'])
        })

pairs = pd.DataFrame(final)

if len(pairs) > 0:
    pairs = pairs.sort_values('max_corr', ascending=False)

print(f"\n최종 쌍 수: {len(pairs)}")
if len(pairs) > 0:
    print(f"평균 상관: {pairs['max_corr'].mean():.4f}")

print("\n개별 가설:")
for name, df in hypotheses.items():
    if len(df) > 0:
        print(f"{name}: {len(df)}개, 평균={df['max_corr'].mean():.4f}")

창의적 가설 기반 공행성 탐색

[가설 1] 대체재/보완재


대체재/보완재 가설: 100%|██████████| 100/100 [00:19<00:00,  5.08it/s]


쌍 수: 5920

[가설 2] 글로벌 이벤트 동조화


글로벌 이벤트 가설: 100%|██████████| 100/100 [00:24<00:00,  4.09it/s]


쌍 수: 7179

[가설 3] 공급망 연쇄


...

쌍 수: 5808

[가설 4] 계절성 리더십


계절성 리더십: 100%|██████████| 100/100 [00:48<00:00,  2.05it/s]


쌍 수: 7674

[가설 5] 거래량 영향력


거래량 영향력: 100%|██████████| 100/100 [00:22<00:00,  4.53it/s]


쌍 수: 6122

가설 앙상블

최종 쌍 수: 7203
평균 상관: 0.3056

개별 가설:
substitute: 5920개, 평균=0.1801
event: 7179개, 평균=0.4618
supply_chain: 5808개, 평균=0.1429
seasonal: 7674개, 평균=0.4251
volume: 6122개, 평균=0.2442


### 상관계수

In [ ]:
pairs

,leading_item_id,following_item_id,best_lag,max_corr
5354,XIIEJNEE,IGDVVKUD,4,0.977564
5340,XIIEJNEE,DJBLNPNC,5,0.961626
3123,NAQIHUKZ,FTSVTTSR,1,0.933376
1688,FTSVTTSR,LLHREMKS,2,0.930652
3132,NAQIHUKZ,LLHREMKS,3,0.923910
...,...,...,...,...
2327,JSLXRQOK,BEZYMBBT,11,-0.160919
4946,VUAFAIYJ,OKMBFVKS,8,-0.161563
3126,NAQIHUKZ,GYHKIVQT,7,-0.168608
1861,GYHKIVQT,IGDVVKUD,8,-0.170999


In [ ]:
import pandas as pd

pairs['abs_corr'] = pairs['max_corr'].abs()

# 1. 전체 평균 상관계수 구하기
mean_corr = pairs['abs_corr'].mean()
print(f"- 전체 평균 상관계수: {mean_corr:.4f}")

# 2. 분위수(Quantile) 계산
# 상위 20% 기준값 (80% 지점)
top_20_cutoff = pairs['abs_corr'].quantile(0.80)

# 하위 20% 기준값 (20% 지점)
bottom_20_cutoff = pairs['abs_corr'].quantile(0.20)

print(f"- 상위 20% 커트라인: {top_20_cutoff:.4f} 이상")
print(f"- 하위 20% 커트라인: {bottom_20_cutoff:.4f} 이하")

# 3. 데이터 추출 (Filtering)
top_20_df = pairs[pairs['abs_corr'] >= top_20_cutoff]
bottom_20_df = pairs[pairs['abs_corr'] <= bottom_20_cutoff]

# 4. 결과 출력
print(f"\n[상위 20% 데이터] - 총 {len(top_20_df)}개")
# 상위 5개만 확인
print(top_20_df[['leading_item_id', 'following_item_id', 'best_lag', 'abs_corr']].head())

print(f"\n[하위 20% 데이터] - 총 {len(bottom_20_df)}개")
# 하위 5개만 확인
print(bottom_20_df[['leading_item_id', 'following_item_id', 'best_lag', 'abs_corr']].head())

- 전체 평균 상관계수: 0.3265
- 상위 20% 커트라인: 0.4696 이상
- 하위 20% 커트라인: 0.0824 이하

[상위 20% 데이터] - 총 1441개
     leading_item_id following_item_id  best_lag  abs_corr
5354        XIIEJNEE          IGDVVKUD         4  0.977564
5340        XIIEJNEE          DJBLNPNC         5  0.961626
3123        NAQIHUKZ          FTSVTTSR         1  0.933376
1688        FTSVTTSR          LLHREMKS         2  0.930652
3132        NAQIHUKZ          LLHREMKS         3  0.923910

[하위 20% 데이터] - 총 1441개
     leading_item_id following_item_id  best_lag  abs_corr
3317        OJIFIHMZ          FWUCPMMW         2  0.082150
3380        OKMBFVKS          FWUCPMMW        10  0.082148
1192        DUCMGGNW          VUAFAIYJ         5  0.081885
4577        UGEQLMXM          ZXERAXWP         6  0.081697
1790        GKQIJYDH          DDEXPPXU         4  0.081540


### 가설9 분석
- 공행성 쌍 개수: 7203
- 전체 평균 상관계수: 0.3265
- 상위 20% 커트라인: 0.4696 이상
- 하위 20% 커트라인: 0.0824 이하
---
- 공행성쌍이 너무 올라간 상태여서, 잘못된 접근이라고 판단
    - 아무리 못해도 1500~3500 사이가 적절한 공행성쌍이라고 판단
- 가설들이 너무 많은 보너스 점수를 줌
- 임계값이 상대적으로 낮아짐
- [품질 < 수량]에 집중한 문제
- 다양한 가설보다는 지금의 올바른 가설 방향에서 수정만 하자

# 가설10. 기존 문제를 해결하고, 보수적으로 접근하는 것이 올바른 공행성쌍의 추출로 이어질 것이다.
- **기존(전략 10)의 문제**
    - 과도한 보너스 점수
    - 낮은 임계값 유지
    - 품질 필터 부재
- **품질에 집중, 파라미터 조정**
    - 품질 필터: 데이터 충분성 + 적절한 변동성
    - 임계값 상향: 0.28 → 0.29
    - 보너스 절제: 0.05 → 0.04
- **최근 품목에 대한 가중치 부여**
    - 최근 12개월 60% 가중
    - 임계값 0.30 (높음)
    - 2025년 예측에 최적화
- **엄격하게 TopK 설정**
    - 리더당 최대 12개
    - 임계값 0.27 (낮지만 TopK로 품질 보장)
- **보수적 앙상블**
    - 높은 임계값들 (0.30, 0.32, 0.34)
    - 최소 2표 이상
- **회귀 모델**
    - RandomForest + Optuna

### 가설10 검증 및 결과

In [ ]:
# =========================================
# 정제된 공행성쌍 탐색
# 목표: 2,000-3,000개 고품질 쌍
# 전략: 기존 최고 방식 유지 + 선택적 개선
# =========================================
import pandas as pd
import numpy as np
from tqdm import tqdm
from scipy.stats import spearmanr, kendalltau
import warnings
warnings.filterwarnings('ignore')


def get_item_meta(train):
    train['hs2'] = train['hs4'] // 100
    meta = train.groupby('item_id').agg({
        'hs4': 'first',
        'hs2': 'first',
        'value': ['mean', 'sum', 'std']
    })
    meta.columns = ['_'.join(col).strip() for col in meta.columns.values]
    return meta.to_dict('index')


# =========================================
# 기본 함수들 (검증된 버전)
# =========================================
def dtw_distance(x, y, window=3):
    n, m = len(x), len(y)
    x = (x - np.mean(x)) / (np.std(x) + 1e-8)
    y = (y - np.mean(y)) / (np.std(y) + 1e-8)

    dtw = np.full((n + 1, m + 1), np.inf)
    dtw[0, 0] = 0

    for i in range(1, n + 1):
        for j in range(max(1, i - window), min(m + 1, i + window + 1)):
            cost = abs(x[i-1] - y[j-1])
            dtw[i, j] = cost + min(dtw[i-1, j], dtw[i, j-1], dtw[i-1, j-1])

    return dtw[n, m]


def dtw_similarity(x, y, window=3):
    dist = dtw_distance(x, y, window)
    return 1.0 / (1.0 + dist)


def cross_correlation_function(x, y, max_lag=12):
    x = (x - np.mean(x)) / (np.std(x) + 1e-8)
    y = (y - np.mean(y)) / (np.std(y) + 1e-8)

    correlations = []
    for lag in range(1, min(max_lag + 1, len(x))):
        if len(x) <= lag:
            break
        corr = np.corrcoef(x[:-lag], y[lag:])[0, 1]
        correlations.append((lag, corr))

    if not correlations:
        return 0, 0.0

    best_lag, best_corr = max(correlations, key=lambda t: abs(t[1]))
    return best_lag, best_corr


def granger_causality_score(x, y, max_lag=6):
    best_score = 0.0
    best_lag = 1

    for lag in range(1, min(max_lag + 1, len(x) - 5)):
        if len(x) <= lag + 2:
            continue

        y_current = y[lag:]
        y_lagged = y[:-lag]
        x_lagged = x[:-lag]

        corr_y_self = abs(np.corrcoef(y_current, y_lagged)[0, 1])
        corr_x_y = abs(np.corrcoef(x_lagged, y_current)[0, 1])

        score = max(0, corr_x_y - corr_y_self * 0.5)

        if score > best_score:
            best_score = score
            best_lag = lag

    return best_lag, best_score


def shape_similarity(x, y, window_size=6):
    x_smooth = pd.Series(x).rolling(window=window_size, min_periods=1).mean().values
    y_smooth = pd.Series(y).rolling(window=window_size, min_periods=1).mean().values

    x_diff = np.diff(x_smooth)
    y_diff = np.diff(y_smooth)

    if len(x_diff) > 0 and len(y_diff) > 0:
        direction_match = np.mean(np.sign(x_diff) == np.sign(y_diff))
    else:
        direction_match = 0

    dtw_sim = dtw_similarity(x_smooth, y_smooth, window=3)

    return 0.5 * direction_match + 0.5 * dtw_sim


# =========================================
# 전략 1: 기존 최고 방식 (0.344 달성) + 품질 필터
# =========================================
def strategy_baseline_refined(pivot, item_meta, max_lag=12, min_nonzero=11,
                               corr_threshold=0.29, quality_filter=True):
    """
    검증된 기존 방식 + 품질 필터 강화
    - CCF + DTW + Granger + Shape
    - 임계값 약간 상향 (0.28 → 0.29)
    - 품질 낮은 품목 사전 제거
    """
    items = pivot.index.to_list()

    # 품질 필터링
    if quality_filter:
        quality_items = []
        for item in items:
            values = pivot.loc[item].values.astype(float)

            # 기준: 충분한 데이터 + 적절한 변동성
            nonzero_ratio = np.count_nonzero(values) / len(values)
            cv = np.std(values) / (np.mean(values) + 1e-6)

            if nonzero_ratio >= 0.7 and 0.2 < cv < 3.0:
                quality_items.append(item)

        print(f"품질 필터: {len(items)} → {len(quality_items)}개")
        items = quality_items

    results = []

    for leader in tqdm(items, desc="정제된 기준선"):
        x = pivot.loc[leader].values.astype(float)

        if np.count_nonzero(x) < min_nonzero:
            continue

        leader_hs2 = item_meta.get(leader, {}).get('hs2_first', None)

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.count_nonzero(y) < min_nonzero:
                continue

            follower_hs2 = item_meta.get(follower, {}).get('hs2_first', None)
            same_hs2 = (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2)

            scores = {}
            lags = {}

            # CCF
            lag_ccf, corr_ccf = cross_correlation_function(x, y, max_lag)
            scores['ccf'] = abs(corr_ccf)
            lags['ccf'] = lag_ccf

            # DTW
            best_dtw = 0
            best_lag_dtw = 1
            for lag in range(1, min(max_lag + 1, len(x))):
                if len(x) <= lag:
                    break
                sim = dtw_similarity(x[:-lag], y[lag:], window=3)
                if sim > best_dtw:
                    best_dtw = sim
                    best_lag_dtw = lag
            scores['dtw'] = best_dtw
            lags['dtw'] = best_lag_dtw

            # Granger
            lag_granger, score_granger = granger_causality_score(x, y, max_lag)
            scores['granger'] = score_granger
            lags['granger'] = lag_granger

            # Shape
            best_shape = 0
            best_lag_shape = 1
            for lag in range(1, min(max_lag + 1, len(x))):
                if len(x) <= lag:
                    break
                sim = shape_similarity(x[:-lag], y[lag:], window_size=3)
                if sim > best_shape:
                    best_shape = sim
                    best_lag_shape = lag
            scores['shape'] = best_shape
            lags['shape'] = best_lag_shape

            avg_score = np.mean(list(scores.values()))

            # 보너스 (절제된)
            if same_hs2:
                avg_score += 0.04  # 0.05 → 0.04로 감소

            lag_values = list(lags.values())
            best_lag = max(set(lag_values), key=lag_values.count)

            if avg_score >= corr_threshold:
                results.append({
                    'leading_item_id': leader,
                    'following_item_id': follower,
                    'best_lag': best_lag,
                    'max_corr': avg_score
                })

    return pd.DataFrame(results)


# =========================================
# 전략 2: 최근 데이터 가중 (보수적)
# =========================================
def strategy_recent_weighted(pivot, item_meta, max_lag=12, min_nonzero=11,
                              corr_threshold=0.30, recent_weight=0.6):
    """
    최근 데이터에 더 높은 가중치
    임계값 상향 조정으로 품질 유지
    """
    items = pivot.index.to_list()
    n_months = len(pivot.columns)
    recent_months = 12

    results = []

    for leader in tqdm(items, desc="최근 가중"):
        x = pivot.loc[leader].values.astype(float)

        if np.count_nonzero(x) < min_nonzero:
            continue

        leader_hs2 = item_meta.get(leader, {}).get('hs2_first', None)

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.count_nonzero(y) < min_nonzero:
                continue

            follower_hs2 = item_meta.get(follower, {}).get('hs2_first', None)
            same_hs2 = (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2)

            best_lag = None
            best_score = 0.0

            for lag in range(1, min(max_lag + 1, n_months)):
                if n_months <= lag:
                    continue

                # 전체 기간 상관
                if np.std(x[:-lag]) > 0 and np.std(y[lag:]) > 0:
                    corr_full = np.corrcoef(x[:-lag], y[lag:])[0, 1]
                else:
                    corr_full = 0

                # 최근 기간 상관
                if n_months > recent_months + lag:
                    x_recent = x[-(recent_months+lag):-lag]
                    y_recent = y[-recent_months:]

                    if np.std(x_recent) > 0 and np.std(y_recent) > 0:
                        corr_recent = np.corrcoef(x_recent, y_recent)[0, 1]
                    else:
                        corr_recent = corr_full
                else:
                    corr_recent = corr_full

                # 가중 조합
                combined = (1 - recent_weight) * abs(corr_full) + recent_weight * abs(corr_recent)

                if same_hs2:
                    combined += 0.04

                if combined > best_score:
                    best_score = combined
                    best_lag = lag

            if best_lag and best_score >= corr_threshold:
                results.append({
                    'leading_item_id': leader,
                    'following_item_id': follower,
                    'best_lag': best_lag,
                    'max_corr': best_score
                })

    return pd.DataFrame(results)


# =========================================
# 전략 3: TopK 강화 (리더당 제한)
# =========================================
def strategy_topk_strict(pivot, item_meta, max_lag=12, min_nonzero=11,
                          corr_threshold=0.27, top_k_per_leader=12):
    """
    낮은 임계값 + 엄격한 TopK
    각 리더당 최대 12개만 선택
    """
    items = pivot.index.to_list()
    all_results = []

    for leader in tqdm(items, desc="엄격 TopK"):
        x = pivot.loc[leader].values.astype(float)

        if np.count_nonzero(x) < min_nonzero:
            continue

        leader_hs2 = item_meta.get(leader, {}).get('hs2_first', None)
        leader_results = []

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.count_nonzero(y) < min_nonzero:
                continue

            follower_hs2 = item_meta.get(follower, {}).get('hs2_first', None)
            same_hs2 = (leader_hs2 and follower_hs2 and leader_hs2 == follower_hs2)

            # 4가지 지표
            lag_ccf, corr_ccf = cross_correlation_function(x, y, max_lag)

            best_dtw = 0
            best_lag_dtw = 1
            for lag in range(1, min(max_lag + 1, len(x))):
                if len(x) <= lag:
                    break
                sim = dtw_similarity(x[:-lag], y[lag:], window=3)
                if sim > best_dtw:
                    best_dtw = sim
                    best_lag_dtw = lag

            lag_granger, score_granger = granger_causality_score(x, y, max_lag)

            best_shape = 0
            best_lag_shape = 1
            for lag in range(1, min(max_lag + 1, len(x))):
                if len(x) <= lag:
                    break
                sim = shape_similarity(x[:-lag], y[lag:], window_size=3)
                if sim > best_shape:
                    best_shape = sim
                    best_lag_shape = lag

            avg_score = np.mean([abs(corr_ccf), best_dtw, score_granger, best_shape])

            if same_hs2:
                avg_score += 0.04

            lags = [lag_ccf, best_lag_dtw, lag_granger, best_lag_shape]
            best_lag = max(set(lags), key=lags.count)

            if avg_score >= corr_threshold:
                leader_results.append({
                    'leading_item_id': leader,
                    'following_item_id': follower,
                    'best_lag': best_lag,
                    'max_corr': avg_score
                })

        # 상위 K개만 선택
        if leader_results:
            leader_df = pd.DataFrame(leader_results)
            leader_df = leader_df.nlargest(top_k_per_leader, 'max_corr')
            all_results.extend(leader_df.to_dict('records'))

    return pd.DataFrame(all_results)


# =========================================
# 전략 4: 다중 임계값 앙상블 (보수적)
# =========================================
def strategy_multi_threshold_conservative(pivot, item_meta):
    """
    높은 임계값들로만 앙상블
    0.30, 0.32, 0.34
    """
    thresholds = [0.30, 0.32, 0.34]
    all_pairs = []

    for threshold in thresholds:
        print(f"  임계값 {threshold} 탐색...")
        pairs = strategy_baseline_refined(pivot, item_meta,
                                          corr_threshold=threshold,
                                          quality_filter=True)
        all_pairs.append(pairs)

    # 투표
    vote_dict = {}

    for pairs_df in all_pairs:
        for _, row in pairs_df.iterrows():
            key = (row['leading_item_id'], row['following_item_id'])

            if key not in vote_dict:
                vote_dict[key] = {
                    'leading_item_id': row['leading_item_id'],
                    'following_item_id': row['following_item_id'],
                    'lags': [],
                    'corrs': [],
                    'votes': 0
                }

            vote_dict[key]['lags'].append(row['best_lag'])
            vote_dict[key]['corrs'].append(row['max_corr'])
            vote_dict[key]['votes'] += 1

    # 2표 이상
    final = []
    for key, info in vote_dict.items():
        if info['votes'] >= 2:
            final.append({
                'leading_item_id': info['leading_item_id'],
                'following_item_id': info['following_item_id'],
                'best_lag': max(set(info['lags']), key=info['lags'].count),
                'max_corr': np.mean(info['corrs'])
            })

    return pd.DataFrame(final)


# =========================================
# 실행
# =========================================
print("="*70)
print("정제된 공행성쌍 탐색 (2,000-3,000개 목표)")
print("="*70)

item_meta = get_item_meta(train)

strategies = {}

print("\n[전략 1] 정제된 기준선 (품질 필터)")
strategies['refined'] = strategy_baseline_refined(
    pivot, item_meta,
    corr_threshold=0.29,
    quality_filter=True
)
print(f"쌍 수: {len(strategies['refined'])}")
if len(strategies['refined']) > 0:
    print(f"평균 상관: {strategies['refined']['max_corr'].mean():.4f}")

print("\n[전략 2] 최근 가중 (보수적)")
strategies['recent'] = strategy_recent_weighted(
    pivot, item_meta,
    corr_threshold=0.30,
    recent_weight=0.6
)
print(f"쌍 수: {len(strategies['recent'])}")
if len(strategies['recent']) > 0:
    print(f"평균 상관: {strategies['recent']['max_corr'].mean():.4f}")

print("\n[전략 3] 엄격 TopK")
strategies['topk'] = strategy_topk_strict(
    pivot, item_meta,
    corr_threshold=0.27,
    top_k_per_leader=12
)
print(f"쌍 수: {len(strategies['topk'])}")
if len(strategies['topk']) > 0:
    print(f"평균 상관: {strategies['topk']['max_corr'].mean():.4f}")

print("\n[전략 4] 보수적 앙상블")
strategies['ensemble'] = strategy_multi_threshold_conservative(pivot, item_meta)
print(f"쌍 수: {len(strategies['ensemble'])}")
if len(strategies['ensemble']) > 0:
    print(f"평균 상관: {strategies['ensemble']['max_corr'].mean():.4f}")

# 최종 선택
print("\n" + "="*70)
print("전략별 요약")
print("="*70)

for name, df in strategies.items():
    if len(df) > 0:
        print(f"{name}: {len(df)}개, 평균상관={df['max_corr'].mean():.4f}")

# 가장 균형잡힌 전략 자동 선택 (2000-3000 범위)
best_strategy = None
best_name = None

for name, df in strategies.items():
    if 2000 <= len(df) <= 3000:
        if best_strategy is None or df['max_corr'].mean() > best_strategy['max_corr'].mean():
            best_strategy = df
            best_name = name

if best_strategy is not None:
    print(f"\n추천 전략: {best_name}")
    pairs = best_strategy
else:
    print("\n기본 선택: refined")
    pairs = strategies['refined']

print(f"최종 선택: {len(pairs)}개 쌍")

정제된 공행성쌍 탐색 (2,000-3,000개 목표)

[전략 1] 정제된 기준선 (품질 필터)
품질 필터: 100 → 76개


정제된 기준선: 100%|██████████| 76/76 [01:53<00:00,  1.50s/it]


쌍 수: 1509
평균 상관: 0.3398

[전략 2] 최근 가중 (보수적)


최근 가중: 100%|██████████| 100/100 [00:37<00:00,  2.69it/s]


쌍 수: 7850
평균 상관: 0.4543

[전략 3] 엄격 TopK


...

  임계값 0.32 탐색...
품질 필터: 100 → 76개


정제된 기준선: 100%|██████████| 76/76 [01:57<00:00,  1.54s/it]


  임계값 0.34 탐색...
품질 필터: 100 → 76개


정제된 기준선: 100%|██████████| 76/76 [01:54<00:00,  1.51s/it]

쌍 수: 874
평균 상관: 0.3658

전략별 요약
refined: 1509개, 평균상관=0.3398
recent: 7850개, 평균상관=0.4543
topk: 1090개, 평균상관=0.3597
ensemble: 874개, 평균상관=0.3658

기본 선택: refined
최종 선택: 1509개 쌍


### 상관계수

In [ ]:
pairs

,leading_item_id,following_item_id,best_lag,max_corr
0,AHMDUILJ,AXULOHBQ,9,0.298966
1,AHMDUILJ,BJALXPFS,10,0.363818
2,AHMDUILJ,BSRMSVTC,12,0.311856
3,AHMDUILJ,DUCMGGNW,1,0.322600
4,AHMDUILJ,FDXPMYGF,11,0.356726
...,...,...,...,...
1504,ZXERAXWP,LRVGFDFM,10,0.338181
1505,ZXERAXWP,MIRCVAMV,11,0.299315
1506,ZXERAXWP,SAAYMURU,12,0.303267
1507,ZXERAXWP,UIFPPCLR,1,0.344870


In [ ]:
import pandas as pd

pairs['abs_corr'] = pairs['max_corr'].abs()

# 1. 전체 평균 상관계수 구하기
mean_corr = pairs['abs_corr'].mean()
print(f"- 전체 평균 상관계수: {mean_corr:.4f}")

# 2. 분위수(Quantile) 계산
# 상위 20% 기준값 (80% 지점)
top_20_cutoff = pairs['abs_corr'].quantile(0.80)

# 하위 20% 기준값 (20% 지점)
bottom_20_cutoff = pairs['abs_corr'].quantile(0.20)

print(f"- 상위 20% 커트라인: {top_20_cutoff:.4f} 이상")
print(f"- 하위 20% 커트라인: {bottom_20_cutoff:.4f} 이하")

# 3. 데이터 추출 (Filtering)
top_20_df = pairs[pairs['abs_corr'] >= top_20_cutoff]
bottom_20_df = pairs[pairs['abs_corr'] <= bottom_20_cutoff]

# 4. 결과 출력
print(f"\n[상위 20% 데이터] - 총 {len(top_20_df)}개")
# 상위 5개만 확인
print(top_20_df[['leading_item_id', 'following_item_id', 'best_lag', 'abs_corr']].head())

print(f"\n[하위 20% 데이터] - 총 {len(bottom_20_df)}개")
# 하위 5개만 확인
print(bottom_20_df[['leading_item_id', 'following_item_id', 'best_lag', 'abs_corr']].head())

- 전체 평균 상관계수: 0.3398
- 상위 20% 커트라인: 0.3729 이상
- 하위 20% 커트라인: 0.3028 이하

[상위 20% 데이터] - 총 302개
   leading_item_id following_item_id  best_lag  abs_corr
34        APQGTRMF          UIFPPCLR         1  0.377874
44        ATLDMDBO          AXULOHBQ         7  0.473868
47        ATLDMDBO          BTMOEMEP         1  0.383921
50        ATLDMDBO          DNMPSKTB         8  0.487006
53        ATLDMDBO          FQCLOEXA        10  0.380171

[하위 20% 데이터] - 총 302개
   leading_item_id following_item_id  best_lag  abs_corr
0         AHMDUILJ          AXULOHBQ         9  0.298966
9         AHMDUILJ          RJGPVEXX         7  0.292081
22        APQGTRMF          GYHKIVQT         8  0.299290
26        APQGTRMF          LUENUFGA        11  0.293104
27        APQGTRMF          OXKURKXR         2  0.299750


### 가설10 분석
- 점수: 0.3855490402
- 공행성 쌍 개수: 1510
- 전체 평균 상관계수: 0.3398
- 상위 20% 커트라인: 0.3729 이상
- 하위 20% 커트라인: 0.3028 이하
---
- 진행 상황 정리
    - baseline: ~0.32
    - 비선형 도입: 0.342
    - 앙상블 도입: 0.344
    - 회귀 피처 확장: 0.350
    - 현재: 0.3855
- lag 분포 ⇒ 12개월이 가장 많음 (206개)
    - 계절성이 강하다는 가설을 입증함

# 회귀 모델 학습

In [ ]:
# baseline

def build_training_data(pivot, pairs):
    """
    공행성쌍 + 시계열을 이용해 (X, y) 학습 데이터를 만드는 함수
    input X:
      - b_t, b_t_1, a_t_lag, max_corr, best_lag
    target y:
      - b_t_plus_1
    """
    months = pivot.columns.to_list()
    n_months = len(months)

    rows = []

    for row in pairs.itertuples(index=False):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)
        corr = float(row.max_corr)

        if leader not in pivot.index or follower not in pivot.index:
            continue

        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)

        # t+1이 존재하고, t-lag >= 0인 구간만 학습에 사용
        for t in range(max(lag, 1), n_months - 1):
            b_t = b_series[t]
            b_t_1 = b_series[t - 1]
            a_t_lag = a_series[t - lag]
            b_t_plus_1 = b_series[t + 1]

            rows.append({
                "b_t": b_t,
                "b_t_1": b_t_1,
                "a_t_lag": a_t_lag,
                "max_corr": corr,
                "best_lag": float(lag),
                "target": b_t_plus_1,
            })

    df_train = pd.DataFrame(rows)
    return df_train

df_train_model = build_training_data(pivot, pairs)
print('생성된 학습 데이터의 shape :', df_train_model.shape)
df_train_model.head()

In [ ]:
# # 회귀모델 학습

# reg = LinearRegression()
# reg.fit(df_train_model[["b_t", "b_t_1", "a_t_lag", "max_corr", "best_lag"]], df_train_model["target"])

In [ ]:
# # 회귀 모델 학습 (RandomForest ver.)

# from sklearn.ensemble import RandomForestRegressor

# reg = RandomForestRegressor(random_state=0)
# reg.fit(df_train_model[["b_t", "b_t_1", "a_t_lag", "max_corr", "best_lag"]], df_train_model["target"])

In [ ]:
# baseline

def predict(pivot, pairs, reg):
    months = pivot.columns.to_list()
    n_months = len(months)

    # 가장 마지막 두 달 index (2025-7, 2025-6)
    t_last = n_months - 1
    t_prev = n_months - 2

    preds = []

    for row in tqdm(pairs.itertuples(index=False)):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)
        corr = float(row.max_corr)

        if leader not in pivot.index or follower not in pivot.index:
            continue

        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)

        # t_last - lag 가 0 이상인 경우만 예측
        if t_last - lag < 0:
            continue

        b_t = b_series[t_last]
        b_t_1 = b_series[t_prev]
        a_t_lag = a_series[t_last - lag]

        X_test = np.array([[b_t, b_t_1, a_t_lag, corr, float(lag)]])
        y_pred = reg.predict(X_test)[0]

        # (후처리 1) 음수 예측 → 0으로 변환
        # (후처리 2) 소수점 → 정수 변환 (무역량은 정수 단위)
        y_pred = max(0.0, float(y_pred))
        y_pred = int(round(y_pred))

        preds.append({
            "leading_item_id": leader,
            "following_item_id": follower,
            "value": y_pred,
        })

    df_pred = pd.DataFrame(preds)
    return df_pred

In [ ]:
# RandomForest + optuna

# =========================================
# 회귀 모델 대폭 개선
# 핵심: 피처 5개 → 20+ 개
# =========================================
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import optuna
import warnings
warnings.filterwarnings('ignore')


# =========================================
# 1. 향상된 학습 데이터 구성 (피처 확장)
# =========================================
def build_enhanced_training_data(pivot, pairs):
    """
    기존 5개 피처 → 20+ 개 피처로 확장
    """
    months = pivot.columns.to_list()
    n_months = len(months)

    rows = []

    for row in pairs.itertuples(index=False):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)
        corr = float(row.max_corr)

        if leader not in pivot.index or follower not in pivot.index:
            continue

        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)

        # t+1이 존재하고, t-lag >= 0 인 구간만 사용
        for t in range(max(lag, 1), n_months - 1):
            # 기본 피처 (기존 5개)
            b_t = b_series[t]
            b_t_1 = b_series[t - 1]
            a_t_lag = a_series[t - lag]
            b_t_plus_1 = b_series[t + 1]  # 타겟

            # ===== 새로운 피처들 =====

            # 1. B의 추가 lag 피처
            b_t_2 = b_series[t - 2] if t >= 2 else b_t_1
            b_t_3 = b_series[t - 3] if t >= 3 else b_t_1

            # 2. A의 추가 lag 피처
            a_t_lag_1 = a_series[t - lag - 1] if t - lag - 1 >= 0 else a_t_lag
            a_t_lag_2 = a_series[t - lag - 2] if t - lag - 2 >= 0 else a_t_lag

            # 3. 변화율 피처
            b_change_1 = (b_t - b_t_1) / (b_t_1 + 1e-6)
            b_change_2 = (b_t_1 - b_t_2) / (b_t_2 + 1e-6)
            a_change = (a_t_lag - a_t_lag_1) / (a_t_lag_1 + 1e-6)

            # 4. 이동 평균 피처
            b_ma_3 = np.mean(b_series[max(0, t-2):t+1])
            b_ma_6 = np.mean(b_series[max(0, t-5):t+1])
            a_ma_3 = np.mean(a_series[max(0, t-lag-2):t-lag+1])

            # 5. 변동성 피처
            b_std_3 = np.std(b_series[max(0, t-2):t+1])
            a_std_3 = np.std(a_series[max(0, t-lag-2):t-lag+1])

            # 6. 비율 피처
            a_b_ratio = a_t_lag / (b_t + 1e-6)
            b_to_ma_ratio = b_t / (b_ma_3 + 1e-6)

            # 7. 교차 피처
            a_b_product = a_t_lag * b_t
            a_b_diff = a_t_lag - b_t

            # 8. 추세 피처
            b_trend = np.polyfit(range(max(0, t-5), t+1),
                                b_series[max(0, t-5):t+1], 1)[0]

            # 9. 시간 피처
            month_idx = t % 12
            quarter = month_idx // 3
            is_year_end = 1 if month_idx in [10, 11] else 0

            # 10. 상관계수 관련 피처
            corr_squared = corr ** 2
            lag_corr_interaction = lag * corr

            rows.append({
                # 기존 피처
                "b_t": b_t,
                "b_t_1": b_t_1,
                "a_t_lag": a_t_lag,
                "max_corr": corr,
                "best_lag": float(lag),

                # 새 피처들
                "b_t_2": b_t_2,
                "b_t_3": b_t_3,
                "a_t_lag_1": a_t_lag_1,
                "a_t_lag_2": a_t_lag_2,

                "b_change_1": b_change_1,
                "b_change_2": b_change_2,
                "a_change": a_change,

                "b_ma_3": b_ma_3,
                "b_ma_6": b_ma_6,
                "a_ma_3": a_ma_3,

                "b_std_3": b_std_3,
                "a_std_3": a_std_3,

                "a_b_ratio": a_b_ratio,
                "b_to_ma_ratio": b_to_ma_ratio,

                "a_b_product": a_b_product,
                "a_b_diff": a_b_diff,

                "b_trend": b_trend,

                "month_idx": month_idx,
                "quarter": quarter,
                "is_year_end": is_year_end,

                "corr_squared": corr_squared,
                "lag_corr_interaction": lag_corr_interaction,

                "target": b_t_plus_1,
            })

    df_train = pd.DataFrame(rows)

    # 결측값 처리
    df_train = df_train.fillna(0)

    # 무한대 처리
    df_train = df_train.replace([np.inf, -np.inf], 0)

    return df_train


# =========================================
# 2. 고급 Optuna 튜닝
# =========================================
def advanced_optuna_tuning(X, y, n_trials=150):
    """
    더 많은 trials + 더 나은 파라미터 범위
    """

    X_train, X_valid, y_train, y_valid = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    def objective(trial):
        # 모델 선택
        model_type = trial.suggest_categorical('model_type',
                                               ['rf', 'gbm'])

        if model_type == 'rf':
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 300, 1000),
                "max_depth": trial.suggest_int("max_depth", 5, 40),
                "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 15),
                "max_features": trial.suggest_categorical("max_features",
                                                         ["sqrt", "log2", 0.8]),
                "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
                "random_state": 42,
            }
            model = RandomForestRegressor(**params)

        else:  # GradientBoosting
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 300, 1000),
                "learning_rate": trial.suggest_loguniform("learning_rate", 0.01, 0.3),
                "max_depth": trial.suggest_int("max_depth", 3, 15),
                "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 15),
                "subsample": trial.suggest_uniform("subsample", 0.6, 1.0),
                "random_state": 42,
            }
            model = GradientBoostingRegressor(**params)

        model.fit(X_train, y_train)
        preds = model.predict(X_valid)
        mse = mean_squared_error(y_valid, preds)

        return mse

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    print(f"Best MSE: {study.best_value:.4f}")
    print(f"Best Params: {study.best_params}")

    return study.best_params


# =========================================
# 3. 앙상블 회귀 모델
# =========================================
def ensemble_regression(X_train, y_train, X_test):
    """
    여러 모델의 앙상블
    """

    # 모델 1: RandomForest
    rf = RandomForestRegressor(
        n_estimators=800,
        max_depth=25,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    pred_rf = rf.predict(X_test)

    # 모델 2: GradientBoosting
    gbm = GradientBoostingRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=7,
        subsample=0.8,
        random_state=42
    )
    gbm.fit(X_train, y_train)
    pred_gbm = gbm.predict(X_test)

    # 모델 3: Ridge (선형 기준선)
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train, y_train)
    pred_ridge = ridge.predict(X_test)

    # 앙상블: 가중 평균
    ensemble_pred = 0.5 * pred_rf + 0.4 * pred_gbm + 0.1 * pred_ridge

    return ensemble_pred


# =========================================
# 실행 예시
# =========================================

# 1. 향상된 학습 데이터 생성
print("향상된 학습 데이터 생성 중...")
df_train_enhanced = build_enhanced_training_data(pivot, pairs)

print(f"생성된 학습 데이터 shape: {df_train_enhanced.shape}")
print(f"피처 수: {len(df_train_enhanced.columns) - 1}")  # target 제외

# 피처 목록 출력
feature_cols = [col for col in df_train_enhanced.columns if col != 'target']
print(f"\n피처 목록:")
for i, col in enumerate(feature_cols, 1):
    print(f"  {i}. {col}")

# 2. 데이터 분리
X = df_train_enhanced[feature_cols]
y = df_train_enhanced["target"]

# 3. Optuna 튜닝 (선택)
print("\n" + "="*70)
print("Optuna 튜닝 (150 trials)")
print("="*70)

# best_params = advanced_optuna_tuning(X, y, n_trials=150)

# 4. 최종 모델 학습 (간단 버전)
print("\n" + "="*70)
print("최종 모델 학습")
print("="*70)

# 간단한 RandomForest
reg = RandomForestRegressor(
    n_estimators=800,  # 증가
    max_depth=30,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

reg.fit(X, y)

print("학습 완료!")

# 피처 중요도 출력
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': reg.feature_importances_
}).sort_values('importance', ascending=False)

print("\n상위 10개 중요 피처:")
print(feature_importance.head(10))


# =========================================
# 5. 예측 (기존 코드와 동일)
# =========================================
def predict_enhanced(pivot, pairs, reg):
    """향상된 피처로 예측"""
    months = pivot.columns.to_list()
    n_months = len(months)

    t_last = n_months - 1
    t_prev = n_months - 2

    preds = []

    for row in pairs.itertuples(index=False):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)
        corr = float(row.max_corr)

        if leader not in pivot.index or follower not in pivot.index:
            continue

        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)

        if t_last - lag < 0:
            continue

        t = t_last

        # 모든 피처 계산 (학습과 동일)
        b_t = b_series[t]
        b_t_1 = b_series[t - 1]
        a_t_lag = a_series[t - lag]

        b_t_2 = b_series[t - 2] if t >= 2 else b_t_1
        b_t_3 = b_series[t - 3] if t >= 3 else b_t_1
        a_t_lag_1 = a_series[t - lag - 1] if t - lag - 1 >= 0 else a_t_lag
        a_t_lag_2 = a_series[t - lag - 2] if t - lag - 2 >= 0 else a_t_lag

        b_change_1 = (b_t - b_t_1) / (b_t_1 + 1e-6)
        b_change_2 = (b_t_1 - b_t_2) / (b_t_2 + 1e-6)
        a_change = (a_t_lag - a_t_lag_1) / (a_t_lag_1 + 1e-6)

        b_ma_3 = np.mean(b_series[max(0, t-2):t+1])
        b_ma_6 = np.mean(b_series[max(0, t-5):t+1])
        a_ma_3 = np.mean(a_series[max(0, t-lag-2):t-lag+1])

        b_std_3 = np.std(b_series[max(0, t-2):t+1])
        a_std_3 = np.std(a_series[max(0, t-lag-2):t-lag+1])

        a_b_ratio = a_t_lag / (b_t + 1e-6)
        b_to_ma_ratio = b_t / (b_ma_3 + 1e-6)

        a_b_product = a_t_lag * b_t
        a_b_diff = a_t_lag - b_t

        b_trend = np.polyfit(range(max(0, t-5), t+1),
                            b_series[max(0, t-5):t+1], 1)[0]

        month_idx = t % 12
        quarter = month_idx // 3
        is_year_end = 1 if month_idx in [10, 11] else 0

        corr_squared = corr ** 2
        lag_corr_interaction = lag * corr

        X_test = np.array([[
            b_t, b_t_1, a_t_lag, corr, float(lag),
            b_t_2, b_t_3, a_t_lag_1, a_t_lag_2,
            b_change_1, b_change_2, a_change,
            b_ma_3, b_ma_6, a_ma_3,
            b_std_3, a_std_3,
            a_b_ratio, b_to_ma_ratio,
            a_b_product, a_b_diff,
            b_trend,
            month_idx, quarter, is_year_end,
            corr_squared, lag_corr_interaction
        ]])

        y_pred = reg.predict(X_test)[0]
        y_pred = max(0.0, float(y_pred))
        y_pred = int(round(y_pred))

        preds.append({
            "leading_item_id": leader,
            "following_item_id": follower,
            "value": y_pred,
        })

    return pd.DataFrame(preds)


submission = predict_enhanced(pivot, pairs, reg)
submission.to_csv('./enhanced_submit.csv', index=False)

print("\n완료! 예측 시 predict_enhanced() 함수 사용")

In [ ]:
# # =========================================
# # 7. 예측 및 제출 파일 생성
# # =========================================
# def predict(pivot, pairs, reg):
#     months = pivot.columns.to_list()
#     n_months = len(months)

#     # 마지막 두 달 index
#     t_last = n_months - 1
#     t_prev = n_months - 2

#     preds = []

#     for row in tqdm(pairs.itertuples(index=False)):
#         leader = row.leading_item_id
#         follower = row.following_item_id
#         lag = int(row.best_lag)
#         corr = float(row.max_corr)
#         # corr = float(row.combined_score)

#         if leader not in pivot.index or follower not in pivot.index:
#             continue

#         a_series = pivot.loc[leader].values.astype(float)
#         b_series = pivot.loc[follower].values.astype(float)

#         if t_last - lag < 0:
#             continue

#         b_t = b_series[t_last]
#         b_t_1 = b_series[t_prev]
#         a_t_lag = a_series[t_last - lag]

#         X_test = np.array([[b_t, b_t_1, a_t_lag, corr, float(lag)]])
#         y_pred = reg.predict(X_test)[0]

#         # 후처리
#         y_pred = max(0.0, float(y_pred))  # 음수 → 0
#         y_pred = int(round(y_pred))       # 정수 변환

#         preds.append({
#             "leading_item_id": leader,
#             "following_item_id": follower,
#             "value": y_pred,
#         })

#     df_pred = pd.DataFrame(preds)
#     return df_pred


# submission = predict(pivot, pairs, reg)
# submission.head()

# submission.to_csv('./baseline_submit.csv', index=False)

In [ ]:
# 1. 탐색된 쌍 개수
print(f"총 쌍 개수: {len(pairs)}")

# 2. 상관계수 분포
print(pairs['max_corr'].describe())

# 3. lag 분포
print(pairs['best_lag'].value_counts())

# 4. 예측값 분포
print(submission['value'].describe())
print(f"0인 비율: {(submission['value'] == 0).mean()}")
print(f"음수 비율: {(submission['value'] < 0).mean()}")